# Extension Run 3: the acoustic-label control arm T_ac on CirCor (Experiment 2)

Scores the frozen control network A (the T architecture trained with S1 and S2 labels derived from the phonocardiogram
alone, no electrocardiogram; plan `Next Study - Beat-Accurate Synthetic ECG/EXP2_PREREG.md`) on CirCor exactly as Run 2
scored T. Built by `extension_study/notebooks/build_run3.py` from the locked Run 1 notebook with Run 2's patch set; the
shared analysed audio, the 464-feature extractor, the five-seed MLP, the random forest, the child-grouped folds and the
bootstrap are Run 1's cells, unchanged.

**What runs.** The frozen A models (checkpoint folder `Extension_Study/checkpoints/20261029_1824_ac`; every file and weight hash
checked against the embedded freeze record) run label-free on each recording; T_ac's cycles come from its S1 and S2
heads through the construction of T. Arms C and E are rebuilt only to show that this run reproduces Run 1 exactly.

**How to run.** Runtime > Change runtime type > **A100 GPU** with High-RAM; leave `DRY_RUN = False`; Runtime > Run all.
Expected time about 1.5 hours.

**Afterwards.** Download `Extension_Study/runs/run3/<timestamp>` (results.json, oof_predictions.npz, diagnostics.csv,
run_manifest.json). H1 and the contrasts are computed locally from this run's predictions merged with Runs 1 and 2.

| arm | key | boundaries |
|---|---|---|
| E | `equal_time` | 75 bpm pseudo-cycles (reproduction control) |
| **T_ac** | `timing_net_ac` | frozen A: S1 events from the shared decoder, S2 from its S2 head, 50% acoustic onsets |
| T_ac_annot | `timing_net_ac_annot` | T_ac cycles inside the annotated stretches |


In [ ]:
# ================================ RUN FLAGS ================================
DRY_RUN = False           # True: class-aware subset (about 60 murmur-present + 60 murmur-absent recordings), 5 folds, 100 bootstrap replicates, own '_dry' run folder
LIMIT_RECORDINGS = 0      # 0 = all recordings; N > 0 = an evenly spaced subset of N (not for the paper; own '_limitN' run folder)
N_BOOT = 2000             # patient-level bootstrap replicates
RESUME_DIR = None         # e.g. '/content/drive/MyDrive/JBHI-Revision/R1_Gating_Campaign/20260930_1200'; its run_flags.json must equal the flags below
FORCE_FIGURES = False     # draw figures even if the validation gate failed (diagnostic use only)
SEED = 42
PREREG_SHA256 = '755b47da43b250671391aae8672b182278cb4561314343581024ab1f581d8974'   # extension_study/prereg/PREREG_EXT.md, frozen 16 Oct 2026
ENSEMBLE_SEEDS = (42, 1042, 2042, 3042, 4042)   # MLP probabilities averaged over these seed bases plus the fold index (08 design 2.2)
R1_RUN_DIR = None          # Run 3 checks reproduction against Run 1's embedded hashes instead
TW_STAMP = '20261029_1824_ac'   # frozen control arm A (Extension_Study/checkpoints/<stamp>)
TW_CHECKPOINT_DIR = None   # None = the Drive folder above; a local path for a local run
REQUIRE_GPU = 'A100'       # full run only: Run 1's GPU class, so arm E reproduces bit for bit
EXP2_PLAN_SHA256 = 'f0b9fe2b9ddc76a2ffaa0c170cf13d0392e5e68f7a65a4dcdfbf190e5a31a184'   # Next Study - Beat-Accurate Synthetic ECG/EXP2_PREREG.md
S_FALLBACK_FAIL = 0.05    # an arm with more than 5% of recordings on the equal-time fallback is reported as failed
N_SPLITS = 10
GATE_TOL = 3.0            # macro-F1 points allowed between this run and the reference values
CHUNK = 400               # recordings per cache file
N_MIN_CYCLES = 2          # a recording is kept only when every arm gives at least this many cycles
DRY_RUN_SUBSET = dict(present=60, absent=60)   # DRY_RUN sample: murmur-present recordings drawn round-robin over the grading / timing / shape classes, plus murmur-absent ones

if DRY_RUN:
    N_BOOT = 100
    N_SPLITS = 5
VALIDATION_APPLICABLE = (not DRY_RUN) and LIMIT_RECORDINGS == 0    # the reference values exist for the full set only
print('DRY_RUN', DRY_RUN, '| LIMIT_RECORDINGS', LIMIT_RECORDINGS, '| N_BOOT', N_BOOT, '| N_SPLITS', N_SPLITS,
      '| N_MIN_CYCLES', N_MIN_CYCLES, '| RESUME_DIR', RESUME_DIR, '| SEED', SEED, '| validation gate applicable', VALIDATION_APPLICABLE)

## Environment: libraries, Drive, GPU

This cell mounts Google Drive, installs PyWavelets (the only package Colab lacks), imports every library the notebook needs, prints the GPU, CPU count and RAM, and fixes the determinism settings.

BLAS thread counts are pinned to one **before** numpy is imported (BLAS is the linear-algebra
library behind numpy; unpinned, every forked feature worker would open its own thread pool). The
feature workers are forked from this process where the platform has the fork start method (Linux,
so Colab) and inherit that setting; where fork does not exist the same code runs serially through
an executor with the same `.map` interface, and the manifest records which path was used. cuDNN
(the GPU convolution library) is held to its deterministic kernels, TF32 (the reduced-precision
matrix arithmetic of Ampere GPUs) is switched off, and `torch.use_deterministic_algorithms` is on,
so the generator trace is reproducible bit for bit on the same GPU class.

In [ ]:
import os, sys
for _v in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
           'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[_v] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'      # what torch.use_deterministic_algorithms needs from cuBLAS

import subprocess
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print('google.colab is not importable: not running in Colab, Drive not mounted')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyWavelets', 'nbformat'], check=True)

import json, time, pickle, gc, collections, platform, hashlib, random, math, datetime, zipfile
import contextlib, warnings
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path
from typing import List, Dict
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.io import wavfile
from scipy import signal as sig
from scipy.fft import rfft, rfftfreq, dct
from scipy.stats import skew, kurtosis, spearmanr, ttest_rel
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import linear_sum_assignment
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
                             average_precision_score, confusion_matrix)
from tqdm.auto import tqdm
import psutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# the verbatim feature functions emit RuntimeWarnings on short segments by design; keep the rest
warnings.simplefilter('ignore', RuntimeWarning)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)   # warn, not raise, on an op without a deterministic kernel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEV = device
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
GAN_BATCH = None    # Run 3 runs no generator
if (not DRY_RUN) and LIMIT_RECORDINGS == 0 and REQUIRE_GPU:
    assert REQUIRE_GPU in GPU_NAME, 'Run 3 must run on an %s (Run 1 class) so arm E reproduces Run 1; this runtime has %s' % (REQUIRE_GPU, GPU_NAME)
N_PROC = max(1, os.cpu_count() or 1)                       # the design brief: every core
_vm = psutil.virtual_memory()
print('python', platform.python_version(), '| torch', torch.__version__, '| numpy', np.__version__,
      '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)
print('GPU', GPU_NAME, '| CPUs', os.cpu_count(), '| RAM %.1f GB total, %.1f GB free' % (_vm.total / 1e9, _vm.available / 1e9))
print('feature workers', N_PROC)

# ---- worker pool: forked processes where the platform has them, else a serial executor with the same interface ----
class _SerialExecutor:
    """Same .map interface as ProcessPoolExecutor, run in-process. Used where the fork start method is unavailable
    (the workers read inherited globals, so the spawn method cannot stand in for fork)."""
    def __init__(self, initializer=None):
        if initializer is not None:
            initializer()
    def __enter__(self):
        return self
    def __exit__(self, *a):
        return False
    def map(self, fn, it, chunksize=1):
        return map(fn, it)

POOL_KIND = 'fork' if 'fork' in mp.get_all_start_methods() else 'sequential'
POOL_USES = []

def _pool(max_workers, initializer=None):
    if not POOL_USES:
        print('   worker pool: %s' % (POOL_KIND if POOL_KIND == 'fork' else 'sequential (fork start method unavailable on %s)' % sys.platform))
    POOL_USES.append(POOL_KIND)
    if POOL_KIND == 'fork':
        if torch.cuda.is_available():
            torch.cuda.synchronize()                          # no pending GPU work crosses the fork
        kw = dict(max_workers=max_workers, mp_context=mp.get_context('fork'))
        if initializer is not None:
            kw['initializer'] = initializer
        return ProcessPoolExecutor(**kw)
    return _SerialExecutor(initializer)

# ---- stage timer, cumulative across sessions of the same run folder ----
STAGE_SECONDS = {}                 # this session
STAGE_SECONDS_PREV = {}            # cumulative seconds of earlier sessions on a resumed folder (loaded in Stage 0)
STAGE_SESSIONS_PREV = []
SESSION_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

def stage_seconds_cumulative():
    cum = dict(STAGE_SECONDS_PREV)
    for k_, v_ in STAGE_SECONDS.items():
        cum[k_] = cum.get(k_, 0.0) + v_
    return cum

def _persist_stage_seconds():
    if 'RUN_DIR' not in globals():
        return
    tmp = str(RUN_DIR / 'stage_seconds.json') + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(dict(sessions=STAGE_SESSIONS_PREV + [SESSION_ID], this_session=STAGE_SECONDS, cumulative=stage_seconds_cumulative()), fh, indent=1)
    os.replace(tmp, str(RUN_DIR / 'stage_seconds.json'))

@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    print('\n' + '=' * 78 + '\n' + name + '\n' + '=' * 78)
    yield
    dt = time.time() - t0
    STAGE_SECONDS[name] = STAGE_SECONDS.get(name, 0.0) + dt
    _persist_stage_seconds()
    print('--- %s: %.1f s (%.1f min)' % (name, dt, dt / 60))

def sha256_file(path, block=1 << 23):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''):
            h.update(b)
    return h.hexdigest()

def git_blob_sha1(path, block=1 << 23):
    """The hash git would assign to this file as a blob."""
    h = hashlib.sha1()
    h.update(('blob %d\0' % os.path.getsize(path)).encode())
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''):
            h.update(b)
    return h.hexdigest()

# ---- cache writes: atomic (temporary file, then os.replace) and hashed at write time, so Stage 11 re-reads nothing ----
CACHE_HASHES = {}      # {path relative to RUN_DIR: {git_blob_sha1, bytes}}; persisted to cache_hashes.json after every write
CACHE_REBUILDS = []    # (file, reason) for every cache file found unreadable or stale and rebuilt

def _record_cache_hash(path):
    if 'RUN_DIR' not in globals():
        return
    path = Path(path)
    try:
        rel = str(path.relative_to(RUN_DIR))
    except ValueError:
        return
    CACHE_HASHES[rel] = dict(git_blob_sha1=git_blob_sha1(path), bytes=os.path.getsize(path))
    tmp = str(RUN_DIR / 'cache_hashes.json') + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(CACHE_HASHES, fh)
    os.replace(tmp, str(RUN_DIR / 'cache_hashes.json'))

def atomic_savez(path, **arrays):
    tmp = str(path) + '.tmp.npz'
    np.savez(tmp, **arrays)
    os.replace(tmp, str(path))
    _record_cache_hash(path)

def atomic_savez_compressed(path, **arrays):
    tmp = str(path) + '.tmp.npz'
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, str(path))
    _record_cache_hash(path)

def atomic_pickle(path, obj):
    tmp = str(path) + '.tmp'
    with open(tmp, 'wb') as fh:
        pickle.dump(obj, fh, protocol=4)
    os.replace(tmp, str(path))
    _record_cache_hash(path)

def atomic_np_save(path, arr):
    tmp = str(path) + '.tmp.npy'
    np.save(tmp, arr)
    os.replace(tmp, str(path))
    _record_cache_hash(path)

def read_cache(fp, fn):
    """fn(z) on the opened .npz -> value, or None when the file is missing, unreadable or stale (fn returned None).
    An unreadable or stale file is removed and listed in CACHE_REBUILDS; the caller rebuilds it."""
    fp = Path(fp)
    if not fp.exists():
        return None
    try:
        with np.load(fp, allow_pickle=False) as z:
            out = fn(z)
        why = None if out is not None else 'stale'
    except Exception as e:
        out, why = None, type(e).__name__
    if why is not None:
        CACHE_REBUILDS.append((fp.name, why))
        print('   %s cache %s: rebuilding' % ('stale' if why == 'stale' else 'unreadable (%s)' % why, fp.name))
        try:
            fp.unlink()
        except OSError:
            pass
    return out

def jsonable(o):
    if isinstance(o, np.generic):
        o = o.item()
    if isinstance(o, np.ndarray):
        return [jsonable(x) for x in o.tolist()]
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, float) and not np.isfinite(o):
        return None
    if isinstance(o, (str, bool, int, float)) or o is None:
        return o
    return str(o)

## Manual override

Leave every value as `None` and the next cell searches Drive. Set a value to pin that path and skip
the search for it. Paths are printed and asserted either way.

In [ ]:
# ====================== MANUAL OVERRIDE (optional) ======================
MANUAL_TRAINING_DATA = None
# e.g. '/content/drive/MyDrive/JBHI-Revision/October Thesis Dataset/CirCor DigiScope/the-circor-digiscope-phonocardiogram-dataset-1.0.3/the-circor-digiscope-phonocardiogram-dataset-1.0.3/training_data'
MANUAL_CHECKPOINT = None
# e.g. '/content/drive/MyDrive/October Thesis Dataset/trail4_checkpoints/trail4_best.pt'
MANUAL_OUTPUT_ROOT = None
# e.g. '/content/drive/MyDrive/JBHI-Revision/R1_Gating_Campaign'

## Stage 0: resolve Drive paths

One walk of Drive finds both targets. The dataset folder is the `training_data` directory with the
**largest** number of paired `.wav` + `.tsv` files (the complete copy has 3163 pairs; a 3155-pair
copy and a 14-file stale folder also exist and lose on that count). The generator checkpoint is
every `trail4_best.pt` on Drive, hashed with SHA-256; the `trail4_checkpoints` copy is used when
present and a loud warning is printed if the copies differ. A pinned `MANUAL_CHECKPOINT` must exist
and be a torch archive, or the cell stops with a clear message. Outputs go to a dated run folder
(suffix `_dry` for a dry run, `_limitN` for `LIMIT_RECORDINGS = N`) unless `RESUME_DIR` points at
an existing one; the folder's `run_flags.json` is written on a fresh run and must equal the current
flags on a resume, so a full run's caches can never be rebuilt by a dry run by mistake.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive'
SKIP_DIRS = {'.Trash', '.ipynb_checkpoints', '__pycache__', 'node_modules'}

def walk_find_multi(root, targets, maxdepth=9):
    """One walk of Drive for several targets. targets: {name: 'dir' | 'file'} -> {name: sorted hits}."""
    hits = {t: [] for t in targets}
    root = os.path.abspath(root)
    if not os.path.isdir(root):
        return hits
    base_depth = root.rstrip(os.sep).count(os.sep)
    for dp, dns, fns in os.walk(root):
        depth = dp.rstrip(os.sep).count(os.sep) - base_depth
        dns[:] = [d for d in dns if d not in SKIP_DIRS and not d.startswith('.')]
        for t, kind in targets.items():
            hits[t] += [os.path.join(dp, x) for x in (dns if kind == 'dir' else fns) if x == t]
        if depth >= maxdepth - 1:
            dns[:] = []
    return {t: sorted(set(v)) for t, v in hits.items()}

def paired_count(td):
    td = Path(td)
    if not td.is_dir():
        return 0
    return len({p.stem for p in td.glob('*.wav')} & {p.stem for p in td.glob('*.tsv')})

RESUMED_FROM = None
with stage('S0 resolve Drive paths'):
    _targets = {}
    if not MANUAL_TRAINING_DATA:
        _targets['training_data'] = 'dir'
    _found = {}
    if _targets:
        print('searching %s once for %s (this can take 1-3 min on Drive)...' % (DRIVE_ROOT, ' and '.join(_targets)))
        _found = walk_find_multi(DRIVE_ROOT, _targets, 9)

    # ---- dataset ----
    cands = [Path(MANUAL_TRAINING_DATA)] if MANUAL_TRAINING_DATA else [Path(p) for p in _found.get('training_data', [])]
    rows = sorted(((paired_count(p), p) for p in cands), key=lambda r: r[0], reverse=True)
    print('training_data candidates:')
    for n_, p_ in rows:
        print('   paired=%5d  %s' % (n_, p_))
    if not rows or rows[0][0] == 0:
        raise FileNotFoundError('no training_data folder with paired wav+tsv files; use the MANUAL OVERRIDE cell')
    N_PAIRS, AUDIO_PATH = rows[0]
    if MANUAL_TRAINING_DATA is None and N_PAIRS < 2500:
        raise RuntimeError('largest copy has only %d pairs; expected ~3163 (complete copy)' % N_PAIRS)
    BASE_PATH = AUDIO_PATH.parent
    METADATA_PATH = BASE_PATH / 'training_data.csv'
    # ---- Extension Run 1: stage the dataset on the Colab disk. Drive's mount dropped ("Transport endpoint is not
    # connected") when the shared-audio stage read thousands of WAV files from it in parallel; the copy runs one file at a
    # time with retries and a forced remount, sizes are checked, and every later read uses the local copy ----
    import shutil as _shutil, errno as _errno
    def _drive_retry(fn, *a, tries=6):
        for _k in range(tries):
            try:
                return fn(*a)
            except OSError as _e:
                if _k == tries - 1:
                    raise
                if _e.errno in (_errno.ENOTCONN, _errno.ECONNABORTED, _errno.EIO) and IN_COLAB:
                    try:
                        from google.colab import drive as _gd
                        _gd.mount('/content/drive', force_remount=True)
                    except Exception as _me:
                        print('   remount attempt failed:', type(_me).__name__)
                time.sleep(3 * (_k + 1))
    AUDIO_PATH_DRIVE = AUDIO_PATH
    if str(AUDIO_PATH).startswith('/content/drive'):
        _local = Path('/content/circor_local/training_data')
        _local.mkdir(parents=True, exist_ok=True)
        _src = _drive_retry(lambda: sorted([p for p in AUDIO_PATH.iterdir() if p.suffix in ('.wav', '.tsv')], key=lambda p: p.name))
        _t0, _n_new = time.time(), 0
        for _p in tqdm(_src, desc='staging CirCor on the Colab disk', leave=False):
            _q = _local / _p.name
            _size = _drive_retry(lambda: _p.stat().st_size)
            if _q.exists() and _q.stat().st_size == _size:
                continue
            _drive_retry(_shutil.copyfile, _p, _q)
            assert _q.stat().st_size == _size, 'size mismatch after copy: %s' % _p.name
            _n_new += 1
        _drive_retry(_shutil.copyfile, METADATA_PATH, _local.parent / 'training_data.csv')
        assert paired_count(_local) == N_PAIRS, 'local copy has %d pairs, Drive has %d' % (paired_count(_local), N_PAIRS)
        AUDIO_PATH, BASE_PATH = _local, _local.parent
        METADATA_PATH = BASE_PATH / 'training_data.csv'
        print('dataset staged on the Colab disk: %d files (%d copied now) in %.0f s -> %s' % (len(_src), _n_new, time.time() - _t0, AUDIO_PATH))

    # ---- Run 3: no generator; the frozen control arm's checkpoint folder ----
    GAN_CHECKPOINT, CKPT_SHA256, CKPT_HASHES = None, None, {}
    TW_SRC = Path(TW_CHECKPOINT_DIR) if TW_CHECKPOINT_DIR else Path(DRIVE_ROOT + '/JBHI-Revision/Extension_Study/checkpoints/' + TW_STAMP)
    assert (TW_SRC / 'freeze.json').exists(), 'frozen checkpoint folder not found: %s' % TW_SRC

    # ---- output folder and run flags ----
    OUT_ROOT = Path(MANUAL_OUTPUT_ROOT or (DRIVE_ROOT + '/JBHI-Revision/Extension_Study/runs/run3'))
    if RESUME_DIR:
        RUN_DIR = Path(RESUME_DIR)
        if not RUN_DIR.is_dir():
            raise FileNotFoundError('RESUME_DIR does not exist: %s' % RUN_DIR)
        RESUMED_FROM = str(RUN_DIR)
    else:
        _suffix = '_dry' if DRY_RUN else ('_limit%d' % LIMIT_RECORDINGS if LIMIT_RECORDINGS > 0 else '')
        RUN_DIR = OUT_ROOT / (datetime.datetime.now().strftime('%Y%m%d_%H%M') + _suffix)
    # the folder suffix must agree with the flags before anything is created or written in the folder
    assert RUN_DIR.name.endswith('_dry') == DRY_RUN, 'DRY_RUN and the run folder suffix disagree: %s' % RUN_DIR.name
    if LIMIT_RECORDINGS > 0 and not DRY_RUN:
        assert RUN_DIR.name.endswith('_limit%d' % LIMIT_RECORDINGS), 'LIMIT_RECORDINGS and the run folder suffix disagree: %s' % RUN_DIR.name
    FOLD_DIR, CACHE_DIR, FIG_DIR = RUN_DIR / 'folds', RUN_DIR / 'cache', RUN_DIR / 'figures'
    CV_DIR = CACHE_DIR / 'cv'
    for d_ in (RUN_DIR, FOLD_DIR, CACHE_DIR, FIG_DIR, CV_DIR):
        d_.mkdir(parents=True, exist_ok=True)
    RUN_FLAGS = dict(dry_run=DRY_RUN, limit_recordings=LIMIT_RECORDINGS, seed=SEED, n_splits=N_SPLITS, chunk=CHUNK,
                     n_min_cycles=N_MIN_CYCLES, dry_run_subset=DRY_RUN_SUBSET if DRY_RUN else None,
                     run='extension_run3_acoustic_control', ensemble_seeds=list(ENSEMBLE_SEEDS), groups='child', prereg_sha256=PREREG_SHA256, tw_stamp=TW_STAMP, exp2_plan_sha256=EXP2_PLAN_SHA256)
    _flags_fp = RUN_DIR / 'run_flags.json'
    if _flags_fp.exists():
        with open(_flags_fp) as fh:
            _prev = json.load(fh)
        assert _prev == RUN_FLAGS, 'RESUME_DIR was produced with different flags %s; refusing to resume with %s' % (_prev, RUN_FLAGS)
        print('run_flags.json matches this session:', RUN_FLAGS)
    else:
        with open(_flags_fp, 'w') as fh:
            json.dump(RUN_FLAGS, fh, indent=1)
    if (RUN_DIR / 'stage_seconds.json').exists():
        with open(RUN_DIR / 'stage_seconds.json') as fh:
            _ss = json.load(fh)
        STAGE_SECONDS_PREV.update(_ss.get('cumulative', {}))
        STAGE_SESSIONS_PREV.extend(_ss.get('sessions', []))
    if (RUN_DIR / 'cache_hashes.json').exists():
        with open(RUN_DIR / 'cache_hashes.json') as fh:
            CACHE_HASHES.update(json.load(fh))
    if RESUMED_FROM:
        print('resuming %s: %d earlier session(s), %d cache files already hashed at write time' % (RUN_DIR, len(STAGE_SESSIONS_PREV), len(CACHE_HASHES)))

    print('\n--- resolved paths ---')
    for k_, v_ in [('AUDIO_PATH', AUDIO_PATH), ('METADATA_PATH', METADATA_PATH), ('TW_SRC', TW_SRC),
                   ('RUN_DIR', RUN_DIR), ('FOLD_DIR', FOLD_DIR), ('CACHE_DIR', CACHE_DIR), ('FIG_DIR', FIG_DIR)]:
        print('%-15s %s' % (k_, v_))
        assert Path(v_).exists(), '%s does not exist: %s' % (k_, v_)
    print('paired wav+tsv files: %d | frozen control-arm folder: %s' % (N_PAIRS, TW_SRC))
    print('run flags:', RUN_FLAGS)

*Run 3: the generator checkpoint is not loaded.*

In [ ]:
GAN_SR = 8000
SIGNAL_LENGTH = 64000   # 8 s at 8 kHz (the GAN's fixed input length)
TARGET_SR = 4000

CONFIG = {
    'target_sr': 4000, 'recording_level_n_fft': 2048, 'n_mfcc_recording': 40,
    'n_mfcc_phase': 13, 'detection_freq_low': 20, 'detection_freq_high': 800,
    'diagnosis_freq_low': 50, 'diagnosis_freq_high': 400, 'envelope_smooth_ms': 50,
    'subphase_boundaries': {'early': (0.0, 0.4), 'mid': (0.2, 0.8), 'late': (0.6, 1.0)},
    'wavelet': 'db4', 'wavelet_levels': 5,
}

def ensure_positive_polarity(ecg):
    if np.abs(ecg.min()) > np.abs(ecg.max()): ecg = -ecg
    return ecg

def segment_phases_samples(r_peaks, sig_len):
    phases = []
    for i in range(len(r_peaks) - 1):
        s, e = int(r_peaks[i]), int(r_peaks[i+1]); rr = e - s
        if rr <= 0: continue
        phases.append({'cycle': i, 's1': (s, s + int(0.15*rr)),
                       'systole': (s + int(0.15*rr), s + int(0.50*rr)),
                       's2': (s + int(0.50*rr), s + int(0.60*rr)),
                       'diastole': (s + int(0.60*rr), e)})
    return phases

In [ ]:
def _fast_mfcc(y, sr, n_mfcc=13, n_fft=512, hop=256, n_mels=40):
    if len(y) < n_fft: y = np.pad(y, (0, n_fft - len(y)))
    f, t, Zxx = sig.stft(y.astype(np.float64), fs=sr, nperseg=n_fft, noverlap=n_fft - hop, window='hann')
    power = np.abs(Zxx) ** 2
    fmin, fmax = 0, sr / 2
    mel_min = 2595 * np.log10(1 + fmin / 700); mel_max = 2595 * np.log10(1 + fmax / 700)
    mel_pts = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_pts = 700 * (10 ** (mel_pts / 2595) - 1)
    bins = np.clip(np.floor((n_fft + 1) * hz_pts / sr).astype(int), 0, n_fft // 2)
    fbank = np.zeros((n_mels, n_fft // 2 + 1))
    for i in range(n_mels):
        left, center, right = bins[i], bins[i+1], bins[i+2]
        if center > left: fbank[i, left:center] = np.linspace(0, 1, center - left, endpoint=False)
        if right > center: fbank[i, center:right] = np.linspace(1, 0, right - center, endpoint=False)
    mel_spec = np.maximum(fbank @ power, 1e-10)
    log_mel = np.log(mel_spec)
    return dct(log_mel, type=2, axis=0, norm='ortho')[:n_mfcc]

def _fast_spectral_features(y, sr, n_fft=2048):
    if len(y) < n_fft: return 0, 0, 0, 0, 0, 0
    spectrum = np.abs(rfft(y[:n_fft])) ** 2; freqs = rfftfreq(n_fft, 1.0 / sr)
    spec_sum = spectrum.sum() + 1e-10
    centroid = np.sum(freqs * spectrum) / spec_sum
    bandwidth = np.sqrt(np.sum(((freqs - centroid) ** 2) * spectrum) / spec_sum)
    cumsum = np.cumsum(spectrum); rolloff = freqs[min(np.searchsorted(cumsum, 0.85 * cumsum[-1]), len(freqs) - 1)]
    flatness = np.exp(np.mean(np.log(spectrum + 1e-10))) / (np.mean(spectrum) + 1e-10)
    zcr = np.sum(np.abs(np.diff(np.sign(y)))) / (2 * len(y)); rms = np.sqrt(np.mean(y ** 2))
    return centroid, bandwidth, rolloff, flatness, zcr, rms

def compute_envelope(segment, sr, fc_low, fc_high, smooth_ms=50):
    if len(segment) < 64: return np.abs(segment)
    nyq = sr / 2; low = max(fc_low / nyq, 0.01); high = min(fc_high / nyq, 0.99)
    try:
        b, a = sig.butter(4, [low, high], btype='band'); filtered = sig.filtfilt(b, a, segment)
        rectified = np.abs(filtered); fc_env = min(10 / nyq, 0.99)
        b_env, a_env = sig.butter(2, fc_env, btype='low'); envelope = sig.filtfilt(b_env, a_env, rectified)
        return gaussian_filter1d(envelope, max(1, int(smooth_ms * sr / 1000)) / 4)
    except: return np.abs(segment)

def compute_envelope_detection(seg, sr): return compute_envelope(seg, sr, CONFIG['detection_freq_low'], CONFIG['detection_freq_high'])
def compute_envelope_diagnosis(seg, sr): return compute_envelope(seg, sr, CONFIG['diagnosis_freq_low'], CONFIG['diagnosis_freq_high'])

def extract_recording_level_features(pcg, sr):
    features = {}; n_fft = CONFIG['recording_level_n_fft']; n_mfcc = CONFIG['n_mfcc_recording']
    def zeros():
        for s in ['mean','std','min','max']:
            for i in range(n_mfcc): features[f'rec_mfcc_{i}_{s}'] = 0
        for i in range(13): features[f'rec_mfcc_delta_{i}_mean'] = 0; features[f'rec_mfcc_delta_{i}_std'] = 0
        for k in ['rec_spectral_centroid_mean','rec_spectral_centroid_std','rec_spectral_bandwidth_mean','rec_spectral_bandwidth_std','rec_spectral_rolloff_mean','rec_spectral_flatness_mean','rec_zcr_mean','rec_zcr_std','rec_rms_mean','rec_rms_std']: features[k] = 0
    if len(pcg) < n_fft: zeros(); return features
    try:
        mfccs = _fast_mfcc(pcg, sr, n_mfcc=n_mfcc, n_fft=n_fft, hop=n_fft//4, n_mels=64)
        for i in range(n_mfcc):
            features[f'rec_mfcc_{i}_mean'] = np.mean(mfccs[i]); features[f'rec_mfcc_{i}_std'] = np.std(mfccs[i])
            features[f'rec_mfcc_{i}_min'] = np.min(mfccs[i]); features[f'rec_mfcc_{i}_max'] = np.max(mfccs[i])
        delta = np.diff(mfccs[:13], axis=1)
        for i in range(13):
            features[f'rec_mfcc_delta_{i}_mean'] = np.mean(delta[i]) if delta.shape[1] > 0 else 0
            features[f'rec_mfcc_delta_{i}_std'] = np.std(delta[i]) if delta.shape[1] > 0 else 0
        c, bw, ro, fl, zcr, rms = _fast_spectral_features(pcg, sr, n_fft)
        features['rec_spectral_centroid_mean'] = c; features['rec_spectral_centroid_std'] = 0
        features['rec_spectral_bandwidth_mean'] = bw; features['rec_spectral_bandwidth_std'] = 0
        features['rec_spectral_rolloff_mean'] = ro; features['rec_spectral_flatness_mean'] = fl
        features['rec_zcr_mean'] = zcr; features['rec_zcr_std'] = 0
        features['rec_rms_mean'] = rms; features['rec_rms_std'] = 0
    except: zeros()
    return features

def extract_soft_murmur_features(pcg, sr):
    features = {}; keys = ['adaptive_snr_low','adaptive_snr_mid','adaptive_snr_high','spectral_stability','pitch_stability','harmonic_ratio','energy_variance','low_freq_ratio','mid_freq_ratio','high_freq_ratio']
    if len(pcg) < 1024:
        for k in keys: features[k] = 0
        return features
    try:
        nyq = sr / 2; bands = {}
        for name, lo, hi in [('low',25,100),('mid',100,300),('high',300,600)]:
            b, a = sig.butter(4, [lo/nyq, hi/nyq], btype='band'); bands[name] = sig.filtfilt(b, a, pcg)
            features[f'adaptive_snr_{name}'] = np.percentile(np.abs(bands[name]),90) / (np.percentile(np.abs(bands[name]),25) + 1e-10)
        frame_len = 1024; hop = 256; n_frames = max(1, (len(pcg) - frame_len) // hop)
        fe = np.array([np.sum(pcg[i*hop:i*hop+frame_len]**2) for i in range(n_frames)])
        features['spectral_stability'] = 1 / (np.std(fe) / (np.mean(fe) + 1e-10) + 1e-10)
        features['pitch_stability'] = features['spectral_stability']; features['harmonic_ratio'] = 0.5
        features['energy_variance'] = np.std(fe) / (np.mean(fe) + 1e-10)
        total = sum(np.sum(v**2) for v in bands.values()) + 1e-10
        for name in ['low','mid','high']: features[f'{name}_freq_ratio'] = np.sum(bands[name]**2) / total
    except:
        for k in keys: features[k] = 0
    return features

def extract_phase_specific_mfccs(s1, sys, s2, dia, sr):
    features = {}; n = CONFIG['n_mfcc_phase']; pm = {}
    for pn, seg in [('s1',s1),('systole',sys),('s2',s2),('diastole',dia)]:
        if len(seg) < 64:
            for i in range(n): features[f'{pn}_mfcc_{i}'] = 0
            pm[pn] = np.zeros(n); continue
        n_fft = min(512, len(seg) - len(seg) % 2)
        try:
            mm = np.mean(_fast_mfcc(seg, sr, n_mfcc=n, n_fft=n_fft, hop=n_fft//2), axis=1)
            for i in range(n): features[f'{pn}_mfcc_{i}'] = mm[i]
            pm[pn] = mm
        except:
            for i in range(n): features[f'{pn}_mfcc_{i}'] = 0
            pm[pn] = np.zeros(n)
    try:
        features['mfcc_systole_diastole_dist'] = np.linalg.norm(pm['systole'] - pm['diastole'])
        features['mfcc_systole_s1_dist'] = np.linalg.norm(pm['systole'] - pm['s1'])
        features['mfcc_systole_s2_dist'] = np.linalg.norm(pm['systole'] - pm['s2'])
        features['mfcc_diastole_hs_dist'] = np.linalg.norm(pm['diastole'] - (pm['s1'] + pm['s2'])/2)
    except:
        for k in ['mfcc_systole_diastole_dist','mfcc_systole_s1_dist','mfcc_systole_s2_dist','mfcc_diastole_hs_dist']: features[k] = 0
    return features

def extract_diastolic_features(seg, sr):
    features = {}; ks = ['diastole_rms','diastole_peak','diastole_spectral_centroid','diastole_energy_ratio','diastole_spectral_bandwidth','diastole_spectral_rolloff','diastole_zcr']
    if len(seg) < 64:
        for k in ks: features[k] = 0
        return features
    try:
        features['diastole_rms'] = np.sqrt(np.mean(seg**2)); features['diastole_peak'] = np.max(np.abs(seg))
        c, bw, ro, fl, zcr, _ = _fast_spectral_features(seg, sr, min(512, len(seg)))
        features['diastole_spectral_centroid'] = c; features['diastole_spectral_bandwidth'] = bw
        features['diastole_spectral_rolloff'] = ro; features['diastole_zcr'] = zcr
        env = compute_envelope_detection(seg, sr); features['diastole_energy_ratio'] = np.sum(env > np.max(env)*0.2) / len(env)
    except:
        for k in ks: features[k] = 0
    return features

def extract_absolute_amplitude_features(pcg, s1, sys, s2, dia, sr):
    features = {}; rec_rms = np.sqrt(np.mean(pcg**2)) + 1e-10; rec_max = np.max(np.abs(pcg)) + 1e-10
    for name, seg in [('s1',s1),('systole',sys),('s2',s2),('diastole',dia)]:
        features[f'abs_{name}_rms'] = np.sqrt(np.mean(seg**2)) if len(seg) > 0 else 1e-10
    features['rel_systole_intensity'] = features['abs_systole_rms'] / rec_rms
    features['rel_diastole_intensity'] = features['abs_diastole_rms'] / rec_rms
    for name, seg in [('systole',sys),('diastole',dia)]:
        features[f'abs_{name}_peak'] = np.max(np.abs(seg)) if len(seg) > 0 else 1e-10
        features[f'rel_{name}_peak'] = features[f'abs_{name}_peak'] / rec_max
    noise = np.percentile(np.abs(pcg), 10) + 1e-10
    features['systole_snr'] = features['abs_systole_peak'] / noise; features['diastole_snr'] = features['abs_diastole_peak'] / noise
    features['total_murmur_energy'] = features['abs_systole_rms'] + features['abs_diastole_rms']
    return features

def extract_boundary_features(s1, sys, s2, sr):
    features = {}; bs = int(20 * sr / 1000)
    for prefix, a_, b_ in [('s1_systole', s1, sys), ('systole_s2', sys, s2)]:
        if len(a_) > bs and len(b_) > bs:
            tail, head = a_[-bs:], b_[:bs]
            features[f'{prefix}_transition_rms'] = np.sqrt(np.mean(np.concatenate([tail, head])**2))
            c = np.corrcoef(np.abs(tail), np.abs(head))[0,1] if len(tail)==len(head) else 0
            features[f'{prefix}_continuity'] = c if not np.isnan(c) else 0
        else:
            features[f'{prefix}_transition_rms'] = 0; features[f'{prefix}_continuity'] = 0
    return features

def extract_systolic_diagnosis_features(seg, sr):
    features = {}
    if len(seg) < 64:
        for k in ['timing_occupation_ratio','timing_s1_gap','timing_s2_gap','timing_holosystolic_score','shape_monotonicity','shape_convexity','shape_plateau_cv','shape_time_to_peak','shape_skewness','shape_kurtosis']: features[k] = 0
        for i in range(10): features[f'envelope_sample_{i}'] = 0
        return features
    env = compute_envelope_diagnosis(seg, sr); env_n = env / (np.max(env) + 1e-10); above = env_n > 0.1
    if np.sum(above) > 0:
        on = np.argmax(above); off = len(env_n) - np.argmax(above[::-1])
        features['timing_s1_gap'] = on / len(env_n); features['timing_s2_gap'] = (len(env_n) - off) / len(env_n)
        features['timing_occupation_ratio'] = np.sum(above) / len(env_n)
        if features['timing_occupation_ratio'] > 0.85:
            cv = np.std(env_n[above]) / (np.mean(env_n[above]) + 1e-10); features['timing_holosystolic_score'] = max(0, 1 - cv)
        else: features['timing_holosystolic_score'] = features['timing_occupation_ratio'] * 0.5
    else:
        features['timing_s1_gap'] = 0.5; features['timing_s2_gap'] = 0.5; features['timing_occupation_ratio'] = 0; features['timing_holosystolic_score'] = 0
    corr, _ = spearmanr(np.arange(len(env_n)), env_n); features['shape_monotonicity'] = corr if not np.isnan(corr) else 0
    idx = np.linspace(0, len(env_n)-1, 5).astype(int)
    features['shape_convexity'] = np.mean(np.diff(np.diff(env_n[idx])))
    features['shape_plateau_cv'] = np.std(env_n) / (np.mean(env_n) + 1e-10)
    features['shape_time_to_peak'] = np.argmax(env_n) / len(env_n)
    features['shape_skewness'] = float(skew(env_n)); features['shape_kurtosis'] = float(kurtosis(env_n))
    for i, ix in enumerate(np.linspace(0, len(env_n)-1, 10).astype(int)): features[f'envelope_sample_{i}'] = env_n[ix]
    for k in features:
        if np.isnan(features[k]): features[k] = 0
    return features

def extract_subphase_features(seg, sr):
    features = {}; n = len(seg)
    if n < 64:
        for p in ['early','mid','late']: features[f'subphase_{p}_energy'] = 0; features[f'subphase_{p}_ratio'] = 0.33
        features['subphase_energy_centroid'] = 0.5; return features
    energies = {}
    for p, (s, e) in CONFIG['subphase_boundaries'].items():
        sub = seg[int(s*n):int(e*n)]; energies[p] = np.sqrt(np.mean(sub**2)) if len(sub) > 0 else 0
        features[f'subphase_{p}_energy'] = energies[p]
    total = sum(energies.values()) + 1e-10
    for p in CONFIG['subphase_boundaries']: features[f'subphase_{p}_ratio'] = energies[p] / total
    env = compute_envelope_diagnosis(seg, sr)
    features['subphase_energy_centroid'] = np.sum(np.arange(len(env)) * env) / (np.sum(env) * len(env) + 1e-10)
    return features

def extract_cross_phase_features(s1, sys, s2, dia, sr):
    features = {}
    rms = {n: np.sqrt(np.mean(s**2)) if len(s) > 0 else 1e-10 for n, s in [('s1',s1),('systole',sys),('s2',s2),('diastole',dia)]}
    features['cross_murmur_to_s1'] = np.clip(rms['systole']/(rms['s1']+1e-10), 0.01, 100)
    features['cross_murmur_to_s2'] = np.clip(rms['systole']/(rms['s2']+1e-10), 0.01, 100)
    features['cross_murmur_to_hs'] = np.clip(rms['systole']/((rms['s1']+rms['s2'])/2+1e-10), 0.01, 100)
    features['cross_systole_to_diastole'] = np.clip(rms['systole']/(rms['diastole']+1e-10), 0.01, 100)
    features['cross_murmur_prominence'] = rms['systole']/(rms['s1']+rms['s2']+rms['diastole']+1e-10)
    features['cross_diastole_to_systole'] = np.clip(rms['diastole']/(rms['systole']+1e-10), 0.01, 100)
    return features

def extract_wavelet_features(seg, sr):
    features = {}; levels = CONFIG['wavelet_levels']
    if len(seg) < 64:
        for i in range(levels + 1): features[f'wavelet_level_{i}'] = 0
        features['wavelet_murmur_to_hs'] = 1; return features
    try:
        coeffs = pywt.wavedec(seg, CONFIG['wavelet'], level=levels)
        for i, c in enumerate(coeffs): features[f'wavelet_level_{i}'] = np.sum(c**2) / (len(c) + 1e-10)
        if len(coeffs) > 5:
            me = np.sum(coeffs[2]**2) + np.sum(coeffs[3]**2); he = np.sum(coeffs[4]**2) + np.sum(coeffs[5]**2) + 1e-10
            features['wavelet_murmur_to_hs'] = np.clip(me / he, 0.01, 100)
        else: features['wavelet_murmur_to_hs'] = 1
    except:
        for i in range(levels + 1): features[f'wavelet_level_{i}'] = 0
        features['wavelet_murmur_to_hs'] = 1
    return features

def extract_cycle_variability_features(cfl):
    features = {}
    if len(cfl) < 2:
        for k in ['cycle_rms_cv','cycle_rms_range','cycle_spectral_cv','cycle_spectral_range','cycle_timing_cv','cycle_timing_range','cycle_consistency','n_cycles']: features[k] = 0
        features['n_cycles'] = len(cfl); return features
    for name, key in [('rms','abs_systole_rms'),('spectral','systole_mfcc_0'),('timing','timing_occupation_ratio')]:
        vals = [c.get(key, 0) for c in cfl]; vals = [v for v in vals if not np.isnan(v) and v != 0]
        if len(vals) > 1:
            features[f'cycle_{name}_cv'] = np.std(vals) / (np.mean(vals) + 1e-10); features[f'cycle_{name}_range'] = np.max(vals) - np.min(vals)
        else: features[f'cycle_{name}_cv'] = 0; features[f'cycle_{name}_range'] = 0
    cvs = [features.get(f'cycle_{n}_cv', 0) for n in ['rms','spectral','timing']]
    features['cycle_consistency'] = 1 - np.mean(cvs); features['n_cycles'] = len(cfl)
    return features

def extract_all_v4_features(pcg_signal, cycles, sr):
    """cycles: list of dicts with s1/systole/s2/diastole as SAMPLE-INDEX (start,end) tuples."""
    all_features = {}
    all_features.update(extract_recording_level_features(pcg_signal, sr))
    all_features.update(extract_soft_murmur_features(pcg_signal, sr))
    cfl = []
    for cycle in cycles:
        cf = {}
        def get_seg(s, e):
            si, ei = max(0, int(s)), min(len(pcg_signal), int(e)); return pcg_signal[si:ei]
        s1 = get_seg(*cycle['s1']); sys_ = get_seg(*cycle['systole']); s2 = get_seg(*cycle['s2']); dia = get_seg(*cycle['diastole'])
        cf.update(extract_phase_specific_mfccs(s1, sys_, s2, dia, sr))
        cf.update(extract_diastolic_features(dia, sr))
        cf.update(extract_absolute_amplitude_features(pcg_signal, s1, sys_, s2, dia, sr))
        cf.update(extract_boundary_features(s1, sys_, s2, sr))
        cf.update(extract_systolic_diagnosis_features(sys_, sr))
        cf.update(extract_subphase_features(sys_, sr))
        cf.update(extract_cross_phase_features(s1, sys_, s2, dia, sr))
        cf.update(extract_wavelet_features(sys_, sr))
        d = {n: (cycle[n][1] - cycle[n][0]) / sr for n in ['s1','systole','s2','diastole']}; total = sum(d.values())
        cf['duration_systole_ratio'] = d['systole'] / (total + 1e-10); cf['duration_diastole_ratio'] = d['diastole'] / (total + 1e-10)
        cf['duration_systole_ms'] = d['systole'] * 1000; cf['duration_cycle_ms'] = total * 1000; cf['duration_heart_rate'] = 60 / (total + 1e-10)
        cfl.append(cf)
    if cfl:
        for fn in list(cfl[0].keys()):
            vals = [c[fn] for c in cfl if fn in c]; vals = [v for v in vals if not np.isnan(v) and not np.isinf(v)]
            all_features[f'{fn}_median'] = np.median(vals) if vals else 0
            all_features[f'{fn}_std'] = np.std(vals) if len(vals) > 1 else 0
    all_features.update(extract_cycle_variability_features(cfl))
    return all_features

In [ ]:
def parse_tsv_annotations(tsv_path):
    """Returns cycles with s1/systole/s2/diastole as (start,end) in SECONDS."""
    cycles = []; current = {}
    with open(tsv_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3: continue
            s, e, p = float(parts[0]), float(parts[1]), int(parts[2])
            if p == 0: continue
            if p == 1:
                if len(current) == 4: cycles.append(current)
                current = {'s1': (s, e)}
            elif p == 2 and 's1' in current and 'systole' not in current: current['systole'] = (s, e)
            elif p == 3 and 'systole' in current and 's2' not in current: current['s2'] = (s, e)
            elif p == 4 and 's2' in current and 'diastole' not in current: current['diastole'] = (s, e)
    if len(current) == 4: cycles.append(current)
    return cycles

def tsv_cycles_to_samples(tsv_path, sr=4000):
    out = []
    for c in parse_tsv_annotations(tsv_path):
        out.append({'cycle': len(out),
                    's1': (int(c['s1'][0]*sr), int(c['s1'][1]*sr)),
                    'systole': (int(c['systole'][0]*sr), int(c['systole'][1]*sr)),
                    's2': (int(c['s2'][0]*sr), int(c['s2'][1]*sr)),
                    'diastole': (int(c['diastole'][0]*sr), int(c['diastole'][1]*sr))})
    return out

def equal_time_cycles(n_samples, sr=4000, bpm=75):
    """No cardiac gating: equal-length cycles, each split into 4 equal phases (ablation)."""
    cycle_len = int(sr * 60 / bpm); n_cycles = max(1, n_samples // cycle_len); out = []
    for i in range(n_cycles):
        s = i*cycle_len; e = min(n_samples, (i+1)*cycle_len); rr = e - s
        if rr <= 0: continue
        out.append({'cycle': i, 's1': (s, s+int(0.25*rr)), 'systole': (s+int(0.25*rr), s+int(0.50*rr)),
                    's2': (s+int(0.50*rr), s+int(0.75*rr)), 'diastole': (s+int(0.75*rr), e)})
    return out

def fast_load_audio(wav_path, target_sr=4000):
    sr_raw, audio = wavfile.read(wav_path); audio = audio.astype(np.float32)
    if audio.ndim > 1: audio = audio[:, 0]
    if sr_raw != target_sr: audio = sig.resample(audio, int(len(audio) * target_sr / sr_raw))
    return audio / (np.max(np.abs(audio)) + 1e-10), target_sr

## Reused machinery from the diagnostic notebooks

Extracted verbatim (by function name, with the `ast` module) from the mechanism study, the
peak-detection diagnostic, the jitter-tolerance sweep, the fair-MLP bootstrap rerun and the
published gating ablation:

* `_sos_env`, `_detect_sos`: the second-order-section R-peak detector with a finite check.
* `phases_from`: clamps a peak train to the signal before fixed-fraction segmentation, so no
  boundary can exceed the signal length.
* `cycles_to_peaks`, `rr_intervals`, `rhythm`: RR statistics over contiguous cycles only.
* `match_one_to_one`, `match_stats`, `best_offset`: one-to-one peak matching and the offset search.
* `jitter_cycles`, `equivalent_sigma`: the timing-budget perturbation and its inverse read-off.
* `boot_ci`, `boot_diff_ci`: the original patient-level bootstrap, kept as an oracle. The
  vectorised bootstrap used below draws the same patient resamples and is checked against it.
* `get_nongated_cycles`: the published equal-time control (arm E2), boundaries in seconds.

In [ ]:
SOS = sig.butter(4, [5, 30], btype='bandpass', fs=GAN_SR, output='sos')
PHASE_KEYS = ('s1', 'systole', 's2', 'diastole')
np.seterr(over='ignore', invalid='ignore')

def _sos_env(ecg, fs):
    f = sig.sosfiltfilt(SOS, ecg.astype(np.float64))
    w = int(0.08 * fs)
    return np.convolve(f ** 2, np.ones(w) / w, mode='same')


def _detect_sos(ecg, fs, prom=(0.3, 0.15, 0.08, 0.03), min_ms=300):
    env = _sos_env(ecg, fs)
    ref = np.std(env)
    if not (np.isfinite(ref) and ref > 0):
        return np.array([], int)
    md = int(min_ms / 1000 * fs)
    for pf in prom:
        pk, _ = sig.find_peaks(env, distance=md, prominence=ref * pf)
        if len(pk) >= 3:
            return pk
    return np.array([], int)


def phases_from(peaks, n_samples):
    """Clamp to the signal, then segment. segment_phases_samples ignores sig_len."""
    pk = [x for x in peaks if 0 <= x <= n_samples]
    return segment_phases_samples(pk, n_samples) if len(pk) > 2 else []


def cycles_to_peaks(cyc, contiguous=True, tol=2):
    """Peak train from a cycle list. With contiguous=True, an interval is emitted only
    when consecutive cycles actually abut, so annotation gaps do not become RR intervals."""
    if not cyc:
        return []
    if not contiguous:
        return sorted(set([int(c['s1'][0]) for c in cyc] + [int(cyc[-1]['diastole'][1])]))
    runs, cur = [], [int(cyc[0]['s1'][0]), int(cyc[0]['diastole'][1])]
    for a, b in zip(cyc, cyc[1:]):
        if abs(int(b['s1'][0]) - int(a['diastole'][1])) <= tol:
            cur.append(int(b['diastole'][1]))
        else:
            runs.append(cur)
            cur = [int(b['s1'][0]), int(b['diastole'][1])]
    runs.append(cur)
    return runs if False else sorted(set(x for run in runs for x in run))


def rr_intervals(cyc, tol=2):
    """RR intervals across contiguous cycles only."""
    if len(cyc) < 2:
        return np.array([])
    out = []
    for a, b in zip(cyc, cyc[1:]):
        if abs(int(b['s1'][0]) - int(a['diastole'][1])) <= tol:
            out.append(int(b['s1'][0]) - int(a['s1'][0]))
    return np.array([x for x in out if x > 0], float)


def rhythm(cyc, sr=None, tol=2):
    sr = sr or TARGET_SR
    rr = rr_intervals(cyc, tol)
    if len(rr) < 3:
        return dict(cv=np.nan, jump=np.nan, within20=np.nan, fast=np.nan,
                    n=len(cyc), bpm=np.nan, rr_ms=np.nan)
    dr = np.abs(np.diff(rr)) / np.maximum(rr[:-1], 1e-9)
    med = float(np.median(rr))
    return dict(cv=float(rr.std() / rr.mean()),
                jump=float(np.mean(dr > 0.5)),
                within20=float(np.mean(np.abs(rr - med) <= 0.2 * med)),
                fast=float(np.mean(rr / sr * 1000.0 < 333.0)),
                n=len(cyc), bpm=60.0 / (med / sr), rr_ms=med / sr * 1000.0)


def match_one_to_one(a, b, tol):
    """Greedy nearest one-to-one match of a onto b. Returns (n_matched, deltas)."""
    if len(a) == 0 or len(b) == 0:
        return 0, np.array([])
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    order = np.argsort(np.abs(a[:, None] - b[None, :]).min(axis=1))
    usedb = np.zeros(len(b), bool)
    n, d = 0, []
    for i in order:
        dist = np.abs(b - a[i])          # float, so np.inf masking is safe
        dist[usedb] = np.inf
        j = int(np.argmin(dist))
        if dist[j] <= tol:
            usedb[j] = True
            n += 1
            d.append(float(a[i] - b[j]))
    return n, np.array(d)


def match_stats(syn, exp, tol_samples, offset=0):
    m, d = match_one_to_one(syn - offset, exp, tol_samples)
    recall = m / len(exp) if len(exp) else np.nan
    prec = m / len(syn) if len(syn) else np.nan
    f1 = 2 * recall * prec / (recall + prec) if (recall + prec) > 0 else 0.0
    return recall, prec, f1, d


def best_offset(pk_of, names, tol_ms=75, lo=-250, hi=250, step=5):
    """Offset that maximises median F1. A median of matched deltas is not usable here:
    with several times too many peaks the match is dominated by spurious pairs, so the
    'offset' it returns describes the contamination rather than the physiology."""
    ts = int(tol_ms / 1000.0 * TARGET_SR)
    best = (0, -1.0)
    curve = []
    for om in range(lo, hi + 1, step):
        off = int(round(om / 1000.0 * TARGET_SR))
        f = [match_stats(pk_of(n), EXP_PK[n], ts, off)[2] for n in names]
        m = float(np.median(f))
        curve.append((om, m))
        if m > best[1]:
            best = (om, m)
    return best[0], best[1], curve


def jitter_cycles(cycles, sigma_ms, sr, rng, n_samples):
    if sigma_ms <= 0:
        return cycles
    sd = sigma_ms / 1000.0 * sr
    out = []
    for c in cycles:
        d = int(round(rng.normal(0.0, sd)))
        nc, ok = {'cycle': len(out)}, True
        for k in PHASE_KEYS:
            a, b = c[k]
            a2, b2 = a + d, b + d
            if a2 < 0 or b2 > n_samples or b2 <= a2:
                ok = False
                break
            nc[k] = (a2, b2)
        if ok:
            out.append(nc)
    return out


def equivalent_sigma(sigmas, f1s, target):
    s = np.asarray(sigmas, dtype=float)
    f = np.asarray(f1s, dtype=float)
    if target is None or not np.isfinite(target):
        return float('nan')
    if target >= f[0]:
        return 0.0
    if target <= f[-1]:
        return float('inf')
    for i in range(len(s) - 1):
        if f[i] >= target >= f[i + 1]:
            if f[i] == f[i + 1]:
                return float(s[i])
            frac = (f[i] - target) / (f[i] - f[i + 1])
            return float(s[i] + frac * (s[i + 1] - s[i]))
    return float('nan')


def boot_ci(y, pred, groups, metric='f1', n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    pats = np.unique(groups)
    idx_by_pat = {p: np.where(groups == p)[0] for p in pats}
    fn = (lambda a, b: f1_score(a, b, average='macro')) if metric == 'f1' else accuracy_score
    stats = []
    for _ in range(n_boot):
        take = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by_pat[p] for p in take])
        if len(np.unique(y[idx])) < 2:
            continue
        stats.append(fn(y[idx], pred[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return fn(y, pred) * 100, lo * 100, hi * 100


def boot_diff_ci(y, pred_a, pred_b, groups, metric='f1', n_boot=2000, seed=SEED):
    """Paired difference A - B with a patient-level bootstrap CI."""
    rng = np.random.default_rng(seed)
    pats = np.unique(groups)
    idx_by_pat = {p: np.where(groups == p)[0] for p in pats}
    fn = (lambda a, b: f1_score(a, b, average='macro')) if metric == 'f1' else accuracy_score
    diffs = []
    for _ in range(n_boot):
        take = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by_pat[p] for p in take])
        if len(np.unique(y[idx])) < 2:
            continue
        diffs.append(fn(y[idx], pred_a[idx]) - fn(y[idx], pred_b[idx]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return (fn(y, pred_a) - fn(y, pred_b)) * 100, lo * 100, hi * 100


def get_nongated_cycles(n_samples: int, sr: int, n_real_cycles: int) -> List[Dict]:
    """Synthetic equal-time cycles, same count as TSV cycles, 4×25% phases."""
    total_dur = n_samples / sr
    cycle_dur = total_dur / max(n_real_cycles, 1)
    phase_dur = cycle_dur / 4.0
    return [
        {
            's1'      : (i * cycle_dur,                 i * cycle_dur + phase_dur),
            'systole' : (i * cycle_dur + phase_dur,     i * cycle_dur + 2 * phase_dur),
            's2'      : (i * cycle_dur + 2 * phase_dur, i * cycle_dur + 3 * phase_dur),
            'diastole': (i * cycle_dur + 3 * phase_dur, i * cycle_dur + 4 * phase_dur),
        }
        for i in range(n_real_cycles)
    ]

_pole_sos = float(max(np.abs(np.roots(s[3:])).max() for s in SOS))
print('SOS bandpass 5-30 Hz at %d Hz: max pole radius %.6f (%s)' % (GAN_SR, _pole_sos, 'stable' if _pole_sos < 1 else 'UNSTABLE'))
assert _pole_sos < 1.0, 'the SOS design must be stable'
print('reused functions defined')

## Stage 1: recording index and labels

Labels come from `training_data.csv` with the source's coding, verbatim: murmur `Unknown` is
dropped; grading I / II / III+ is 0 / 1 / 2; timing Holosystolic / Early / Mid is 0 / 1 / 2 with
Late-systolic coded -1; shape Plateau / Decrescendo / Diamond is 0 / 1 / 2 with Crescendo coded -1;
murmur-negative recordings carry -1 on the three characterisation tasks. A recording is indexed only
if it has a WAV, a TSV annotation and a label. Everything excluded is listed with its reason and
saved to `index.json`.

In [ ]:
metadata_df = pd.read_csv(METADATA_PATH)
label_lookup = {}
for _, row in metadata_df.iterrows():
    pid = int(row['Patient ID']); murmur = row.get('Murmur', 'Unknown')
    if murmur == 'Unknown': continue
    labels = {'murmur': 1 if murmur == 'Present' else 0}
    if labels['murmur'] == 1:
        g = str(row.get('Systolic murmur grading', '')); labels['grading'] = 0 if g in ['I/VI','I'] else (1 if g in ['II/VI','II'] else (2 if g in ['III/VI','III'] else -1))
        t = str(row.get('Systolic murmur timing', '')); labels['timing'] = 0 if 'Holosystolic' in t else (1 if 'Early' in t else (2 if 'Mid' in t else -1))
        s = str(row.get('Systolic murmur shape', '')); labels['shape'] = 0 if 'Plateau' in s else (1 if 'Decresc' in s else (2 if 'Diamond' in s else -1))
    else:
        labels['grading'] = -1; labels['timing'] = -1; labels['shape'] = -1
    label_lookup[pid] = labels

In [ ]:
def dry_run_subset(records_all, n_present, n_absent, seed):
    """Class-aware DRY_RUN sample. Murmur-present recordings are drawn round-robin over the nine
    (task, class) buckets of grading / timing / shape, one recording per patient while distinct
    patients last; murmur-absent recordings come from further distinct patients. Deterministic in `seed`."""
    rng = np.random.default_rng(seed)

    def _perm(rows):
        return [rows[i] for i in rng.permutation(len(rows))]

    present = [r for r in records_all if r[2]['murmur'] == 1]
    absent = [r for r in records_all if r[2]['murmur'] == 0]
    buckets = {}
    for r in present:
        for key_ in ('grading', 'timing', 'shape'):
            if r[2][key_] >= 0:
                buckets.setdefault((key_, int(r[2][key_])), []).append(r)
    buckets = {k: _perm(v) for k, v in sorted(buckets.items())}
    chosen, chosen_names, used_pat = [], set(), set()
    for allow_repeat_patient in (False, True):
        progress = True
        while len(chosen) < n_present and progress:
            progress = False
            for key_, rows in buckets.items():
                if len(chosen) >= n_present:
                    break
                for r in rows:
                    if r[0] in chosen_names or (not allow_repeat_patient and r[1] in used_pat):
                        continue
                    chosen.append(r)
                    chosen_names.add(r[0])
                    used_pat.add(r[1])
                    progress = True
                    break
    picked_absent = []
    absent_p = _perm(absent)
    for allow_repeat_patient in (False, True):
        for r in absent_p:
            if len(picked_absent) >= n_absent:
                break
            if r[0] in chosen_names or (not allow_repeat_patient and r[1] in used_pat):
                continue
            picked_absent.append(r)
            chosen_names.add(r[0])
            used_pat.add(r[1])
    sub = sorted(chosen + picked_absent, key=lambda r: r[0])
    print('DRY_RUN class-aware subset: %d murmur-present (round-robin over %d (task, class) buckets) + %d murmur-absent = %d recordings from %d patients' % (
        len(chosen), len(buckets), len(picked_absent), len(sub), len({r[1] for r in sub})))
    for key_ in ('grading', 'timing', 'shape'):
        cnt = collections.Counter(int(r[2][key_]) for r in chosen if r[2][key_] >= 0)
        print('   %-8s classes in the subset: %s | patients per class: %s' % (
            key_, dict(sorted(cnt.items())), {c: len({r[1] for r in chosen if r[2][key_] == c}) for c in sorted(cnt)}))
    return sub

with stage('S1 index and labels'):
    UNKNOWN_PIDS = set(int(p) for p in metadata_df.loc[metadata_df['Murmur'] == 'Unknown', 'Patient ID'])
    AGE_OF = {}
    for _, _row in metadata_df.iterrows():
        _a = _row.get('Age', None)
        AGE_OF[int(_row['Patient ID'])] = _a if isinstance(_a, str) and _a.strip() else 'Unknown'

    EXCLUDED = []
    records_all = []
    # sorted by file name as a plain string: the same order on every platform (Path objects sort case-insensitively on
    # Windows), and the order of the reference run, so the folds drawn from it are the same everywhere
    for wav in sorted(AUDIO_PATH.glob('*.wav'), key=lambda p: p.name):
        name = wav.stem
        try:
            pid = int(name.split('_')[0])
        except ValueError:
            EXCLUDED.append((name, 'filename does not start with a patient id'))
            continue
        if pid not in label_lookup:
            EXCLUDED.append((name, 'murmur label Unknown' if pid in UNKNOWN_PIDS else 'patient absent from training_data.csv'))
            continue
        tsv = AUDIO_PATH / (name + '.tsv')
        if not tsv.exists():
            EXCLUDED.append((name, 'no tsv annotation'))
            continue
        records_all.append((name, pid, label_lookup[pid], wav, tsv))

    print('wav files: %d | indexed (wav + tsv + label): %d | excluded: %d' % (
        len(list(AUDIO_PATH.glob('*.wav'))), len(records_all), len(EXCLUDED)))
    for reason, cnt in collections.Counter(r for _, r in EXCLUDED).most_common():
        print('   excluded %4d  %s' % (cnt, reason))

    if DRY_RUN:
        records = dry_run_subset(records_all, DRY_RUN_SUBSET['present'], DRY_RUN_SUBSET['absent'], SEED)
    elif LIMIT_RECORDINGS and LIMIT_RECORDINGS < len(records_all):
        _pick = np.unique(np.linspace(0, len(records_all) - 1, LIMIT_RECORDINGS).astype(int))
        records = [records_all[i] for i in _pick]
        print('LIMIT_RECORDINGS: using an evenly spaced subset of %d recordings' % len(records))
    else:
        records = list(records_all)

    NAMES_ALL = [r[0] for r in records]
    REC = {r[0]: r for r in records}
    PID_OF = {r[0]: r[1] for r in records}
    LAB_OF = {r[0]: r[2] for r in records}
    TASKS = {'detection': 'murmur', 'grading': 'grading', 'timing': 'timing', 'shape': 'shape'}
    NCLS = {'detection': 2, 'grading': 3, 'timing': 3, 'shape': 3}
    print('recordings in this run: %d from %d patients' % (len(NAMES_ALL), len(set(PID_OF.values()))))
    for t_, key_ in TASKS.items():
        _y = np.array([LAB_OF[n][key_] for n in NAMES_ALL])
        _c = collections.Counter(_y[_y >= 0].tolist())
        print('   %-10s labelled %5d  classes %s' % (t_, int((_y >= 0).sum()), dict(sorted(_c.items()))))

    with open(RUN_DIR / 'index.json', 'w') as fh:
        json.dump({'names': NAMES_ALL, 'patient_ids': [PID_OF[n] for n in NAMES_ALL],
                   'labels': {n: LAB_OF[n] for n in NAMES_ALL},
                   'age_band': {n: AGE_OF.get(PID_OF[n], 'Unknown') for n in NAMES_ALL},
                   'excluded': EXCLUDED, 'n_wav_files': len(list(AUDIO_PATH.glob('*.wav'))),
                   'limit_recordings': LIMIT_RECORDINGS, 'dry_run': DRY_RUN,
                   'dry_run_subset': DRY_RUN_SUBSET if DRY_RUN else None}, fh, indent=1)

## Stage 2: shared audio

Each WAV is read at its native rate and resampled **once** to 8 kHz (the generator's rate). The
analysed portion is every whole 8 s window plus a trailing window when at least 2 s remain; a
shorter tail is dropped, and a recording shorter than 2 s is kept whole and padded to one window at
generation time. The 4 kHz array used by the feature extractor is the native 4 kHz samples of the
analysed portion (every CirCor file is 4 kHz; a file at another rate is resampled once from the
8 kHz portion), max-abs normalised over the analysed portion as the source's loader normalises a
whole file, so on an untrimmed recording it is sample-identical to that loader. Both arrays are
cached per chunk of 400 recordings. Every arm reads only these arrays, so the arms cannot differ in
audio. The 8 kHz arrays are loaded only for the chunks whose generator cache is missing.

In [ ]:
def chunk_slices(names):
    return [(k, names[k * CHUNK:(k + 1) * CHUNK]) for k in range(int(math.ceil(len(names) / CHUNK)))]

def analysed_length_8k(n8):
    """(analysed samples at 8 kHz, number of generator windows)"""
    n_full, tail = divmod(int(n8), SIGNAL_LENGTH)
    if n_full == 0:
        return int(n8), 1                            # short recording: whole thing, one padded window
    if tail >= 2 * GAN_SR:
        return int(n8), n_full + 1                   # trailing window kept (>= 2 s)
    return n_full * SIGNAL_LENGTH, n_full            # tail under 2 s dropped

def load_shared_audio(name):
    sr_raw, audio = wavfile.read(str(REC[name][3]))
    audio = audio.astype(np.float32)
    if audio.ndim > 1:
        audio = audio[:, 0]
    if sr_raw == GAN_SR:
        x8 = audio
    else:
        x8 = sig.resample(audio, int(len(audio) * GAN_SR / sr_raw)).astype(np.float32)
    n8_total = len(x8)
    L8, n_win = analysed_length_8k(n8_total)
    x8 = np.ascontiguousarray(x8[:L8], dtype=np.float32)
    L4 = int(round(L8 * TARGET_SR / GAN_SR))
    if sr_raw == TARGET_SR:
        # native 4 kHz (every CirCor file): the reference loader's own samples cut to the analysed portion, which equals
        # x8[::2] up to float32 rounding; no second resampling of a truncated array
        x4 = audio[:L4].astype(np.float64)
    else:
        x4 = sig.resample(x8.astype(np.float64), L4)
    assert len(x4) == L4, (name, len(x4), L4)
    x4 = (x4 / (np.max(np.abs(x4)) + 1e-10)).astype(np.float32)   # max-abs normaliser over the analysed portion (the reference normalised the whole file)
    meta = dict(sr_raw=int(sr_raw), n_raw=int(len(audio)), n8_total=int(n8_total), L8=int(L8), L4=int(L4),
                n_win=int(n_win), dropped_tail_s=float((n8_total - L8) / GAN_SR), padded_short=bool(L8 < SIGNAL_LENGTH))
    return x8, x4, meta

# per chunk: is the synthetic-ECG cache usable (same recordings, same generator checkpoint)? The 8 kHz audio is loaded
# only for the chunks that must be generated.
def ecg_chunk_matches(z, names):
    return list(z['names']) == list(names) and str(z['ckpt_sha256']) == CKPT_SHA256

ECG_CHUNK_OK = {}
for k, names in chunk_slices(NAMES_ALL):
    ECG_CHUNK_OK[k] = True       # Run 3: no generator, so the 8 kHz audio is never loaded
ECG_CACHE_COMPLETE = all(ECG_CHUNK_OK.values())
print('generator cache: %d of %d chunks complete; 8 kHz audio is loaded only for the incomplete chunks' % (sum(ECG_CHUNK_OK.values()), len(ECG_CHUNK_OK)))

AUDIO8, AUDIO4, AUDIO_META = {}, {}, {}
with stage('S2 shared audio'):
    for k, names in chunk_slices(NAMES_ALL):
        fp = CACHE_DIR / ('audio_%03d.npz' % k)
        need8 = not ECG_CHUNK_OK[k]

        def _load(z, names=names, need8=need8):
            if list(z['names']) != list(names):
                return None
            metas = json.loads(str(z['meta']))
            return {n: (z[n + '__8k'] if need8 else None, z[n + '__4k'], metas[n]) for n in names}

        got = read_cache(fp, _load)
        if got is not None:
            for n, (x8, x4, meta) in got.items():
                AUDIO4[n], AUDIO_META[n] = x4, meta
                if need8:
                    AUDIO8[n] = x8
            print('   chunk %03d: %d recordings loaded from cache' % (k, len(names)))
            continue
        t0 = time.time()
        arrays, metas = {}, {}
        with ThreadPoolExecutor(max_workers=16) as ex:
            for n, (x8, x4, meta) in zip(names, ex.map(load_shared_audio, names)):
                arrays[n + '__8k'] = x8
                arrays[n + '__4k'] = x4
                metas[n] = meta
                AUDIO4[n] = x4
                AUDIO_META[n] = meta
                if need8:
                    AUDIO8[n] = x8
        atomic_savez(fp, names=np.array(names), meta=np.array(json.dumps(metas)), **arrays)
        del arrays
        print('   chunk %03d: %d recordings decoded and cached in %.0f s' % (k, len(names), time.time() - t0))

    for n in NAMES_ALL:
        assert len(AUDIO4[n]) == AUDIO_META[n]['L4'] and AUDIO_META[n]['L4'] >= 1, n
        assert np.all(np.isfinite(AUDIO4[n])), 'non-finite audio in ' + n
    _L4 = np.array([AUDIO_META[n]['L4'] for n in NAMES_ALL]) / TARGET_SR
    _nw = np.array([AUDIO_META[n]['n_win'] for n in NAMES_ALL])
    _drop = np.array([AUDIO_META[n]['dropped_tail_s'] for n in NAMES_ALL])
    print('analysed audio: %.1f min total | per recording median %.1f s (min %.1f, max %.1f)' % (
        _L4.sum() / 60, np.median(_L4), _L4.min(), _L4.max()))
    print('windows per recording: median %d, max %d, total %d | tails dropped (<2 s): %d recordings, %.1f s in all | padded short recordings: %d' % (
        np.median(_nw), _nw.max(), _nw.sum(), int((_drop > 0).sum()), _drop.sum(),
        sum(AUDIO_META[n]['padded_short'] for n in NAMES_ALL)))
    print('native sample rates:', dict(collections.Counter(AUDIO_META[n]['sr_raw'] for n in NAMES_ALL)))
    print('RAM free: %.1f GB' % (psutil.virtual_memory().available / 1e9))

## Stage 3: not run

Run 3 builds no GAN arms.

## Stage 4: boundary arms and rate-fidelity diagnostics

Every arm is a function that returns a list of cycle dictionaries with `s1`, `systole`, `s2`,
`diastole` as 4 kHz sample-index pairs, plus a per-recording diagnostics dictionary. No boundary
may lie outside the analysed signal; this is asserted for every cycle of every arm. Arms A and B
and arms C, D, E, E2 are built in the worker pool (the TSV parsing included); a worker exception is
recorded in `ARM_FAILURES` (persisted to `arm_failures.json`) and the recording is dropped with the
reason "worker error", never as "too few cycles". The arm caches carry a hash of every arm constant,
a code version and the generator checkpoint's SHA-256, so a change to `FUSION`, `JOIN_MS`,
`REALIGN_MS`, `MATCH_MS`, `N_MIN_CYCLES`, to the arm code or to the checkpoint rebuilds them
instead of reusing stale boundaries.

**Arm A, `syn_fixed`.** The generator's windows are z-scored (mean removed, divided by the SD) and
polarity-normalised one by one, so the concatenated trace has a step at every window join. The
detector's 5 to 30 Hz band-pass would turn that step into a spurious peak, so the SOS envelope
(4th-order Butterworth 5 to 30 Hz in second-order sections, `sosfiltfilt`, squared, 80 ms boxcar,
that is a moving average over 80 ms) is computed **per generator window** and concatenated
(`_env_by_window`); the prominence ladder (0.3 / 0.15 / 0.08 / 0.03 of the envelope SD, tried in
turn until at least three peaks appear) and the 300 ms minimum distance are then applied once,
globally, exactly as the verbatim `_detect_sos` applies them. `_detect_sos` itself stays in the
notebook for provenance and is no longer called. If the ladder yields fewer than three peaks the
recording falls to an absolute-envelope detector and that path is recorded (`envelope-fallback`); if
that also fails the path is `none` and the recording is later dropped from the common set. Peaks
within 100 ms of a window join are de-duplicated (the larger envelope value wins), every peak is
realigned to the raw maximum within 40 ms, two realigned peaks closer than 300 ms lose the smaller
one (`n_close_pairs_dropped`), and the fixed 15 / 35 / 10 / 40 %RR phases follow after
`phases_from` clamps the train to the signal.

**Arm B, `syn_fusion`.** Acoustic S1 evidence is the 20 to 150 Hz band of the 4 kHz PCG,
rectified, smoothed with a 20 ms boxcar and normalised by its 95th percentile. Two peak lists are
taken from that envelope: the thinned list (250 ms minimum distance, prominence 0.1), which drives
the insertions, and the rescue list (prominence 0.1, no spacing rule, height at least
`rescue_height` = 0.50 of the p95 level). Each synthetic R-peak searches [R - 40 ms, R + 120 ms]
for the strongest thinned peak; if none is there but a rescue-list maximum is, the candidate is
still supported and anchored at that maximum's 50 % onset (`n_rescued`). The rescue rule exists
because the 250 ms thinning suppresses S1 wherever S2 is louder: with perfect anchors it raises the
support fraction from 0.82 to 0.98 and lowers false fallbacks from 11.7 % to 1.0 %, while wrong
anchors stay unsupported (0.39). When at least half the candidates are supported, unsupported ones
are removed and unmatched thinned peaks (no synthetic candidate within 150 ms) are inserted when
they split a gap of at least 1.6 times the median supported RR into two plausible halves. Otherwise
`fusion_fallback` is set and the supported acoustic onsets are kept as they are: only the removals
and insertions are skipped (a design choice; reverting to the raw synthetic anchors would discard
validated timing). The S1 anchor is the acoustic onset, the last 50 % crossing before the peak,
which sits about 17 ms after the annotated S1 onset. S2 onset is the 50 % crossing of the strongest
envelope local maximum inside [anchor + 0.25 RR, anchor + 0.55 RR] (the 250 ms spacing is a
beat-spacing rule and is not applied inside the systolic search window, because S2 sits 150 to
330 ms after S1 in this cohort); it lands about 18 to 22 ms after the annotated S2 onset and is
mildly murmur-dependent (23 % of cycles more than 40 ms late in murmur-present recordings against
10 % in murmur-absent ones); without one, the fixed-fraction S2 is used. Phases: S1 runs from onset
for min(0.15 RR, 120 ms), systole to the S2 onset, S2 for min(0.10 RR, 100 ms), diastole to the
next anchor. Malformed cycles are dropped and counted. Every constant is fixed in advance; nothing
is tuned on labels.

**Arm C, `pcg_seg`.** The dataset's per-beat annotation via `tsv_cycles_to_samples`, restricted to
cycles that lie inside the analysed portion. The verbatim converter truncates each boundary with
`int(t * sr)` separately, so adjacent annotated phases can overlap or leave a gap of one sample;
`phase_overlaps()` counts these and arm C is checked with the ordering rule only. The fraction of
the analysed portion covered by annotated cycles is recorded as `annotated_coverage_fraction`.

**Arm D, `pcg_seg_fixed`.** The fixed fractions applied inside each annotated cycle's own extent
(S1 onset to diastole end); the next cycle's S1 onset is used as the end only when the two cycles
abut within two samples. Interior annotation gaps are counted (`n_annotation_gaps`) and never
spanned by a pseudo-cycle. This isolates the cost of the fixed phase model from the cost of anchor
timing.

**Arm E, `equal_time`.** 75 bpm pseudo-cycles in four equal quarters (`equal_time_cycles`,
verbatim). This is the mechanism study's control and the one the validation gate checks.

**Arm E2, `equal_time_matched`.** `get_nongated_cycles`, verbatim from the published ablation: the
analysed portion is cut into N equal pseudo-cycles, N being the number of annotated cycles of arm C
inside that portion, each in four equal quarters; seconds are converted to samples with
`int(t * sr)` as that pipeline did, so adjacent pseudo-cycles can differ by one sample at a join.
E2 inherits the annotated **count**, not the annotated rate: wherever annotation covers only part
of the portion its pseudo-cycles run slow, which the diagnostics show next to the coverage
fraction. Stage 4 asserts per recording that the E2 cycle count equals the arm C count.

**Diagnostics.** For every recording and arm, against arm C: cycles, median RR and implied rate,
RR-CV, RR-jump rate (over 50 %), fraction of cycles above 180 bpm, Hungarian one-to-one matching of
cycle starts to annotated S1 onsets at 150 ms (the assignment that minimises the total absolute
offset, each start matched at most once) giving insertion rate, deletion rate and median signed
offset, over-detection ratio, and the recording-level median RR for the across-recording
correlation. Sign convention: `median_offset_ms` is the arm's cycle start minus the annotated S1
onset (negative = the arm precedes S1); `arm_mean_syn_ac_offset_ms` is the acoustic S1 onset
minus the synthetic R-peak (positive = the acoustic onset follows R). Arm A adds the detector path
and the non-finite fraction; arm B adds the support fraction, rescues, insertions, removals, the
fallback flag and the mean absolute synthetic-to-acoustic offset.

In [ ]:
JOIN_MS, REALIGN_MS, MATCH_MS = 100, 40, 150
FUSION = dict(band_hz=(20, 150), env_smooth_ms=20, peak_dist_ms=250, peak_prom=0.10,
              search_lo_ms=-40, search_hi_ms=120, match_ms=150, gap_factor=1.6, support_min=0.5,
              s2_lo=0.25, s2_hi=0.55, s1_frac=0.15, s1_cap_ms=120, s2_frac=0.10, s2_cap_ms=100,
              s1_onset_back_ms=150, s2_onset_back_ms=100, rescue_height=0.50)
AC_SOS = sig.butter(4, list(FUSION['band_hz']), btype='bandpass', fs=TARGET_SR, output='sos')
ARM_CODE_VERSION = 'R1-2026-09-30'          # bumped whenever an arm's construction changes, so earlier arm caches are rebuilt
# the generator checkpoint is part of the hash: boundaries built from another checkpoint's synthetic ECG are never reused
ARM_CFG_HASH = hashlib.sha1(json.dumps(dict(FUSION=FUSION, JOIN_MS=JOIN_MS, REALIGN_MS=REALIGN_MS, MATCH_MS=MATCH_MS,
                                            N_MIN_CYCLES=N_MIN_CYCLES, code=ARM_CODE_VERSION, checkpoint=CKPT_SHA256),
                                       sort_keys=True, default=str).encode()).hexdigest()[:16]

def check_cycles(cyc, L, tag, strict=True):
    """Every boundary inside [0, L], every phase ordered (start <= end), phase starts non-decreasing.
    strict=True additionally requires the eight boundaries to be monotone; that holds for every arm
    built here. The dataset annotation (arm C) occasionally overlaps adjacent phases by a sample or
    two, so arm C and its jittered copies are checked with strict=False and phase_overlaps() counts them."""
    for c in cyc:
        seq = [v for k in PHASE_KEYS for v in c[k]]
        assert all(0 <= v <= L for v in seq), '%s: boundary outside the signal %s (L=%d)' % (tag, seq, L)
        assert all(c[k][0] <= c[k][1] for k in PHASE_KEYS), '%s: phase with end before start %s' % (tag, seq)
        starts = [c[k][0] for k in PHASE_KEYS]
        assert all(a <= b for a, b in zip(starts, starts[1:])), '%s: phase starts not ordered %s' % (tag, seq)
        if strict:
            assert all(a <= b for a, b in zip(seq, seq[1:])), '%s: boundaries not monotone %s' % (tag, seq)
    return cyc

def phase_overlaps(cyc):
    """(number of adjacent-phase overlaps, largest overlap in samples) in an annotated cycle list."""
    n, mx = 0, 0
    for c in cyc:
        for k1, k2 in zip(PHASE_KEYS, PHASE_KEYS[1:]):
            ov = c[k1][1] - c[k2][0]
            if ov > 0:
                n += 1
                mx = max(mx, ov)
    return n, mx

def envelope_fallback_peaks(ecg, fs, min_ms=300):
    w = int(0.05 * fs)
    sm = np.convolve(np.abs(ecg), np.ones(w) / w, mode='same')
    pk, _ = sig.find_peaks(sm, distance=int(min_ms / 1000 * fs), height=np.percentile(sm, 75))
    return pk

def _env_by_window(ecg, fs, n_win):
    """5-30 Hz SOS envelope per generator window, concatenated: the step between independently
    z-scored, independently polarity-flipped windows never enters the filter."""
    return np.concatenate([_sos_env(ecg[w * SIGNAL_LENGTH:(w + 1) * SIGNAL_LENGTH], fs) for w in range(n_win)])

def detect_arm_a(ecg32, fs, n_win):
    """Per-window SOS envelope, one global prominence ladder and minimum distance -> (peaks at 8 kHz, diagnostics).
    The verbatim _detect_sos is kept for provenance; its ladder and distance are reproduced here on the per-window envelope."""
    d = {}
    env = _env_by_window(ecg32, fs, n_win)
    assert len(env) == len(ecg32), (len(env), len(ecg32))
    d['nonfinite_frac'] = float(np.mean(~np.isfinite(env)))
    ref = float(np.std(env))
    md = int(300 / 1000 * fs)
    pk, path = np.array([], int), 'none'
    if np.isfinite(ref) and ref > 0:
        for pf in (0.3, 0.15, 0.08, 0.03):                     # the same ladder as _detect_sos
            cand, _ = sig.find_peaks(env, distance=md, prominence=ref * pf)
            if len(cand) >= 3:
                pk, path = cand, 'primary'
                break
    if len(pk) < 3:
        pk = envelope_fallback_peaks(ecg32, fs)
        path = 'envelope-fallback' if len(pk) >= 3 else 'none'
        if len(pk) < 3:
            pk = np.array([], int)
        env = np.abs(ecg32)
    d['detector_path'] = path
    pk = np.asarray(pk, int)
    tol = int(JOIN_MS / 1000 * fs)
    keep = np.ones(len(pk), bool)
    near = removed = 0
    for j in [w * SIGNAL_LENGTH for w in range(1, n_win)]:
        idx = np.flatnonzero(np.abs(pk - j) <= tol)
        near += len(idx)
        if len(idx) > 1:
            best = idx[int(np.argmax(env[pk[idx]]))]
            for q in idx:
                if q != best:
                    keep[q] = False
                    removed += 1
    pk = pk[keep]
    r = int(REALIGN_MS / 1000 * fs)
    out = []
    for p in pk:
        a, b = max(0, int(p) - r), min(len(ecg32), int(p) + r + 1)
        out.append(a + int(np.argmax(ecg32[a:b])))
    pk = np.array(sorted(set(out)), int)
    # realignment can pull two peaks inside the 300 ms minimum distance: drop the smaller (by envelope) of such a pair
    close_dropped = 0
    while len(pk) >= 2:
        j = np.flatnonzero(np.diff(pk) < md)
        if len(j) == 0:
            break
        j0 = int(j[0])
        pk = np.delete(pk, j0 if env[pk[j0]] < env[pk[j0 + 1]] else j0 + 1)
        close_dropped += 1
    d.update(n_peaks_near_join=int(near), n_join_dedup=int(removed), n_close_pairs_dropped=int(close_dropped), n_peaks=int(len(pk)))
    return pk, d

def arm_a_cycles(name):
    ecg = ECG[name].astype(np.float32)          # float16 cache -> float32 for detection, always
    L4 = AUDIO_META[name]['L4']
    pk8, d = detect_arm_a(ecg, GAN_SR, AUDIO_META[name]['n_win'])
    pk4 = (pk8 * TARGET_SR / GAN_SR).astype(int)
    cyc = phases_from(pk4, L4)
    return check_cycles(cyc, L4, 'syn_fixed ' + name), d, pk4

def acoustic_envelope(x4, sr=TARGET_SR):
    f = sig.sosfiltfilt(AC_SOS, x4.astype(np.float64))
    w = int(FUSION['env_smooth_ms'] / 1000 * sr)
    env = np.convolve(np.abs(f), np.ones(w) / w, mode='same')
    return env / (np.percentile(env, 95) + 1e-12)

def onset_50(env, peak, back):
    """Last sample before `peak` where env crosses 50 % of the peak height, within `back` samples.
    Returns (onset, found)."""
    h = 0.5 * env[peak]
    a = max(0, int(peak) - int(back))
    below = np.flatnonzero(env[a:peak] < h)
    if len(below) == 0:
        return int(a), False
    return int(a + below[-1] + 1), True

def arm_b_cycles(name, pk4):
    x4 = AUDIO4[name]
    L4 = len(x4)
    sr = TARGET_SR
    env = acoustic_envelope(x4)
    ac_pk, _ = sig.find_peaks(env, distance=int(FUSION['peak_dist_ms'] / 1000 * sr), prominence=FUSION['peak_prom'])
    ac_pk = np.asarray(ac_pk, int)
    raw_pk, _ = sig.find_peaks(env, prominence=FUSION['peak_prom'])
    strong = raw_pk[env[raw_pk] >= FUSION['rescue_height']]     # local maxima at >= 50 % of the p95 level, no spacing rule (rescue list)
    back1 = int(FUSION['s1_onset_back_ms'] / 1000 * sr)
    ac_on = [onset_50(env, p, back1) for p in ac_pk]          # (onset sample, crossing found) per thinned acoustic peak
    r4 = np.asarray(pk4, int)
    d = dict(n_syn=int(len(r4)), n_acoustic=int(len(ac_pk)), supported_frac=np.nan, mean_abs_syn_ac_offset_ms=np.nan,
             mean_syn_ac_offset_ms=np.nan, n_removed=0, n_inserted=0, n_rescued=0, fusion_fallback=False, n_onset_fallback=0,
             n_malformed=0, frac_s2_acoustic=np.nan)
    if len(r4) < 2:
        return [], d
    lo, hi = int(FUSION['search_lo_ms'] / 1000 * sr), int(FUSION['search_hi_ms'] / 1000 * sr)
    used = np.zeros(len(ac_pk), bool)
    supported = np.zeros(len(r4), bool)
    anchors, offs, onset_fb, rescued = [], [], 0, 0
    for i, r in enumerate(r4):
        cand = np.flatnonzero((ac_pk >= r + lo) & (ac_pk <= r + hi) & ~used)
        if len(cand):
            j = cand[int(np.argmax(env[ac_pk[cand]]))]
            used[j] = True
            on, found = ac_on[j]
        else:
            resc = strong[(strong >= r + lo) & (strong <= r + hi)]
            if len(resc) == 0:
                anchors.append(int(r))
                continue
            rescued += 1
            on, found = onset_50(env, int(resc[int(np.argmax(env[resc]))]), back1)
        supported[i] = True
        if found:
            anchors.append(int(on))
            offs.append((int(on) - int(r)) / sr * 1000.0)
        else:
            anchors.append(int(r))
            onset_fb += 1
    frac = float(supported.mean())
    d.update(supported_frac=frac, n_onset_fallback=int(onset_fb), n_rescued=int(rescued),
             mean_abs_syn_ac_offset_ms=float(np.mean(np.abs(offs))) if offs else np.nan,
             mean_syn_ac_offset_ms=float(np.mean(offs)) if offs else np.nan)
    if frac >= FUSION['support_min']:
        anc = np.array([a for a, s in zip(anchors, supported) if s], int)
        d['n_removed'] = int((~supported).sum())
        rr_sup = np.diff(anc)
        med = float(np.median(rr_sup)) if len(rr_sup) else np.nan
        ins = 0
        if np.isfinite(med) and med > 0:
            mtol = int(FUSION['match_ms'] / 1000 * sr)
            for j in range(len(ac_pk)):
                if used[j] or np.min(np.abs(r4 - ac_pk[j])) <= mtol:
                    continue
                o = int(ac_on[j][0]) if ac_on[j][1] else int(ac_pk[j])
                k = int(np.searchsorted(anc, o))
                if k == 0 or k == len(anc):
                    continue
                gap = anc[k] - anc[k - 1]
                if gap >= FUSION['gap_factor'] * med and (o - anc[k - 1]) >= 0.5 * med and (anc[k] - o) >= 0.5 * med:
                    anc = np.insert(anc, k, o)
                    ins += 1
        d['n_inserted'] = int(ins)
        anc = list(anc)
    else:
        anc = list(anchors)                  # fallback: the supported acoustic onsets stay; only removals and insertions are skipped
        d['fusion_fallback'] = True
    anc = sorted(set(int(a) for a in anc if 0 <= a <= L4))
    cyc, malformed, s2_ac = [], 0, 0
    s1_cap, s2_cap = int(FUSION['s1_cap_ms'] / 1000 * sr), int(FUSION['s2_cap_ms'] / 1000 * sr)
    back2 = int(FUSION['s2_onset_back_ms'] / 1000 * sr)
    for a, b in zip(anc, anc[1:]):
        rr = b - a
        if rr <= 0:
            malformed += 1
            continue
        s1e = a + min(int(FUSION['s1_frac'] * rr), s1_cap)
        w0, w1 = a + int(FUSION['s2_lo'] * rr), min(a + int(FUSION['s2_hi'] * rr), L4)
        s2on, found = a + int(0.50 * rr), False
        if w1 > w0 + 2:
            loc, _ = sig.find_peaks(env[w0:w1], prominence=FUSION['peak_prom'])
            if len(loc):
                p = w0 + int(loc[int(np.argmax(env[w0:w1][loc]))])
                cand_on, ok = onset_50(env, p, back2)
                if ok:
                    s2on, found = cand_on, True
        s2e = s2on + min(int(FUSION['s2_frac'] * rr), s2_cap)
        if not (a < s1e < s2on < s2e < b):
            malformed += 1
            continue
        s2_ac += int(found)
        cyc.append({'cycle': len(cyc), 's1': (a, s1e), 'systole': (s1e, s2on), 's2': (s2on, s2e), 'diastole': (s2e, b)})
    d.update(n_malformed=int(malformed), frac_s2_acoustic=(s2_ac / len(cyc)) if cyc else np.nan)
    return check_cycles(cyc, L4, 'syn_fusion ' + name), d

def arm_c_cycles(name):
    L4 = AUDIO_META[name]['L4']
    # tsv_cycles_to_samples (verbatim) truncates each boundary with int(t * sr) separately, so adjacent annotated phases
    # can overlap or leave a gap of one sample; phase_overlaps() counts them and arm C is checked with strict=False
    cyc_all = tsv_cycles_to_samples(REC[name][4], TARGET_SR)
    cyc = [c for c in cyc_all if all(0 <= c[k][0] <= L4 and 0 <= c[k][1] <= L4 for k in PHASE_KEYS)]
    n_ov, max_ov = phase_overlaps(cyc)
    covered = sum(int(c['diastole'][1]) - int(c['s1'][0]) for c in cyc)                          # annotated samples inside the portion
    span = (int(cyc[-1]['diastole'][1]) - int(cyc[0]['s1'][0])) if cyc else 0                     # first S1 onset to last diastole end
    return (check_cycles(cyc, L4, 'pcg_seg ' + name, strict=False),
            dict(n_annotated=len(cyc_all), n_beyond_portion=len(cyc_all) - len(cyc), n_phase_overlaps=n_ov, max_phase_overlap_samples=max_ov,
                 annotated_coverage_fraction=float(covered / L4) if L4 else np.nan, annotated_span_fraction=float(span / L4) if L4 else np.nan))

def arm_d_cycles(name, cyc_c):
    """Fixed fractions inside each annotated cycle's own extent; the next S1 onset is the end only when the cycles abut."""
    L4 = AUDIO_META[name]['L4']
    cyc, n_gap = [], 0
    for a, b in zip(cyc_c, list(cyc_c[1:]) + [None]):
        s = int(a['s1'][0])
        abut = b is not None and abs(int(b['s1'][0]) - int(a['diastole'][1])) <= 2
        e = int(b['s1'][0]) if abut else int(a['diastole'][1])
        n_gap += int(b is not None and not abut)
        seg = segment_phases_samples([s, min(e, L4)], L4)
        if seg:
            seg[0]['cycle'] = len(cyc)
            cyc.append(seg[0])
    return check_cycles(cyc, L4, 'pcg_seg_fixed ' + name), dict(n_anchors=len(cyc_c), n_annotation_gaps=int(n_gap))

def arm_e_cycles(name):
    L4 = AUDIO_META[name]['L4']
    return check_cycles(equal_time_cycles(L4, TARGET_SR), L4, 'equal_time ' + name), {}

def arm_e2_cycles(name, cyc_c):
    L4 = AUDIO_META[name]['L4']
    sec = get_nongated_cycles(L4, TARGET_SR, len(cyc_c))
    # seconds -> samples with int(t * sr), as the published pipeline did: adjacent pseudo-cycles can differ by one sample at a join
    cyc = [{'cycle': i, **{k: (int(c[k][0] * TARGET_SR), int(c[k][1] * TARGET_SR)) for k in PHASE_KEYS}} for i, c in enumerate(sec)]
    return check_cycles(cyc, L4, 'equal_time_matched ' + name), dict(n_matched_to_annotation=len(cyc_c))

def match_hungarian(a, b, tol):
    """Optimal one-to-one matching within tol samples -> (n_matched, signed deltas a - b)."""
    if len(a) == 0 or len(b) == 0:
        return 0, np.array([], float)
    D = np.abs(np.asarray(a, float)[:, None] - np.asarray(b, float)[None, :])
    ri, ci = linear_sum_assignment(np.where(D <= tol, D, 1e9))
    ok = D[ri, ci] <= tol
    return int(ok.sum()), (np.asarray(a, float)[ri[ok]] - np.asarray(b, float)[ci[ok]])

def cycle_starts(cyc):
    return np.array(sorted(set(int(c['s1'][0]) for c in cyc)), int)

def rate_diag(cyc, ref_cyc, sr=TARGET_SR):
    st, ref = rhythm(cyc), rhythm(ref_cyc)
    a, b = cycle_starts(cyc), cycle_starts(ref_cyc)
    m, dl = match_hungarian(a, b, int(MATCH_MS / 1000 * sr))
    dl_ms = dl / sr * 1000.0
    return dict(n_cycles=int(len(cyc)), median_rr_ms=st['rr_ms'], bpm=st['bpm'], rr_cv=st['cv'], rr_jump=st['jump'],
                frac_fast=st['fast'], ref_n_cycles=int(len(ref_cyc)), ref_median_rr_ms=ref['rr_ms'], ref_bpm=ref['bpm'],
                n_matched=int(m), insertion_rate=(len(a) - m) / len(a) if len(a) else np.nan,
                deletion_rate=(len(b) - m) / len(b) if len(b) else np.nan,
                median_offset_ms=float(np.median(dl_ms)) if len(dl_ms) else np.nan,
                median_abs_offset_ms=float(np.median(np.abs(dl_ms))) if len(dl_ms) else np.nan,
                overdetection_ratio=len(a) / len(b) if len(b) else np.nan), dl_ms

def _ab_worker(name):
    try:
        cyc_a, d_a, pk4 = arm_a_cycles(name)
        cyc_b, d_b = arm_b_cycles(name, pk4)
        return name, (cyc_a, d_a, pk4, cyc_b, d_b), None
    except Exception as e:
        return name, None, '%s: %s' % (type(e).__name__, e)

def _cd_worker(name):
    try:
        cyc_c, d_c = arm_c_cycles(name)
        cyc_d, d_d = arm_d_cycles(name, cyc_c)
        cyc_e, d_e = arm_e_cycles(name)
        cyc_e2, d_e2 = arm_e2_cycles(name, cyc_c)
        return name, (cyc_c, d_c, cyc_d, d_d, cyc_e, d_e, cyc_e2, d_e2), None
    except Exception as e:
        return name, None, '%s: %s' % (type(e).__name__, e)

ARMS = ['syn_fixed', 'syn_fusion', 'pcg_seg', 'pcg_seg_fixed', 'equal_time', 'equal_time_matched']
ARM_LETTER = {'syn_fixed': 'A', 'syn_fusion': 'B', 'pcg_seg': 'C', 'pcg_seg_fixed': 'D', 'equal_time': 'E', 'equal_time_matched': 'E2'}
ARM_LABEL = {'syn_fixed': 'A synthetic-fixed', 'syn_fusion': 'B synthetic-fusion', 'pcg_seg': 'C PCG segmentation',
             'pcg_seg_fixed': 'D PCG-seg + fixed', 'equal_time': 'E equal-time 75 bpm', 'equal_time_matched': 'E2 equal-time matched'}
EQUAL_TIME_DEFINITIONS = {
    'equal_time': 'equal_time_cycles (verbatim, analysis source): fixed 75 bpm pseudo-cycles, four equal quarters; the mechanism study control used by the validation gate',
    'equal_time_matched': 'get_nongated_cycles (verbatim, ABLATION_GATING_VALIDATION.ipynb): the analysed portion cut into N equal pseudo-cycles, N = annotated cycles of arm C inside that portion, four equal quarters, seconds -> samples by int(t*sr); the published control, which inherits the annotated count and not the annotated rate'}
R1_ARMS = list(ARMS)
S_ARMS = ['springer_wpk', 'springer_pub']
COV_ARMS = ['springer_wpk_annot', 'equal_time_annot']
TW_ARMS = ['timing_net_ac']
TW_COV_ARMS = ['timing_net_ac_annot']
ARMS = ['equal_time'] + TW_ARMS + TW_COV_ARMS                              # classified in Run 3 (E: reproduction control)
CYC_ARMS = ['pcg_seg', 'equal_time', 'equal_time_annot'] + TW_ARMS + TW_COV_ARMS
ARM_LETTER.update({'springer_wpk': 'S', 'springer_pub': 'S_pub', 'springer_wpk_annot': 'S_annot', 'equal_time_annot': 'E_annot'})
ARM_LABEL.update({'springer_wpk': 'S Springer (40-220 bpm, peak)', 'springer_pub': 'S_pub Springer (published HR rule)',
                  'springer_wpk_annot': 'S_annot Springer, annotated stretches', 'equal_time_annot': 'E_annot equal-time, annotated stretches'})
ARM_LETTER.update({'equal_time_annot': 'E_annot', 'timing_net_ac': 'T_ac', 'timing_net_ac_annot': 'T_ac_annot'})
ARM_LABEL.update({'equal_time_annot': 'E_annot equal-time, annotated stretches', 'timing_net_ac': 'T_ac acoustic-label control (frozen)',
                  'timing_net_ac_annot': 'T_ac_annot, annotated stretches'})
print('arm definitions ready: classified', ARMS, '| cycles', CYC_ARMS, '| arm configuration hash', ARM_CFG_HASH)

In [ ]:
# ---------------- Extension Run 1: embedded folds, child map, Springer (label-free) ----------------
import base64 as _b64, zlib as _zlib, types as _types, io as _io
import scipy.io
FOLD_B64 = '...'
assert hashlib.sha256(FOLD_B64.encode()).hexdigest() == 'd9b05b380cb57397c08dbe5a5de8202b739a7be5e0fe76496d4561d311349a94', 'embedded fold data corrupted'
EMBED = json.loads(_zlib.decompress(_b64.b64decode(FOLD_B64)).decode())
EMBED_FOLDS, PINNED_KEEP = EMBED['folds'], EMBED['pinned_keep']

def child_map(meta):
    """Patient ID -> child ID: connected components of Patient ID and Additional ID, labelled by the smallest Patient ID."""
    parent = {int(p): int(p) for p in meta['Patient ID']}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for p, a in zip(meta['Patient ID'], meta['Additional ID']):
        if pd.notna(a) and int(a) in parent:
            ra, rb = find(int(p)), find(int(a))
            if ra != rb:
                parent[max(ra, rb)] = min(ra, rb)
    return {p: find(p) for p in parent}

CHILD_OF = child_map(metadata_df)
print('children: %d patients in %d children (%d enrolled twice)' % (len(CHILD_OF), len(set(CHILD_OF.values())), len(CHILD_OF) - len(set(CHILD_OF.values()))))

def fold_record_hash(names, child, y, fold_id):
    """extstudy.splits.fold_hash: SHA-256 over names (fixed width 11, as in the fold files), child IDs, labels, and fold
    IDs (int64)."""
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(np.array(names, dtype='<U11')).tobytes())
    for a in (child, y, fold_id):
        h.update(np.ascontiguousarray(np.asarray(a, np.int64)).tobytes())
    return h.hexdigest()

for _t, _f in EMBED_FOLDS.items():
    assert fold_record_hash(_f['names'], _f['child'], _f['y'], _f['fold_id']) == _f['sha256'], 'embedded %s folds do not match their hash' % _t
print('embedded child-grouped folds verified against their SHA-256:', {t: f['sha256'][:12] for t, f in EMBED_FOLDS.items()})

# Springer: Panel E's GPL-3 port of the PhysioNet 2016 sample2016 code, executed in its own namespace
SPRINGER_SRC = _b64.b64decode('...').decode('utf-8')
assert hashlib.sha256(SPRINGER_SRC.encode('utf-8')).hexdigest() == '33bf4c10b8ab6d1041c8eb951317c40400ff7cb735d39c1a6e2e0406991499f7'
# written to a real file and imported (numba's on-disk cache needs a source file); the Viterbi forward pass then runs
# as plain Python, the code Panel E checked and the local tests ran (about 0.3 s per recording and variant), not
# numba's compiled copy, whose floating-point results can differ in the last bits and flip a tied path
import importlib.util as _ilu
_sp_path = (Path('/content') if IN_COLAB else RUN_DIR) / 'springer_py_run1.py'
os.environ.setdefault('NUMBA_CACHE_DIR', str(_sp_path.parent / 'numba_cache'))
_sp_path.write_text(SPRINGER_SRC, encoding='utf-8')
_spec = _ilu.spec_from_file_location('springer_py_run1', str(_sp_path))
springer = _ilu.module_from_spec(_spec)
sys.modules['springer_py_run1'] = springer
_spec.loader.exec_module(springer)
if hasattr(springer._viterbi_forward, 'py_func'):
    springer._viterbi_forward = springer._viterbi_forward.py_func
SPRINGER_VITERBI = 'plain Python' if not hasattr(springer._viterbi_forward, 'py_func') else 'compiled'
assert SPRINGER_VITERBI == 'plain Python'
_PARAM_B64 = {"Springer_B_matrix.mat": "TUFUTEFCIDUuMCBNQVQtZmlsZSwgUGxhdGZvcm06IFBDV0lONjQsIENyZWF0ZWQgb246IFR1ZSBGZWIgMjMgMTg6NDE6MjggMjAxNiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAAAAAAAAAAAAABSU0PAAAA+AAAAHic42NgYDjAyMDABqQ5gBjIBANWJD4LlBYE4uCCosy89NSieKf43MSSoswKqHo+II5gQJjDhmYOK9QMmPmcQKwBxA43uG7o5T+xv8azsVlyM+MB4UUuxyfXfLE/Ufls6g35Lft53B69VJ6ycj+55n/VWaU2v267ve7aNdlF2T/tF819/LpB/sX+7CXfFyh2P7Tn2Hx/xZl1F+zJNV/vT0oaZ915+95TKssjZ93enyLDw3Xg14H95hWpBauEL9r7b/GV2DPrPNnu75w493fv4Qv2wiLFyg/z2Rxan5gliXp82e91JVv89rtZ9gIVpjHbre7YAwAqCFy5", "Springer_pi_vector.mat": "TUFUTEFCIDUuMCBNQVQtZmlsZSwgUGxhdGZvcm06IFBDV0lONjQsIENyZWF0ZWQgb246IFR1ZSBGZWIgMjMgMTg6NDE6MjggMjAxNiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAAAAAAAAAAAAABSU0PAAAAPQAAAHic42NgYMgAYjYg5oDSIMAK5TMCMQuUFgLi4IKizLz01KL4gsz4stTkkvwiiHpOIFZggIEL9rhoACI8DIk=", "Springer_total_obs_distribution.mat": "TUFUTEFCIDUuMCBNQVQtZmlsZSwgUGxhdGZvcm06IFBDV0lONjQsIENyZWF0ZWQgb246IFR1ZSBGZWIgMjMgMTg6NDE6MjggMjAxNiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAAAAAAAAAAAAABSU0PAAAA0wAAAHic42NgYIhgZGBgA9IcQAxkggErlM8EFQNheSAOLijKzEtPLYovyS9JzInPTyqOT8ksLinKTCotyczPY+ADqglgQJjHhmYeyBwWJHs4gVgBiGM2fRC41/3apuaJZey3wwdt6v/PbSxeG7PnpU+84qFpt/eAzN2Ax1wWLOY2ALHdTL2LjJ/f26txR7hfefnGfqZc63Phw8/tvwm2+OrPfwYX3z8Doq5F/19DnNlr+zPPoj/OznwJVw8Tj4WaN5vrpI/ZvGdwc2DqYeLLoOYBAAtMZDk="}
_PARAM_SHA = {"Springer_B_matrix.mat": "457621a433dc5ca6bd9faa80b63fdd427b176eba9ab2c888a803dbbf85ab35ea", "Springer_pi_vector.mat": "dbcd79495dc89bae1e1c48a1a0f58c0667ab3cb3992a10bd87610b5962a5fed9", "Springer_total_obs_distribution.mat": "54949bf43814373b2028e23659619af2e27e4a97f7fbe2a9334996eb7d297fd3"}
for _k, _v in _PARAM_B64.items():
    assert hashlib.sha256(_b64.b64decode(_v)).hexdigest() == _PARAM_SHA[_k], 'Springer parameter file %s corrupted' % _k
_mats = {k: scipy.io.loadmat(_io.BytesIO(_b64.b64decode(v)), squeeze_me=True) for k, v in _PARAM_B64.items()}
SPRINGER_PARAMS = ([np.asarray(b, float) for b in _mats['Springer_B_matrix.mat']['Springer_B_matrix']],
                   np.asarray([float(p) for p in _mats['Springer_pi_vector.mat']['Springer_pi_vector']]),
                   (np.asarray(_mats['Springer_total_obs_distribution.mat']['Springer_total_obs_distribution'][0], float),
                    np.asarray(_mats['Springer_total_obs_distribution.mat']['Springer_total_obs_distribution'][1], float)))
S_VARIANTS = {'springer_wpk': dict(hr_min=40, hr_max=220, sys_min_s=0.10, hr_method='peak'),
              'springer_pub': dict(hr_min=30, hr_max=120, sys_min_s=0.20, hr_method='argmax')}
S_CFG_HASH = hashlib.sha1(json.dumps(dict(src='33bf4c10b8ab6d1041c8eb951317c40400ff7cb735d39c1a6e2e0406991499f7', params=_PARAM_SHA, variants=S_VARIANTS, n_min=N_MIN_CYCLES, viterbi=SPRINGER_VITERBI),
                                     sort_keys=True).encode()).hexdigest()[:16]

def states_to_cycles(states_1k, L4, sr_states=1000, sr_out=TARGET_SR):
    """Complete S1 -> systole -> S2 -> diastole runs of the state sequence as R1 cycles at 4 kHz."""
    st = np.asarray(states_1k)
    ch = np.flatnonzero(np.diff(st)) + 1
    starts, ends = np.concatenate([[0], ch]), np.concatenate([ch, [len(st)]])
    runs = [(int(st[s]), int(s), int(e)) for s, e in zip(starts, ends)]
    k = sr_out / sr_states
    cyc, i = [], 0
    while i + 3 < len(runs):
        if [r[0] for r in runs[i:i + 4]] == [1, 2, 3, 4]:
            b = [min(L4, int(round(x * k))) for r in runs[i:i + 4] for x in (r[1], r[2])]
            c = dict(cycle=len(cyc), s1=(b[0], b[1]), systole=(b[2], b[3]), s2=(b[4], b[5]), diastole=(b[6], b[7]))
            if c['diastole'][1] > c['s1'][0]:
                cyc.append(c)
            i += 4
        else:
            i += 1
    return cyc

def annotated_spans(cyc_c):
    """Annotated stretches of arm C: runs of abutting cycles, (first S1 onset, last diastole end) in samples."""
    spans = []
    for c in cyc_c:
        a, b = int(c['s1'][0]), int(c['diastole'][1])
        if spans and a - spans[-1][1] <= 2:
            spans[-1][1] = max(spans[-1][1], b)
        else:
            spans.append([a, b])
    return spans

def restrict_to(cyc, spans):
    """Cycles whose S1 onset lies inside an annotated stretch (selection only; boundaries untouched)."""
    out = [c for c in cyc if any(a <= c['s1'][0] < b for a, b in spans)]
    return [dict(c, cycle=i) for i, c in enumerate(out)]

def _s_worker(name):
    """Both Springer variants on the shared analysed audio; fewer than N_MIN_CYCLES cycles -> equal-time fallback."""
    try:
        L4 = AUDIO_META[name]['L4']
        x4 = np.asarray(AUDIO4[name][:L4], float)
        out = {}
        for arm, kw in S_VARIANTS.items():
            states, info = springer.run_springer(x4, TARGET_SR, SPRINGER_PARAMS, return_info=True, **kw)
            cyc = states_to_cycles(states, L4)
            d = dict(hr_bpm=float(info['hr']), systolic_s=float(info['systolic_time']), n_cycles_springer=len(cyc),
                     unassigned_frac=float(np.mean(np.asarray(states) == 0)), fallback=False)
            if len(cyc) < N_MIN_CYCLES:
                cyc, d['fallback'] = equal_time_cycles(L4, TARGET_SR), True
            out[arm] = (cyc, d)
        return name, out, None
    except Exception as e:
        return name, None, '%s: %s' % (type(e).__name__, e)
print('Springer arms ready: %s | configuration hash %s | source sha256 %s | Viterbi %s' % (list(S_VARIANTS), S_CFG_HASH, '33bf4c10b8ab6d1041c8eb951317c40400ff7cb735d39c1a6e2e0406991499f7'[:16], SPRINGER_VITERBI))

In [ ]:
# ---------------- Extension Run 3: the frozen control arm A, label-free inference and cycle export ----------------
import base64 as _b64r, zlib as _zr, shutil as _sh2
_MOD_B64 = {"__init__.py": "IiIiZXh0c3R1ZHk6IGNvZGUgZm9yIHRoZSBFSEotREggZXh0ZW5zaW9uIGV4cGVyaW1lbnRzIGFuZCB0aGUgZmlyc3Qgc2xpY2Ugb2YgdGhlIFBoRCBwcm9ncmFtbWUuIiIiCg==", "frontend.py": "IiIiTW9kZWwgaW5wdXQ6IGxvZy1wb3dlciBzcGVjdHJvZ3JhbSBvZiB0aGUgUENHICgwOCBkZXNpZ24sIHNlY3Rpb24gMi43KS4KClBDRyBhdCAxIGtIeiwgYmFuZC1wYXNzIDI1IHRvIDQwMCBIeiAoc2Vjb25kLW9yZGVyIHNlY3Rpb25zLCBwb2xlIGFzc2VydGlvbikgYXBwbGllZCBvbmNlIHRvIHRoZSB3aG9sZSByZWNvcmRpbmc7CmZyYW1lcyBvZiAzMiBzYW1wbGVzICgzMiBtcyBIYW5uKSBldmVyeSA0IHNhbXBsZXMgKDI1MCBIeiksIHplcm8tcGFkZGVkIHRvIDY0IHBvaW50cywgYmlucyAyIHRvIDI1ICgzMS4yNSB0byAzOTAuNiBIeiwKMjQgYmlucyk7IGxvZyBwb3dlcjsgZWFjaCBiaW4ncyBtZWRpYW4gb3ZlciB0aGUgd2hvbGUgcmVjb3JkaW5nIHN1YnRyYWN0ZWQuIEZyYW1lIGogY292ZXJzIHNhbXBsZXMgWzRqLCA0aiArIDMyKQphbmQgaXMgY2VudHJlZCBhdCAoNGogKyAxNikgLyAxMDAwIHMuIEZyYW1lcyB1c2Ugb25seSBzYW1wbGVzIGluc2lkZSB0aGVpciB3aW5kb3csIHNvIGEgY3JvcCB0aGF0IHN0YXJ0cyBvbiB0aGUgNC1zYW1wbGUKZ3JpZCB5aWVsZHMgZXhhY3RseSB0aGUgZnJhbWVzIG9mIHRoZSB3aG9sZSByZWNvcmRpbmcuIFRoZSBudW1weSB2ZXJzaW9uIGJ1aWxkcyB0aGUgY29ycHVzOyB0aGUgdG9yY2ggdmVyc2lvbiBydW5zCm9uIENvbGFiLCBhbmQgYSB0ZXN0IGhvbGRzIHRoZW0gZXF1YWwuCiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHNjaXB5LnNpZ25hbCBhcyBzZwoKRlMgPSAxMDAwCldJTiwgSE9QLCBORkZUID0gMzIsIDQsIDY0CkJJTlMgPSBzbGljZSgyLCAyNikgICAgICAgICAgICAgICAgICAgICAgICAjIDMxLjI1IHRvIDM5MC42IEh6CkJBTkQgPSAoMjUuMCwgNDAwLjApCkVQUyA9IDFlLTEwCgoKZGVmIGJhbmRwYXNzX3Nvcyhmcz1GUyk6CiAgICBzb3MgPSBzZy5idXR0ZXIoNCwgQkFORCwgYnR5cGU9J2JhbmRwYXNzJywgZnM9ZnMsIG91dHB1dD0nc29zJykKICAgIGFzc2VydCBucC5hYnMoc2cuc29zMnpwayhzb3MpWzFdKS5tYXgoKSA8IDAuOTk5OSwgJ3Vuc3RhYmxlIGJhbmQtcGFzcycKICAgIHJldHVybiBzb3MKCgpkZWYgYmFuZHBhc3MocGNnLCBmcz1GUyk6CiAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkoc2cuc29zZmlsdGZpbHQoYmFuZHBhc3Nfc29zKGZzKSwgbnAuYXNhcnJheShwY2csIG5wLmZsb2F0NjQpKSkKCgpkZWYgdG9fMWsocGNnLCBmc19pbik6CiAgICBmcm9tIG1hdGggaW1wb3J0IGdjZAogICAgZnNfaW4gPSBpbnQocm91bmQoZnNfaW4pKQogICAgaWYgZnNfaW4gPT0gRlM6CiAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkocGNnLCBucC5mbG9hdDY0KQogICAgZyA9IGdjZChmc19pbiwgRlMpCiAgICByZXR1cm4gc2cucmVzYW1wbGVfcG9seShucC5hc2FycmF5KHBjZywgbnAuZmxvYXQ2NCksIEZTIC8vIGcsIGZzX2luIC8vIGcpCgoKZGVmIG5fZnJhbWVzKG5fc2FtcGxlcyk6CiAgICByZXR1cm4gbWF4KDAsIChuX3NhbXBsZXMgLSBXSU4pIC8vIEhPUCArIDEpCgoKZGVmIGZyYW1lX3RpbWVzX3Mobl9zYW1wbGVzLCBzdGFydF9zYW1wbGU9MCk6CiAgICByZXR1cm4gKHN0YXJ0X3NhbXBsZSArIG5wLmFyYW5nZShuX2ZyYW1lcyhuX3NhbXBsZXMpKSAqIEhPUCArIFdJTiAvIDIpIC8gRlMKCgpkZWYgbG9ncG93ZXIoeCk6CiAgICAiIiJCYW5kLXBhc3NlZCBQQ0cgKDEga0h6KSAtPiAoMjQsIGZyYW1lcykgbG9nIHBvd2VyLCBiZWZvcmUgbWVkaWFuIHN1YnRyYWN0aW9uLiIiIgogICAgeCA9IG5wLmFzYXJyYXkoeCwgbnAuZmxvYXQ2NCkKICAgIG5mID0gbl9mcmFtZXMobGVuKHgpKQogICAgaWYgbmYgPT0gMDoKICAgICAgICByZXR1cm4gbnAuemVyb3MoKEJJTlMuc3RvcCAtIEJJTlMuc3RhcnQsIDApKQogICAgaWR4ID0gbnAuYXJhbmdlKFdJTilbTm9uZSwgOl0gKyBIT1AgKiBucC5hcmFuZ2UobmYpWzosIE5vbmVdCiAgICBmciA9IHhbaWR4XSAqIG5wLmhhbm5pbmcoV0lOICsgMilbMTotMV1bTm9uZSwgOl0KICAgIHNwZWMgPSBucC5hYnMobnAuZmZ0LnJmZnQoZnIsIG49TkZGVCwgYXhpcz0xKSkgKiogMgogICAgcmV0dXJuIG5wLmxvZyhzcGVjWzosIEJJTlNdICsgRVBTKS5UCgoKZGVmIGJpbl9tZWRpYW5zKHhfYnApOgogICAgcmV0dXJuIG5wLm1lZGlhbihsb2dwb3dlcih4X2JwKSwgYXhpcz0xKQoKCmRlZiBsb2dzcGVjKHhfYnAsIG1lZGlhbnM9Tm9uZSk6CiAgICAiIiJNZWRpYW4tbm9ybWFsaXNlZCBzcGVjdHJvZ3JhbTsgbWVkaWFucyBmcm9tIHRoZSB3aG9sZSByZWNvcmRpbmcgKHBhc3MgdGhlbSBmb3IgY3JvcHMpLiIiIgogICAgTCA9IGxvZ3Bvd2VyKHhfYnApCiAgICBtID0gbnAubWVkaWFuKEwsIGF4aXM9MSkgaWYgbWVkaWFucyBpcyBOb25lIGVsc2UgbnAuYXNhcnJheShtZWRpYW5zKQogICAgcmV0dXJuIChMIC0gbVs6LCBOb25lXSkuYXN0eXBlKG5wLmZsb2F0MzIpCgoKZGVmIGxvZ3NwZWNfdG9yY2goeF9icCwgbWVkaWFucyk6CiAgICAiIiJUb3JjaCB0d2luIG9mIGxvZ3NwZWMgZm9yIGJhdGNoZXM6IHhfYnAgKEIsIG4pIGZsb2F0MzIsIG1lZGlhbnMgKEIsIDI0KS4gUmV0dXJucyAoQiwgMjQsIGZyYW1lcykuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIHdpbiA9IHRvcmNoLmhhbm5fd2luZG93KFdJTiArIDIsIHBlcmlvZGljPUZhbHNlLCBkdHlwZT14X2JwLmR0eXBlLCBkZXZpY2U9eF9icC5kZXZpY2UpWzE6LTFdCiAgICBmciA9IHhfYnAudW5mb2xkKC0xLCBXSU4sIEhPUCkgKiB3aW4gICAgICAgICAgICAgICAgICAgICAgIyAoQiwgZnJhbWVzLCAzMikKICAgIHNwZWMgPSB0b3JjaC5mZnQucmZmdChmciwgbj1ORkZULCBkaW09LTEpLmFicygpICoqIDIKICAgIEwgPSB0b3JjaC5sb2coc3BlY1suLi4sIEJJTlNdICsgRVBTKS50cmFuc3Bvc2UoLTEsIC0yKSAgICAjIChCLCAyNCwgZnJhbWVzKQogICAgcmV0dXJuIEwgLSBtZWRpYW5zWy4uLiwgTm9uZV0K", "model.py": "IiIiMS1EIFUtTmV0IG9uIHRoZSBsb2ctcG93ZXIgc3BlY3Ryb2dyYW0gKDA4IGRlc2lnbiwgc2VjdGlvbiAyLjc7IHByZXJlZyBzZWN0aW9uIDcpLg0KDQpJbnB1dCAoQiwgMjQsIGZyYW1lcykgYXQgMjUwIEh6LiBFbmNvZGVyIDY0IC0+IDk2IC0+IDEyOCBjaGFubmVscyAodHdvIHN0cmlkZS0yIHN0ZXBzLCAzLXRhcCBjb252b2x1dGlvbnMpLA0KZWlnaHQgZGlsYXRlZCByZXNpZHVhbCBibG9ja3MgYXQgMTI4IGNoYW5uZWxzIChhIGRpbGF0ZWQgMy10YXAgY29udm9sdXRpb24gYW5kIGEgMXgxIGNvbnZvbHV0aW9uOyBkaWxhdGlvbnMgMSwgMiwgNCwgOCwNCjEsIDIsIDQsIDg7IHJlY2VwdGl2ZSBmaWVsZCBhYm91dCAxIHMpLCBkZWNvZGVyIGJhY2sgdG8gNjQgY2hhbm5lbHMgd2l0aCBza2lwIGNvbm5lY3Rpb25zLCBhbmQgMXgxIGhlYWRzIGF0IHRoZSBpbnB1dA0KZnJhbWUgcmF0ZS4gVDogdGhyZWUgZXZlbnQgbG9naXRzIChTMSwgUzIsIFIpLiBXOiBvbmUgRUNHIGNoYW5uZWwuIEdyb3VwTm9ybSAoaW5kZXBlbmRlbnQgb2YgYmF0Y2gNCmNvbXBvc2l0aW9uKSwgODcyLDM4NyBwYXJhbWV0ZXJzIHdpdGggdGhlIFQgaGVhZHMuIEFueSBmcmFtZSBjb3VudCB3b3JrczogdGhlIGlucHV0IGlzIHBhZGRlZCB0byBhIG11bHRpcGxlIG9mIDQNCmJ5IHJlZmxlY3Rpb24gYW5kIHRoZSBvdXRwdXQgY3JvcHBlZCBiYWNrLg0KIiIiDQppbXBvcnQgdG9yY2gNCmltcG9ydCB0b3JjaC5ubiBhcyBubg0KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRg0KDQpIRUFEUyA9IHsnVCc6ICgnczEnLCAnczInLCAncicpLCAnVyc6ICgnZWNnJywpLCAnQSc6ICgnczEnLCAnczInKX0gICAgICAgIyBBOiB0aGUgYWNvdXN0aWMtbGFiZWwgY29udHJvbCBhcm0gKEV4cGVyaW1lbnQgMiksIG5vIFIgaGVhZA0KDQoNCmRlZiBfbm9ybShjKToNCiAgICByZXR1cm4gbm4uR3JvdXBOb3JtKDgsIGMpDQoNCg0KY2xhc3MgQ29udkJsb2NrKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNpbiwgY291dCwgaz0zKToNCiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgIHNlbGYubmV0ID0gbm4uU2VxdWVudGlhbChubi5Db252MWQoY2luLCBjb3V0LCBrLCBwYWRkaW5nPWsgLy8gMiksIF9ub3JtKGNvdXQpLCBubi5HRUxVKCksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBubi5Db252MWQoY291dCwgY291dCwgaywgcGFkZGluZz1rIC8vIDIpLCBfbm9ybShjb3V0KSwgbm4uR0VMVSgpKQ0KDQogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6DQogICAgICAgIHJldHVybiBzZWxmLm5ldCh4KQ0KDQoNCmNsYXNzIFJlc0Jsb2NrKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGMsIGRpbGF0aW9uLCBkcm9wb3V0PTAuMSk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBzZWxmLm5ldCA9IG5uLlNlcXVlbnRpYWwobm4uQ29udjFkKGMsIGMsIDMsIHBhZGRpbmc9ZGlsYXRpb24sIGRpbGF0aW9uPWRpbGF0aW9uKSwgX25vcm0oYyksIG5uLkdFTFUoKSwgbm4uRHJvcG91dChkcm9wb3V0KSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5uLkNvbnYxZChjLCBjLCAxKSwgX25vcm0oYykpDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgcmV0dXJuIEYuZ2VsdSh4ICsgc2VsZi5uZXQoeCkpDQoNCg0KY2xhc3MgVU5ldDFEKG5uLk1vZHVsZSk6DQogICAgZGVmIF9faW5pdF9fKHNlbGYsIGtpbmQ9J1QnLCBuX2luPTI0LCB3aWR0aHM9KDY0LCA5NiwgMTI4KSwgZGlsYXRpb25zPSgxLCAyLCA0LCA4LCAxLCAyLCA0LCA4KSk6DQogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQ0KICAgICAgICBhc3NlcnQga2luZCBpbiBIRUFEUw0KICAgICAgICBzZWxmLmtpbmQgPSBraW5kDQogICAgICAgIHcwLCB3MSwgdzIgPSB3aWR0aHMNCiAgICAgICAgc2VsZi5lbmMwID0gQ29udkJsb2NrKG5faW4sIHcwKQ0KICAgICAgICBzZWxmLmRvd24xID0gbm4uQ29udjFkKHcwLCB3MSwgNCwgc3RyaWRlPTIsIHBhZGRpbmc9MSkNCiAgICAgICAgc2VsZi5lbmMxID0gQ29udkJsb2NrKHcxLCB3MSkNCiAgICAgICAgc2VsZi5kb3duMiA9IG5uLkNvbnYxZCh3MSwgdzIsIDQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQogICAgICAgIHNlbGYubWlkID0gbm4uU2VxdWVudGlhbCgqW1Jlc0Jsb2NrKHcyLCBkKSBmb3IgZCBpbiBkaWxhdGlvbnNdKQ0KICAgICAgICBzZWxmLnVwMiA9IG5uLkNvbnZUcmFuc3Bvc2UxZCh3MiwgdzEsIDQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQogICAgICAgIHNlbGYuZGVjMSA9IENvbnZCbG9jaygyICogdzEsIHcxKQ0KICAgICAgICBzZWxmLnVwMSA9IG5uLkNvbnZUcmFuc3Bvc2UxZCh3MSwgdzAsIDQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpDQogICAgICAgIHNlbGYuZGVjMCA9IENvbnZCbG9jaygyICogdzAsIHcwKQ0KICAgICAgICBzZWxmLmhlYWQgPSBubi5Db252MWQodzAsIGxlbihIRUFEU1traW5kXSksIDEpDQogICAgICAgIGlmIGtpbmQgIT0gJ1cnOiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBldmVudCBoZWFkcyBzdGFydCBuZWFyIGEgbG93IHByaW9yIChmb2NhbC1sb3NzIHByYWN0aWNlKQ0KICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8oc2VsZi5oZWFkLmJpYXMsIC00LjApDQoNCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToNCiAgICAgICAgbiA9IHguc2hhcGVbLTFdDQogICAgICAgIHBhZCA9ICgtbikgJSA0DQogICAgICAgIGlmIHBhZDoNCiAgICAgICAgICAgIHggPSBGLnBhZCh4LCAoMCwgcGFkKSwgbW9kZT0ncmVmbGVjdCcpDQogICAgICAgIGUwID0gc2VsZi5lbmMwKHgpDQogICAgICAgIGUxID0gc2VsZi5lbmMxKHNlbGYuZG93bjEoZTApKQ0KICAgICAgICBtID0gc2VsZi5taWQoc2VsZi5kb3duMihlMSkpDQogICAgICAgIGQxID0gc2VsZi5kZWMxKHRvcmNoLmNhdChbc2VsZi51cDIobSksIGUxXSwgMSkpDQogICAgICAgIGQwID0gc2VsZi5kZWMwKHRvcmNoLmNhdChbc2VsZi51cDEoZDEpLCBlMF0sIDEpKQ0KICAgICAgICByZXR1cm4gc2VsZi5oZWFkKGQwKVsuLi4sIDpuXQ0KDQoNCmRlZiBuX3BhcmFtcyhtb2RlbCk6DQogICAgcmV0dXJuIHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQ0K", "robust_beats.py": "...", "decoder.py": "IiIiQmVhdCBkZWNvZGVyIHNoYXJlZCBieSBhcm1zIFQgYW5kIFcgKFBhbmVsIEgsIGRlY2lzaW9uIDIpLCBzbyB0aGF0IFcgYWdhaW5zdCBUIGNvbXBhcmVzIHRyYWNlcywgbm90IGRlY29kZXJzLg0KDQpJbnB1dDogYW4gZXZlbnQgdHJhY2UgKFQ6IHRoZSBSIG9yIFMxIGhlYWQgb3V0cHV0KSBvciBhIHN5bnRoZXRpYyBFQ0cgKFc6IGRlcml2YXRpdmUgZW5lcmd5IG9mIGl0cyA1IHRvIDI1IEh6IGJhbmQpLg0KU3RlcHMsIGFsbCBhdCA1MDAgSHo6DQogIDEuIGRldGVjdGlvbiBlbmVyZ3kgKEVDRykgb3IgdGhlIHRyYWNlIGl0c2VsZiAoZXZlbnRzKTsgNjAgbXMgc21vb3RoaW5nIGZvciBjYW5kaWRhdGVzOw0KICAyLiBleHBlY3RlZCBSUiBmcm9tIGF1dG9jb3JyZWxhdGlvbiAoZnJvemVuIGRldGVjdG9yJ3MgcnJfcHJpb3I6IDMwIHRvIDI0MiBicG0gc2VhcmNoLCBoaWdoZXN0IGxvY2FsIG1heGltdW0sDQogICAgIGhhbGYtbGFnIHN3aXRjaCBhdCAwLjY7IDIwIHMgd2luZG93cyBhY2NlcHRlZCB3aXRoaW4gMC41NSB0byAxLjggeCB0aGUgZ2xvYmFsIHZhbHVlKTsNCiAgMy4gY2FuZGlkYXRlcyBhdCBsZWFzdCAxMjAgbXMgYXBhcnQgd2l0aCBwcm9taW5lbmNlIGF0IGxlYXN0IDIlIG9mIHRoZSBsb2NhbCBiZWF0IGxldmVsOw0KICA0LiBlbWlzc2lvbiBmcm9tIGFtcGxpdHVkZSBhbG9uZSAocmVsYXRpdmUgdG8gdGhlIGxvY2FsIGJlYXQgbGV2ZWwpOw0KICA1LiB0d28tcGFzcyBWaXRlcmJpIG92ZXIgbG9nLVJSIHRyYW5zaXRpb25zIChzaW51cyBTRCAwLjI1LCB0aGVuIDAuMTUgYXJvdW5kIGEgOS1iZWF0IHJ1bm5pbmcgbWVkaWFuOyBwcmVtYXR1cmUNCiAgICAgMC4xMCBvdmVyIDAuNCB0byAxLjAgUlI7IGxvbmcgMC4wNSBvdmVyIDEuMCB0byAxLjggUlI7IG1pc3NlZCBiZWF0IDAuMDEgYXQgMiBhbmQgMyBSUiksIHJlZnJhY3RvcnkNCiAgICAgbWF4KDIwMCBtcywgMC4zNSB4IGV4cGVjdGVkIFJSKSwgcmVzdGFydCBwZW5hbHR5IC0xMiAodGhlIGZyb3plbiBkZXRlY3RvcidzIHZhbHVlcyk7DQogIDYuIGJlYXQgcG9zaXRpb246IGVuZXJneSBjZW50cm9pZCAoRUNHKSBvciB0aGUgdHJhY2UgbWF4aW11bSB3aXRoIHBhcmFib2xpYyByZWZpbmVtZW50IChldmVudHMpOw0KICA3LiBubyBsYWJlbHMgYWNyb3NzIGdhcHMgbG9uZ2VyIHRoYW4gMS44IHggdGhlIGV4cGVjdGVkIFJSLg0KTm90aGluZyBoZXJlIHJlYWRzIGEgZGF0YXNldDsgdGhlIHRyYWNlIGNvbWVzIGZyb20gdGhlIGNhbGxlci4NCiIiIg0KaW1wb3J0IG51bXB5IGFzIG5wDQppbXBvcnQgc2NpcHkuc2lnbmFsIGFzIHNnDQoNCmZyb20gZXh0c3R1ZHkgaW1wb3J0IHJvYnVzdF9iZWF0cyBhcyBSQg0KDQpGUyA9IFJCLkZTDQojIGVtaXNzaW9uIDQgbG9nMihhbXBsaXR1ZGUgLyBsb2NhbCBiZWF0IGxldmVsKSArIDI6IHplcm8gYXQgMC43MSBvZiB0aGUgbG9jYWwgYmVhdCBsZXZlbCAoYXMgaW4gdGhlIGZyb3plbiBkZXRlY3RvciwNCiMgMiBsb2cyICsgMSksIGJ1dCBhIGdvb2QgYmVhdCBzY29yZXMgKzIsIG5vdCArMS4gV2l0aCArMSwgZHJvcHBpbmcgYSBzdHJldGNoIG9mIGJlYXRzIHRocm91Z2ggb25lIHJlc3RhcnQgKC0xMikgYmVhdA0KIyBrZWVwaW5nIGl0IHdoZW5ldmVyIGEgdGhpcmQgb2YgaXRzIGJlYXRzIHdlcmUgbWlzc2luZyAoZWFjaCBtaXNzZWQgYmVhdCBjb3N0cyBhYm91dCAtMy40KTogMTEgdHJ1ZSBiZWF0cyBsb3N0IGluDQojIGEgc2ltdWxhdGVkIHJlY29yZCB3aXRoIGEgMyBzIHBhdXNlDQpQQVJBTVMgPSBkaWN0KGNhbmRfbWluX21zPTEyMCwgY2FuZF9wcm9tPTAuMDIsIHJlZl93aW5fcz01LjAsIHdfYW1wPTQuMCwgYW1wX2NsaXA9KC00LjAsIDEuMCksIGVfYmlhcz0yLjAsDQogICAgICAgICAgICAgIHNpZ21hMT1SQi5QWydzaWdtYTEnXSwgbWF4X2dhcDE9UkIuUFsnbWF4X2dhcDEnXSwgc2lnbWEyPVJCLlBbJ3NpZ21hJ10sIG1heF9nYXAyPVJCLlBbJ21heF9nYXBfcmF0aW8nXSwNCiAgICAgICAgICAgICAgcnJfcmVsX21pbj1SQi5QWydycl9yZWxfbWluJ10sIGdhcF9mbGFnPTEuOCwgY2VudF9tcz02MCkNCg0KDQpkZWYgYmVhdF90cmFjZV9mcm9tX2hlYWRzKHBfYmVhdCwgcF9zMik6DQogICAgIiIiQXJtIFQncyBpbnB1dCB0byB0aGUgZGVjb2RlcjogdGhlIGJlYXQgaGVhZCAoUiBvciBTMSkgbWludXMgdGhlIFMyIGhlYWQsIGZsb29yZWQgYXQgMC4gQW4gUzIgc2VxdWVuY2UgaXMgYXMNCiAgICByZWd1bGFyIGFzIHRoZSBiZWF0IHNlcXVlbmNlOyB3aXRoIFMxIG1pc3NpbmcgaW4gYSBzdHJldGNoLCBhbXBsaXR1ZGUgYW5kIHJoeXRobSBhbG9uZSBwcmVmZXJyZWQgdGhlIFMyIGNoYWluDQogICAgKHNpbXVsYXRlZCB0cmFjZXMpLCBzbyBTMiBldmlkZW5jZSBlbnRlcnMgdGhyb3VnaCB0aGUgdHJhY2UgYW5kIHRoZSBkZWNvZGVyIHN0YXlzIHRoZSBzYW1lIGZvciBUIGFuZCBXLiIiIg0KICAgIHJldHVybiBucC5jbGlwKG5wLmFzYXJyYXkocF9iZWF0LCBucC5mbG9hdDY0KSAtIG5wLmFzYXJyYXkocF9zMiwgbnAuZmxvYXQ2NCksIDAuMCwgTm9uZSkNCg0KDQpkZWYgZWNnX2VuZXJneShlY2c1MDApOg0KICAgICIiIlNxdWFyZWQgNS1wb2ludCBkZXJpdmF0aXZlIG9mIHRoZSA1IHRvIDI1IEh6IGJhbmQgKFBhbi1Ub21wa2lucyBmb3JtKSwgYW5kIHRoZSBiYW5kIGl0c2VsZi4iIiINCiAgICBxID0gUkIuX2ZpbHRfc2VnbWVudHMobnAuYXNhcnJheShlY2c1MDAsIG5wLmZsb2F0NjQpLCBSQi5fc29zKCdxcnMnKSwgWzAsIGxlbihlY2c1MDApXSkNCiAgICBxID0gcSAvIChucC5tZWRpYW4obnAuYWJzKHEpKSArIDFlLTEyKQ0KICAgIGQgPSBucC56ZXJvc19saWtlKHEpDQogICAgZFsyOi0yXSA9ICgtcVs0Ol0gLSAyICogcVszOi0xXSArIDIgKiBxWzE6LTNdICsgcVs6LTRdKSAvIDguMA0KICAgIHJldHVybiBkICoqIDIsIHENCg0KDQpkZWYgX2xvY2FsX2xldmVsKHBrLCBoLCBycmUsIFcpOg0KICAgIHJlZiA9IG5wLmVtcHR5KGxlbihwaykpDQogICAgZm9yIGksIHAgaW4gZW51bWVyYXRlKHBrKToNCiAgICAgICAgYSwgYiA9IG5wLnNlYXJjaHNvcnRlZChwaywgcCAtIFcpLCBucC5zZWFyY2hzb3J0ZWQocGssIHAgKyBXKQ0KICAgICAgICBrID0gbWF4KDIsIGludChyb3VuZCgyICogUEFSQU1TWydyZWZfd2luX3MnXSAvIG1heChycmVbbWluKHAsIGxlbihycmUpIC0gMSldLCAwLjIpKSkpDQogICAgICAgIHJlZltpXSA9IG5wLm1lZGlhbihucC5zb3J0KGhbYTpiXSlbOjotMV1bOmtdKQ0KICAgIHJldHVybiByZWYNCg0KDQpkZWYgX3JlZmluZV9wZWFrKHksIHApOg0KICAgIGlmIDEgPD0gcCA8IGxlbih5KSAtIDE6DQogICAgICAgIHkwLCB5MSwgeTIgPSB5W3AgLSAxXSwgeVtwXSwgeVtwICsgMV0NCiAgICAgICAgZGVuID0geTAgLSAyICogeTEgKyB5Mg0KICAgICAgICByZXR1cm4gcCArIChucC5jbGlwKDAuNSAqICh5MCAtIHkyKSAvIGRlbiwgLTAuNSwgMC41KSBpZiBkZW4gIT0gMCBlbHNlIDAuMCkNCiAgICByZXR1cm4gZmxvYXQocCkNCg0KDQpkZWYgZGVjb2RlX2JlYXRzKHRyYWNlLCBmcywga2luZD0nZXZlbnQnKToNCiAgICAiIiJ0cmFjZTogZXZlbnQgcHJvYmFiaWxpdHkgKGtpbmQ9J2V2ZW50Jykgb3Igc3ludGhldGljIEVDRyAoa2luZD0nZWNnJykgYXQgZnMuDQogICAgUmV0dXJucyBkaWN0KGJlYXRzX3MsIGJlYXRzNTAwLCBtYXNrNTAwIChUcnVlID0gbGFiZWxsZWQpLCBycmVfcywgcmVjKS4iIiINCiAgICB4ID0gUkIucmVzYW1wbGVfdG8obnAubmFuX3RvX251bShucC5hc2FycmF5KHRyYWNlLCBucC5mbG9hdDY0KSksIGZzLCBGUykNCiAgICBuID0gbGVuKHgpDQogICAgaWYga2luZCA9PSAnZWNnJzoNCiAgICAgICAgZSwgcSA9IGVjZ19lbmVyZ3koeCkNCiAgICAgICAgZXJhdyA9IHEgKiogMg0KICAgIGVsaWYga2luZCA9PSAnZXZlbnQnOg0KICAgICAgICBlLCBlcmF3ID0gbnAubWF4aW11bSh4LCAwLjApLCBOb25lDQogICAgZWxzZToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihraW5kKQ0KICAgIGU2MCA9IFJCLl9tb3ZhdmcoZSwgMC4wNjAgKiBGUykgaWYga2luZCA9PSAnZWNnJyBlbHNlIFJCLl9tb3ZhdmcoZSwgMC4wMjAgKiBGUykNCiAgICBycmUsIGFjZl9zdHJlbmd0aCA9IFJCLnJyX3ByaW9yKGU2MCBpZiBraW5kID09ICdlY2cnIGVsc2UgUkIuX21vdmF2ZyhlLCAwLjA2MCAqIEZTKSkNCiAgICBwaywgcHIgPSBzZy5maW5kX3BlYWtzKGU2MCwgZGlzdGFuY2U9aW50KFBBUkFNU1snY2FuZF9taW5fbXMnXSAvIDEwMDAgKiBGUyksIHByb21pbmVuY2U9MCkNCiAgICBlbXB0eSA9IGRpY3QoYmVhdHNfcz1ucC5hcnJheShbXSksIGJlYXRzNTAwPW5wLmFycmF5KFtdLCBpbnQpLCBtYXNrNTAwPW5wLnplcm9zKG4sIGJvb2wpLCBycmVfcz1ycmUsDQogICAgICAgICAgICAgICAgIHJlYz1kaWN0KG5fYmVhdHM9MCwgYWNmX3N0cmVuZ3RoPWFjZl9zdHJlbmd0aCwgY292ZXJhZ2U9MC4wLCBocl9tZWRpYW49bnAubmFuKSkNCiAgICBpZiBsZW4ocGspIDwgMzoNCiAgICAgICAgcmV0dXJuIGVtcHR5DQogICAgaCA9IGU2MFtwa10NCiAgICByZWYgPSBfbG9jYWxfbGV2ZWwocGssIGgsIHJyZSwgaW50KFBBUkFNU1sncmVmX3dpbl9zJ10gKiBGUykpDQogICAga2VlcCA9IHByWydwcm9taW5lbmNlcyddID49IFBBUkFNU1snY2FuZF9wcm9tJ10gKiByZWYNCiAgICBwaywgaCwgcmVmID0gcGtba2VlcF0sIGhba2VlcF0sIHJlZltrZWVwXQ0KICAgIGlmIGxlbihwaykgPCAzOg0KICAgICAgICByZXR1cm4gZW1wdHkNCiAgICBlbSA9IFBBUkFNU1snd19hbXAnXSAqIG5wLmNsaXAobnAubG9nMihoIC8gKHJlZiArIDFlLTEyKSArIDFlLTEyKSwgKlBBUkFNU1snYW1wX2NsaXAnXSkgKyBQQVJBTVNbJ2VfYmlhcyddDQogICAgaWYga2luZCA9PSAnZWNnJzoNCiAgICAgICAgaGFsZiA9IGludChQQVJBTVNbJ2NlbnRfbXMnXSAvIDEwMDAgKiBGUykNCiAgICAgICAgcG9zID0gbnAuYXJyYXkoW1JCLl9jZW50cm9pZChlcmF3LCBwLCBoYWxmLCAyKSBmb3IgcCBpbiBwa10pDQogICAgZWxzZToNCiAgICAgICAgcG9zID0gbnAuYXJyYXkoW19yZWZpbmVfcGVhayhlNjAsIHApIGZvciBwIGluIHBrXSkNCiAgICB0YyA9IG5wLnJvdW5kKHBvcykuYXN0eXBlKGludCkNCiAgICBvcmRlciA9IG5wLmFyZ3NvcnQodGMpDQogICAgdGMsIGVtX28gPSB0Y1tvcmRlcl0sIGVtW29yZGVyXQ0KICAgIHBhdGggPSBSQi5fdml0ZXJiaSh0YywgZW1fbywgcnJlW25wLmNsaXAodGMsIDAsIG4gLSAxKV0sIEZTLCBbXSwgbiwgUEFSQU1TWydzaWdtYTEnXSwgUEFSQU1TWydtYXhfZ2FwMSddKQ0KICAgIGJlYXRzID0gdGNbcGF0aF0NCiAgICBpZiBsZW4oYmVhdHMpID49IDU6DQogICAgICAgIHJyID0gbnAuZGlmZihiZWF0cykgLyBGUw0KICAgICAgICBva3JyID0gcnIgPCAxLjcgKiBucC5tZWRpYW4ocnIpDQogICAgICAgIGlmIG9rcnIuc3VtKCkgPj0gMzoNCiAgICAgICAgICAgIG1pZCA9IDAuNSAqIChiZWF0c1sxOl0gKyBiZWF0c1s6LTFdKQ0KICAgICAgICAgICAgayA9IG1pbig5LCBpbnQob2tyci5zdW0oKSkgfCAxKQ0KICAgICAgICAgICAgcnJfcnVuID0gc2cubWVkZmlsdChycltva3JyXSwgaykgaWYgb2tyci5zdW0oKSA+PSBrIGVsc2UgbnAuZnVsbChva3JyLnN1bSgpLCBucC5tZWRpYW4ocnJbb2tycl0pKQ0KICAgICAgICAgICAgcnJlID0gbnAuaW50ZXJwKG5wLmFyYW5nZShuKSwgbWlkW29rcnJdLCBycl9ydW4pDQogICAgICAgICAgICBwYXRoID0gUkIuX3ZpdGVyYmkodGMsIGVtX28sIHJyZVtucC5jbGlwKHRjLCAwLCBuIC0gMSldLCBGUywgW10sIG4sIFBBUkFNU1snc2lnbWEyJ10sIFBBUkFNU1snbWF4X2dhcDInXSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBQQVJBTVNbJ3JyX3JlbF9taW4nXSkNCiAgICBzZWwgPSBvcmRlcltwYXRoXQ0KICAgIGlmIGtpbmQgPT0gJ2VjZyc6DQogICAgICAgIGIgPSBucC5hcnJheShbUkIuX2NlbnRyb2lkKGVyYXcsIHBrW2ldLCBpbnQobWluKFBBUkFNU1snY2VudF9tcyddIC8gMTAwMCwgMC4xNSAqIHJyZVttaW4ocGtbaV0sIG4gLSAxKV0pICogRlMpLCAyKQ0KICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHNlbF0pDQogICAgZWxzZToNCiAgICAgICAgYiA9IHBvc1tzZWxdDQogICAgYjUwMCA9IG5wLnJvdW5kKGIpLmFzdHlwZShpbnQpDQogICAgbWFzayA9IG5wLm9uZXMobiwgYm9vbCkNCiAgICBycmVfYiA9IHJyZVtucC5jbGlwKGI1MDAsIDAsIG4gLSAxKV0gKiBGUw0KICAgIGZvciBhLCBjLCByIGluIHppcChiNTAwWzotMV0sIGI1MDBbMTpdLCBycmVfYls6LTFdKToNCiAgICAgICAgaWYgYyAtIGEgPiBQQVJBTVNbJ2dhcF9mbGFnJ10gKiByOg0KICAgICAgICAgICAgbWFza1thOmNdID0gRmFsc2UNCiAgICBpZiBsZW4oYjUwMCk6DQogICAgICAgIGlmIGI1MDBbMF0gPiBQQVJBTVNbJ2dhcF9mbGFnJ10gKiBycmVfYlswXToNCiAgICAgICAgICAgIG1hc2tbOmI1MDBbMF1dID0gRmFsc2UNCiAgICAgICAgaWYgbiAtIGI1MDBbLTFdID4gUEFSQU1TWydnYXBfZmxhZyddICogcnJlX2JbLTFdOg0KICAgICAgICAgICAgbWFza1tiNTAwWy0xXTpdID0gRmFsc2UNCiAgICByZWMgPSBkaWN0KG5fYmVhdHM9aW50KGxlbihiNTAwKSksIGFjZl9zdHJlbmd0aD1mbG9hdChhY2Zfc3RyZW5ndGgpLCBjb3ZlcmFnZT1mbG9hdChtYXNrLm1lYW4oKSksDQogICAgICAgICAgICAgICBocl9tZWRpYW49ZmxvYXQoNjAgKiBGUyAvIG5wLm1lZGlhbihucC5kaWZmKGI1MDApKSkgaWYgbGVuKGI1MDApID4gMiBlbHNlIG5wLm5hbiwNCiAgICAgICAgICAgICAgIG5fY2FuZGlkYXRlcz1pbnQobGVuKHBrKSkpDQogICAgcmV0dXJuIGRpY3QoYmVhdHNfcz1iIC8gRlMsIGJlYXRzNTAwPWI1MDAsIG1hc2s1MDA9bWFzaywgcnJlX3M9cnJlLCByZWM9cmVjKQ0K", "infer.py": "IiIiTGFiZWwtZnJlZSBpbmZlcmVuY2Ugb2YgdGhlIGZyb3plbiBUIGFuZCBXIG1vZGVscyBvbiBvbmUgUENHIHJlY29yZGluZyAoUnVuIDI7IHByZXJlZyBzZWN0aW9uIDcpLgoKSW5wdXQgcGF0aCBhcyBpbiB0cmFpbmluZzogUENHIHJlc2FtcGxlZCB0byAxIGtIeiwgYmFuZC1wYXNzIDI1IHRvIDQwMCBIeiwgbG9nLXBvd2VyIHNwZWN0cm9ncmFtIHdpdGggZWFjaCBiaW4ncwptZWRpYW4gb3ZlciB0aGUgd2hvbGUgcmVjb3JkaW5nIHN1YnRyYWN0ZWQgKGZyb250ZW5kLmxvZ3NwZWMpOyBmcmFtZSBqIGlzIGNlbnRyZWQgYXQgRlJBTUVfVDAgKyBqIC8gMjUwIHMuCldpbmRvd3M6IDMwIHMgKDcsNTAwIGZyYW1lcyksIHRoZSBjcm9wIGxlbmd0aCBvZiB0aGUgdmFsaWRhdGlvbiBhbmQgR2F0ZSBBIHJ1bnMuIEEgc2hvcnRlciByZWNvcmRpbmcgaXMgemVyby1wYWRkZWQgdG8KNyw1MDAgZnJhbWVzIGFzIHRob3NlIGNyb3BzIHdlcmU7IGEgbG9uZ2VyIG9uZSBydW5zIGluIDMwIHMgd2luZG93cyBvdmVybGFwcGluZyBieSA0IHMgYW5kIGNyb3NzLWZhZGVkIGxpbmVhcmx5LgpTZWVkczogdGhlIHRocmVlIGZyb3plbiBtb2RlbHMgb2YgYW4gYXJtIGFyZSBhdmVyYWdlZCAoVDogc2lnbW9pZCBwcm9iYWJpbGl0aWVzOyBXOiB0aGUgRUNHIG91dHB1dCksIGFzIHRoZQpwcmUtcmVnaXN0cmF0aW9uIGZpeGVzICh0cmFjZXMgYXZlcmFnZWQgYWNyb3NzIHNlZWRzLCBzZWN0aW9uIDcpLgpCZWF0czogdGhlIHNoYXJlZCBkZWNvZGVyIChhbWVuZG1lbnQgMyk7IFQgZGVjb2RlcyBpdHMgUzEgaGVhZCBtaW51cyBpdHMgUzIgaGVhZCwgVyBpdHMgc3ludGhldGljIEVDRy4KTm90aGluZyBoZXJlIHJlYWRzIGEgZGF0YXNldCBvciBhIGxhYmVsOyB0aGUgY2FsbGVyIHBhc3NlcyB0aGUgYXVkaW8uCiIiIgppbXBvcnQgaGFzaGxpYiwganNvbiwgb3MKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIGV4dHN0dWR5IGltcG9ydCBmcm9udGVuZCBhcyBGLCBtb2RlbCBhcyBNLCBkZWNvZGVyIGFzIEQKCkZSQU1FX1QwID0gRi5XSU4gLyAyIC8gRi5GUyAgICAgICAgICAgICMgMC4wMTYgcwpGUkFNRV9TID0gRi5IT1AgLyBGLkZTICAgICAgICAgICAgICAgICAjIDAuMDA0IHMKV0lOX0ZSQU1FUywgT1ZFUkxBUCA9IDc1MDAsIDEwMDAKCgpkZWYgc2hhMjU2X2ZpbGUocGF0aCwgY2h1bms9MSA8PCAyMik6CiAgICBoID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBvcGVuKHBhdGgsICdyYicpIGFzIGY6CiAgICAgICAgZm9yIGIgaW4gaXRlcihsYW1iZGE6IGYucmVhZChjaHVuayksIGInJyk6CiAgICAgICAgICAgIGgudXBkYXRlKGIpCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiBzdGF0ZV9zaGEyNTYoc3RhdGUpOgogICAgIiIiU2FtZSBhcyB0cmFpbi5zdGF0ZV9zaGEyNTYgKHRlc3RzIGhvbGQgdGhlbSBlcXVhbCk7IGtlcHQgaGVyZSBzbyBpbmZlcmVuY2UgZG9lcyBub3QgaW1wb3J0IHRoZSB0cmFpbmVyLiIiIgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIGZvciBrIGluIHNvcnRlZChzdGF0ZSk6CiAgICAgICAgaC51cGRhdGUoay5lbmNvZGUoKSk7IGgudXBkYXRlKHN0YXRlW2tdLmRldGFjaCgpLmNwdSgpLm51bXB5KCkudG9ieXRlcygpKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgc3BlY3Ryb2dyYW0oeCwgZnMpOgogICAgIiIiKDI0LCBmcmFtZXMpIGZsb2F0MzIgbW9kZWwgaW5wdXQgZm9yIGEgd2hvbGUgcmVjb3JkaW5nIHNhbXBsZWQgYXQgZnMuIiIiCiAgICByZXR1cm4gRi5sb2dzcGVjKEYuYmFuZHBhc3MoRi50b18xayhucC5hc2FycmF5KHgsIG5wLmZsb2F0NjQpLCBmcykpKQoKCmRlZiBsb2FkX21vZGVscyhmb2xkZXIsIGtpbmQsIGRldmljZT0nY3B1Jyk6CiAgICAiIiJUaGUgdGhyZWUgZnJvemVuIG1vZGVscyBvZiBhcm0gYGtpbmRgICgnVCcgb3IgJ1cnKSBmcm9tIGEgY2hlY2twb2ludCBmb2xkZXIsIGVhY2ggY2hlY2tlZCBhZ2FpbnN0IGZyZWV6ZS5qc29uCiAgICAoZmlsZSBhbmQgd2VpZ2h0IGhhc2hlcykuIFJldHVybnMgKFsobmFtZSwgbW9kZWwpXSwgZnJlZXplIHJlY29yZCkuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyID0ganNvbi5sb2FkKG9wZW4ob3MucGF0aC5qb2luKGZvbGRlciwgJ2ZyZWV6ZS5qc29uJykpKQogICAgYXNzZXJ0IG5vdCBmci5nZXQoJ3Ntb2tlJyksICdjaGVja3BvaW50IGZvbGRlciBob2xkcyBhIHNtb2tlIHJ1bicKICAgIG91dCA9IFtdCiAgICBmb3IgbmFtZSBpbiBzb3J0ZWQoZnJbJ21vZGVscyddKToKICAgICAgICBpZiBub3QgbmFtZS5zdGFydHN3aXRoKGtpbmQgKyAnXycpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG0gPSBmclsnbW9kZWxzJ11bbmFtZV0KICAgICAgICBwYXRoID0gb3MucGF0aC5qb2luKGZvbGRlciwgbmFtZSArICcucHQnKQogICAgICAgIGFzc2VydCBzaGEyNTZfZmlsZShwYXRoKSA9PSBtWydmaWxlX3NoYTI1NiddLCAnY2hlY2twb2ludCBmaWxlIGRpZmZlcnMgZnJvbSBmcmVlemUuanNvbjogJXMnICUgbmFtZQogICAgICAgIGNrID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249J2NwdScsIHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICBuZXQgPSBNLlVOZXQxRChraW5kKQogICAgICAgIG5ldC5sb2FkX3N0YXRlX2RpY3QoY2tbJ3N0YXRlX2RpY3QnXSkKICAgICAgICBhc3NlcnQgc3RhdGVfc2hhMjU2KG5ldC5zdGF0ZV9kaWN0KCkpID09IG1bJ3dlaWdodHNfc2hhMjU2J10sICd3ZWlnaHRzIGRpZmZlciBmcm9tIGZyZWV6ZS5qc29uOiAlcycgJSBuYW1lCiAgICAgICAgb3V0LmFwcGVuZCgobmFtZSwgbmV0LmV2YWwoKS50byhkZXZpY2UpKSkKICAgIGFzc2VydCBsZW4ob3V0KSA9PSAzLCAnZXhwZWN0ZWQgdGhyZWUgc2VlZHMgZm9yICVzLCBmb3VuZCAlZCcgJSAoa2luZCwgbGVuKG91dCkpCiAgICByZXR1cm4gb3V0LCBmcgoKCmRlZiB3aW5kb3dzKG4sIHdpbj1XSU5fRlJBTUVTLCBvdmVybGFwPU9WRVJMQVApOgogICAgIiIiWyhzdGFydCwgZW5kKV0gZnJhbWUgd2luZG93cyBjb3ZlcmluZyBuIGZyYW1lczogb25lIHdpbmRvdyB3aGVuIG4gPD0gd2luLCBlbHNlIHdpbi1sb25nIHdpbmRvd3MgZXZlcnkKICAgIHdpbiAtIG92ZXJsYXAgZnJhbWVzIHdpdGggdGhlIGxhc3Qgb25lIGFsaWduZWQgdG8gdGhlIGVuZC4iIiIKICAgIGlmIG4gPD0gd2luOgogICAgICAgIHJldHVybiBbKDAsIG4pXQogICAgc3RhcnRzID0gc29ydGVkKHNldChsaXN0KHJhbmdlKDAsIG4gLSB3aW4sIHdpbiAtIG92ZXJsYXApKSArIFtuIC0gd2luXSkpCiAgICByZXR1cm4gWyhzLCBzICsgd2luKSBmb3IgcyBpbiBzdGFydHNdCgoKZGVmIF9mYWRlKGEsIGIsIG4sIG92ZXJsYXA9T1ZFUkxBUCk6CiAgICB3ID0gbnAub25lcyhiIC0gYSkKICAgIGlmIGEgPiAwOgogICAgICAgIGsgPSBtaW4ob3ZlcmxhcCwgYiAtIGEpCiAgICAgICAgd1s6a10gPSBucC5saW5zcGFjZSgwLjAsIDEuMCwgayArIDIpWzE6LTFdCiAgICBpZiBiIDwgbjoKICAgICAgICBrID0gbWluKG92ZXJsYXAsIGIgLSBhKQogICAgICAgIHdbLWs6XSA9IG5wLm1pbmltdW0od1stazpdLCBucC5saW5zcGFjZSgxLjAsIDAuMCwgayArIDIpWzE6LTFdKQogICAgcmV0dXJuIHcKCgpkZWYgZm9yd2FyZChtb2RlbHMsIHNwZWMsIGtpbmQsIGRldmljZT0nY3B1Jyk6CiAgICAiIiJTZWVkLWF2ZXJhZ2VkIHRyYWNlcyAoaGVhZHMsIGZyYW1lcykgZm9yIGEgd2hvbGUgcmVjb3JkaW5nOiBUIC0+IChzMSwgczIsIHIpIHByb2JhYmlsaXRpZXMsIFcgLT4gKGVjZywpLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBuID0gc3BlYy5zaGFwZVsxXQogICAgYWNjID0gbnAuemVyb3MoKGxlbihNLkhFQURTW2tpbmRdKSwgbikpCiAgICB3c3VtID0gbnAuemVyb3MobikKICAgIGZvciBhLCBiIGluIHdpbmRvd3Mobik6CiAgICAgICAgc2VnID0gbnAuemVyb3MoKHNwZWMuc2hhcGVbMF0sIFdJTl9GUkFNRVMpLCBucC5mbG9hdDMyKQogICAgICAgIHNlZ1s6LCA6YiAtIGFdID0gc3BlY1s6LCBhOmJdCiAgICAgICAgeCA9IHRvcmNoLnRlbnNvcihzZWdbTm9uZV0sIGRldmljZT1kZXZpY2UpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIG91dHMgPSBbbmV0KHgpLmZsb2F0KCkuY3B1KCkubnVtcHkoKVswLCA6LCA6YiAtIGFdLmFzdHlwZShucC5mbG9hdDY0KSBmb3IgXywgbmV0IGluIG1vZGVsc10KICAgICAgICBvID0gbnAubWVhbihbMS4wIC8gKDEuMCArIG5wLmV4cCgteSkpIGZvciB5IGluIG91dHNdLCAwKSBpZiBraW5kICE9ICdXJyBlbHNlIG5wLm1lYW4ob3V0cywgMCkKICAgICAgICB3ID0gX2ZhZGUoYSwgYiwgbikKICAgICAgICBhY2NbOiwgYTpiXSArPSBvICogdwogICAgICAgIHdzdW1bYTpiXSArPSB3CiAgICByZXR1cm4gYWNjIC8gd3N1bQoKCmRlZiBiZWF0cyh0cmFjZSwga2luZCk6CiAgICAiIiJCZWF0IHRpbWVzIGluIHNlY29uZHMgZnJvbSB0aGUgc3RhcnQgb2YgdGhlIHJlY29yZGluZyAoVDogUzEgZXZlbnRzOyBXOiBSLXBlYWtzIG9mIHRoZSBzeW50aGV0aWMgRUNHKSwgYW5kIHRoZQogICAgZGVjb2RlcidzIHJlY29yZCBzdW1tYXJ5LiIiIgogICAgaWYga2luZCAhPSAnVyc6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBUIGFuZCB0aGUgYWNvdXN0aWMtbGFiZWwgY29udHJvbCBBOiBTMSBldmVudHMgZnJvbSB0aGUgUzEtbWludXMtUzIgdHJhY2UKICAgICAgICBkZWMgPSBELmRlY29kZV9iZWF0cyhELmJlYXRfdHJhY2VfZnJvbV9oZWFkcyh0cmFjZVswXSwgdHJhY2VbMV0pLCAyNTAsIGtpbmQ9J2V2ZW50JykKICAgIGVsc2U6CiAgICAgICAgZGVjID0gRC5kZWNvZGVfYmVhdHModHJhY2VbMF0sIDI1MCwga2luZD0nZWNnJykKICAgIHJldHVybiBGUkFNRV9UMCArIG5wLmFzYXJyYXkoZGVjWydiZWF0c19zJ10sIGZsb2F0KSwgZGVjWydyZWMnXQoKCmRlZiB0cmFjZV9oYXNoKHRyYWNlKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihucC5hc2NvbnRpZ3VvdXNhcnJheShucC5hc2FycmF5KHRyYWNlLCBucC5mbG9hdDMyKSkudG9ieXRlcygpKS5oZXhkaWdlc3QoKVs6MTZdCg==", "cycles.py": "IiIiQ3ljbGUgZXhwb3J0IGZvciB0aGUgZnJvemVuIFQgYW5kIFcgYXJtcyAoUnVuIDIpOiBSMSBjeWNsZSBmb3JtYXQgYXQgNCBrSHosIGJvdW5kYXJpZXMgYXQgdGhlIDUwJSBhY291c3RpYyBvbnNldC4KClcgaXMgYSBzeW50aGV0aWMgRUNHLCBzbyBpdHMgUi1wZWFrcyBlbnRlciBhcm0gQidzIGNvbnN0cnVjdGlvbiB1bmNoYW5nZWQgKGBiX2N5Y2xlc2AsIHRoZSBSMSBub3RlYm9vaydzIGFybV9iX2N5Y2xlcwp3aXRoIHRoZSBhdWRpbyBwYXNzZWQgaW4pOiBlYWNoIFItcGVhayBzZWFyY2hlcyB0aGUgMjAgdG8gMTUwIEh6IGFjb3VzdGljIGVudmVsb3BlIGZvciBTMSBhbmQgYW5jaG9ycyB0aGUgY3ljbGUgYXQKdGhlIDUwJSBvbnNldDsgUzIgaXMgdGhlIHN0cm9uZ2VzdCBlbnZlbG9wZSBtYXhpbXVtIGluIFswLjI1LCAwLjU1XSBSUi4gVCBwcmVkaWN0cyB0aGUgaGVhcnQgc291bmRzIHRoZW1zZWx2ZXMsIHNvIGl0cwpjeWNsZXMgY29tZSBmcm9tIGl0cyBvd24gaGVhZHMgKGB0X2N5Y2xlc2AsIDA4IGRlc2lnbiBzZWN0aW9uIDIuNyk6IHRoZSBjeWNsZSBhbmNob3IgaXMgdGhlIDUwJSBhY291c3RpYyBvbnNldCBvZiB0aGUKZW52ZWxvcGUgbWF4aW11bSB3aXRoaW4gNDAgbXMgb2YgZWFjaCBkZWNvZGVkIFMxIGV2ZW50OyBTMiBpcyBzZWFyY2hlZCBpbiB0aGUgc2FtZSBoZWFydC1yYXRlLWRlcGVuZGVudCB3aW5kb3cgYXMgYXJtCkIsIGFzIHRoZSBtYXhpbXVtIG9mIFQncyBTMiBwcm9iYWJpbGl0eSB0aW1lcyBhIEdhdXNzaWFuIChTRCA0MCBtcykgY2VudHJlZCBvbiB0aGUgZXhwZWN0ZWQgUzEtdG8tUzIgaW50ZXJ2YWwgb2YgdGhlCnBhaXJlZCBsYXcgYXQgdGhlIGxvY2FsIHJhdGUsIHRoZW4gbW92ZWQgdG8gdGhlIGVudmVsb3BlIG1heGltdW0gd2l0aGluIDQwIG1zIGFuZCB0byBpdHMgNTAlIG9uc2V0LiBQaGFzZXMgZm9sbG93IGFybQpCIChTMSBmb3IgbWluKDAuMTUgUlIsIDEyMCBtcyksIHN5c3RvbGUgdG8gdGhlIFMyIG9uc2V0LCBTMiBmb3IgbWluKDAuMTAgUlIsIDEwMCBtcyksIGRpYXN0b2xlIHRvIHRoZSBuZXh0IGFuY2hvcik7Cm1hbGZvcm1lZCBjeWNsZXMgYXJlIGRyb3BwZWQgYW5kIGNvdW50ZWQuIEZVU0lPTiwgY2hlY2tfY3ljbGVzLCBhY291c3RpY19lbnZlbG9wZSwgb25zZXRfNTAgYW5kIGJfY3ljbGVzIGFyZSBjb3BpZWQKZnJvbSB0aGUgUjEgbm90ZWJvb2s7IHRlc3RzIGhvbGQgdGhlbSBlcXVhbCB0byB0aGUgc291cmNlLiBDb25zdGFudHMgYXJlIGZpeGVkIGluIGFkdmFuY2U7IG5vdGhpbmcgcmVhZHMgYSBsYWJlbC4KIiIiCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgc2NpcHkuc2lnbmFsIGFzIHNpZwoKVEFSR0VUX1NSID0gNDAwMApQSEFTRV9LRVlTID0gKCdzMScsICdzeXN0b2xlJywgJ3MyJywgJ2RpYXN0b2xlJykKCkZVU0lPTiA9IGRpY3QoYmFuZF9oej0oMjAsIDE1MCksIGVudl9zbW9vdGhfbXM9MjAsIHBlYWtfZGlzdF9tcz0yNTAsIHBlYWtfcHJvbT0wLjEwLAogICAgICAgICAgICAgIHNlYXJjaF9sb19tcz0tNDAsIHNlYXJjaF9oaV9tcz0xMjAsIG1hdGNoX21zPTE1MCwgZ2FwX2ZhY3Rvcj0xLjYsIHN1cHBvcnRfbWluPTAuNSwKICAgICAgICAgICAgICBzMl9sbz0wLjI1LCBzMl9oaT0wLjU1LCBzMV9mcmFjPTAuMTUsIHMxX2NhcF9tcz0xMjAsIHMyX2ZyYWM9MC4xMCwgczJfY2FwX21zPTEwMCwKICAgICAgICAgICAgICBzMV9vbnNldF9iYWNrX21zPTE1MCwgczJfb25zZXRfYmFja19tcz0xMDAsIHJlc2N1ZV9oZWlnaHQ9MC41MCkKQUNfU09TID0gc2lnLmJ1dHRlcig0LCBsaXN0KEZVU0lPTlsnYmFuZF9oeiddKSwgYnR5cGU9J2JhbmRwYXNzJywgZnM9VEFSR0VUX1NSLCBvdXRwdXQ9J3NvcycpCgoKZGVmIGNoZWNrX2N5Y2xlcyhjeWMsIEwsIHRhZywgc3RyaWN0PVRydWUpOgogICAgIiIiRXZlcnkgYm91bmRhcnkgaW5zaWRlIFswLCBMXSwgZXZlcnkgcGhhc2Ugb3JkZXJlZCAoc3RhcnQgPD0gZW5kKSwgcGhhc2Ugc3RhcnRzIG5vbi1kZWNyZWFzaW5nLgogICAgc3RyaWN0PVRydWUgYWRkaXRpb25hbGx5IHJlcXVpcmVzIHRoZSBlaWdodCBib3VuZGFyaWVzIHRvIGJlIG1vbm90b25lOyB0aGF0IGhvbGRzIGZvciBldmVyeSBhcm0KICAgIGJ1aWx0IGhlcmUuIFRoZSBkYXRhc2V0IGFubm90YXRpb24gKGFybSBDKSBvY2Nhc2lvbmFsbHkgb3ZlcmxhcHMgYWRqYWNlbnQgcGhhc2VzIGJ5IGEgc2FtcGxlIG9yCiAgICB0d28sIHNvIGFybSBDIGFuZCBpdHMgaml0dGVyZWQgY29waWVzIGFyZSBjaGVja2VkIHdpdGggc3RyaWN0PUZhbHNlIGFuZCBwaGFzZV9vdmVybGFwcygpIGNvdW50cyB0aGVtLiIiIgogICAgZm9yIGMgaW4gY3ljOgogICAgICAgIHNlcSA9IFt2IGZvciBrIGluIFBIQVNFX0tFWVMgZm9yIHYgaW4gY1trXV0KICAgICAgICBhc3NlcnQgYWxsKDAgPD0gdiA8PSBMIGZvciB2IGluIHNlcSksICclczogYm91bmRhcnkgb3V0c2lkZSB0aGUgc2lnbmFsICVzIChMPSVkKScgJSAodGFnLCBzZXEsIEwpCiAgICAgICAgYXNzZXJ0IGFsbChjW2tdWzBdIDw9IGNba11bMV0gZm9yIGsgaW4gUEhBU0VfS0VZUyksICclczogcGhhc2Ugd2l0aCBlbmQgYmVmb3JlIHN0YXJ0ICVzJyAlICh0YWcsIHNlcSkKICAgICAgICBzdGFydHMgPSBbY1trXVswXSBmb3IgayBpbiBQSEFTRV9LRVlTXQogICAgICAgIGFzc2VydCBhbGwoYSA8PSBiIGZvciBhLCBiIGluIHppcChzdGFydHMsIHN0YXJ0c1sxOl0pKSwgJyVzOiBwaGFzZSBzdGFydHMgbm90IG9yZGVyZWQgJXMnICUgKHRhZywgc2VxKQogICAgICAgIGlmIHN0cmljdDoKICAgICAgICAgICAgYXNzZXJ0IGFsbChhIDw9IGIgZm9yIGEsIGIgaW4gemlwKHNlcSwgc2VxWzE6XSkpLCAnJXM6IGJvdW5kYXJpZXMgbm90IG1vbm90b25lICVzJyAlICh0YWcsIHNlcSkKICAgIHJldHVybiBjeWMKCgpkZWYgYWNvdXN0aWNfZW52ZWxvcGUoeDQsIHNyPVRBUkdFVF9TUik6CiAgICBmID0gc2lnLnNvc2ZpbHRmaWx0KEFDX1NPUywgeDQuYXN0eXBlKG5wLmZsb2F0NjQpKQogICAgdyA9IGludChGVVNJT05bJ2Vudl9zbW9vdGhfbXMnXSAvIDEwMDAgKiBzcikKICAgIGVudiA9IG5wLmNvbnZvbHZlKG5wLmFicyhmKSwgbnAub25lcyh3KSAvIHcsIG1vZGU9J3NhbWUnKQogICAgcmV0dXJuIGVudiAvIChucC5wZXJjZW50aWxlKGVudiwgOTUpICsgMWUtMTIpCgoKZGVmIG9uc2V0XzUwKGVudiwgcGVhaywgYmFjayk6CiAgICAiIiJMYXN0IHNhbXBsZSBiZWZvcmUgYHBlYWtgIHdoZXJlIGVudiBjcm9zc2VzIDUwICUgb2YgdGhlIHBlYWsgaGVpZ2h0LCB3aXRoaW4gYGJhY2tgIHNhbXBsZXMuCiAgICBSZXR1cm5zIChvbnNldCwgZm91bmQpLiIiIgogICAgaCA9IDAuNSAqIGVudltwZWFrXQogICAgYSA9IG1heCgwLCBpbnQocGVhaykgLSBpbnQoYmFjaykpCiAgICBiZWxvdyA9IG5wLmZsYXRub256ZXJvKGVudlthOnBlYWtdIDwgaCkKICAgIGlmIGxlbihiZWxvdykgPT0gMDoKICAgICAgICByZXR1cm4gaW50KGEpLCBGYWxzZQogICAgcmV0dXJuIGludChhICsgYmVsb3dbLTFdICsgMSksIFRydWUKCgpkZWYgYl9jeWNsZXMoeDQsIHBrNCk6CiAgICAiIiJhcm1fYl9jeWNsZXMgb2YgdGhlIFIxIG5vdGVib29rIHdpdGggdGhlIGF1ZGlvIHBhc3NlZCBpbiAodmVyYmF0aW0gYm9keTsgdGVzdHMgY29tcGFyZSBpdCB3aXRoIHRoZSBzb3VyY2UpLiIiIgogICAgTDQgPSBsZW4oeDQpCiAgICBzciA9IFRBUkdFVF9TUgogICAgZW52ID0gYWNvdXN0aWNfZW52ZWxvcGUoeDQpCiAgICBhY19waywgXyA9IHNpZy5maW5kX3BlYWtzKGVudiwgZGlzdGFuY2U9aW50KEZVU0lPTlsncGVha19kaXN0X21zJ10gLyAxMDAwICogc3IpLCBwcm9taW5lbmNlPUZVU0lPTlsncGVha19wcm9tJ10pCiAgICBhY19wayA9IG5wLmFzYXJyYXkoYWNfcGssIGludCkKICAgIHJhd19waywgXyA9IHNpZy5maW5kX3BlYWtzKGVudiwgcHJvbWluZW5jZT1GVVNJT05bJ3BlYWtfcHJvbSddKQogICAgc3Ryb25nID0gcmF3X3BrW2VudltyYXdfcGtdID49IEZVU0lPTlsncmVzY3VlX2hlaWdodCddXSAgICAgIyBsb2NhbCBtYXhpbWEgYXQgPj0gNTAgJSBvZiB0aGUgcDk1IGxldmVsLCBubyBzcGFjaW5nIHJ1bGUgKHJlc2N1ZSBsaXN0KQogICAgYmFjazEgPSBpbnQoRlVTSU9OWydzMV9vbnNldF9iYWNrX21zJ10gLyAxMDAwICogc3IpCiAgICBhY19vbiA9IFtvbnNldF81MChlbnYsIHAsIGJhY2sxKSBmb3IgcCBpbiBhY19wa10gICAgICAgICAgIyAob25zZXQgc2FtcGxlLCBjcm9zc2luZyBmb3VuZCkgcGVyIHRoaW5uZWQgYWNvdXN0aWMgcGVhawogICAgcjQgPSBucC5hc2FycmF5KHBrNCwgaW50KQogICAgZCA9IGRpY3Qobl9zeW49aW50KGxlbihyNCkpLCBuX2Fjb3VzdGljPWludChsZW4oYWNfcGspKSwgc3VwcG9ydGVkX2ZyYWM9bnAubmFuLCBtZWFuX2Fic19zeW5fYWNfb2Zmc2V0X21zPW5wLm5hbiwKICAgICAgICAgICAgIG1lYW5fc3luX2FjX29mZnNldF9tcz1ucC5uYW4sIG5fcmVtb3ZlZD0wLCBuX2luc2VydGVkPTAsIG5fcmVzY3VlZD0wLCBmdXNpb25fZmFsbGJhY2s9RmFsc2UsIG5fb25zZXRfZmFsbGJhY2s9MCwKICAgICAgICAgICAgIG5fbWFsZm9ybWVkPTAsIGZyYWNfczJfYWNvdXN0aWM9bnAubmFuKQogICAgaWYgbGVuKHI0KSA8IDI6CiAgICAgICAgcmV0dXJuIFtdLCBkCiAgICBsbywgaGkgPSBpbnQoRlVTSU9OWydzZWFyY2hfbG9fbXMnXSAvIDEwMDAgKiBzciksIGludChGVVNJT05bJ3NlYXJjaF9oaV9tcyddIC8gMTAwMCAqIHNyKQogICAgdXNlZCA9IG5wLnplcm9zKGxlbihhY19wayksIGJvb2wpCiAgICBzdXBwb3J0ZWQgPSBucC56ZXJvcyhsZW4ocjQpLCBib29sKQogICAgYW5jaG9ycywgb2Zmcywgb25zZXRfZmIsIHJlc2N1ZWQgPSBbXSwgW10sIDAsIDAKICAgIGZvciBpLCByIGluIGVudW1lcmF0ZShyNCk6CiAgICAgICAgY2FuZCA9IG5wLmZsYXRub256ZXJvKChhY19wayA+PSByICsgbG8pICYgKGFjX3BrIDw9IHIgKyBoaSkgJiB+dXNlZCkKICAgICAgICBpZiBsZW4oY2FuZCk6CiAgICAgICAgICAgIGogPSBjYW5kW2ludChucC5hcmdtYXgoZW52W2FjX3BrW2NhbmRdXSkpXQogICAgICAgICAgICB1c2VkW2pdID0gVHJ1ZQogICAgICAgICAgICBvbiwgZm91bmQgPSBhY19vbltqXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJlc2MgPSBzdHJvbmdbKHN0cm9uZyA+PSByICsgbG8pICYgKHN0cm9uZyA8PSByICsgaGkpXQogICAgICAgICAgICBpZiBsZW4ocmVzYykgPT0gMDoKICAgICAgICAgICAgICAgIGFuY2hvcnMuYXBwZW5kKGludChyKSkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJlc2N1ZWQgKz0gMQogICAgICAgICAgICBvbiwgZm91bmQgPSBvbnNldF81MChlbnYsIGludChyZXNjW2ludChucC5hcmdtYXgoZW52W3Jlc2NdKSldKSwgYmFjazEpCiAgICAgICAgc3VwcG9ydGVkW2ldID0gVHJ1ZQogICAgICAgIGlmIGZvdW5kOgogICAgICAgICAgICBhbmNob3JzLmFwcGVuZChpbnQob24pKQogICAgICAgICAgICBvZmZzLmFwcGVuZCgoaW50KG9uKSAtIGludChyKSkgLyBzciAqIDEwMDAuMCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBhbmNob3JzLmFwcGVuZChpbnQocikpCiAgICAgICAgICAgIG9uc2V0X2ZiICs9IDEKICAgIGZyYWMgPSBmbG9hdChzdXBwb3J0ZWQubWVhbigpKQogICAgZC51cGRhdGUoc3VwcG9ydGVkX2ZyYWM9ZnJhYywgbl9vbnNldF9mYWxsYmFjaz1pbnQob25zZXRfZmIpLCBuX3Jlc2N1ZWQ9aW50KHJlc2N1ZWQpLAogICAgICAgICAgICAgbWVhbl9hYnNfc3luX2FjX29mZnNldF9tcz1mbG9hdChucC5tZWFuKG5wLmFicyhvZmZzKSkpIGlmIG9mZnMgZWxzZSBucC5uYW4sCiAgICAgICAgICAgICBtZWFuX3N5bl9hY19vZmZzZXRfbXM9ZmxvYXQobnAubWVhbihvZmZzKSkgaWYgb2ZmcyBlbHNlIG5wLm5hbikKICAgIGlmIGZyYWMgPj0gRlVTSU9OWydzdXBwb3J0X21pbiddOgogICAgICAgIGFuYyA9IG5wLmFycmF5KFthIGZvciBhLCBzIGluIHppcChhbmNob3JzLCBzdXBwb3J0ZWQpIGlmIHNdLCBpbnQpCiAgICAgICAgZFsnbl9yZW1vdmVkJ10gPSBpbnQoKH5zdXBwb3J0ZWQpLnN1bSgpKQogICAgICAgIHJyX3N1cCA9IG5wLmRpZmYoYW5jKQogICAgICAgIG1lZCA9IGZsb2F0KG5wLm1lZGlhbihycl9zdXApKSBpZiBsZW4ocnJfc3VwKSBlbHNlIG5wLm5hbgogICAgICAgIGlucyA9IDAKICAgICAgICBpZiBucC5pc2Zpbml0ZShtZWQpIGFuZCBtZWQgPiAwOgogICAgICAgICAgICBtdG9sID0gaW50KEZVU0lPTlsnbWF0Y2hfbXMnXSAvIDEwMDAgKiBzcikKICAgICAgICAgICAgZm9yIGogaW4gcmFuZ2UobGVuKGFjX3BrKSk6CiAgICAgICAgICAgICAgICBpZiB1c2VkW2pdIG9yIG5wLm1pbihucC5hYnMocjQgLSBhY19wa1tqXSkpIDw9IG10b2w6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIG8gPSBpbnQoYWNfb25bal1bMF0pIGlmIGFjX29uW2pdWzFdIGVsc2UgaW50KGFjX3BrW2pdKQogICAgICAgICAgICAgICAgayA9IGludChucC5zZWFyY2hzb3J0ZWQoYW5jLCBvKSkKICAgICAgICAgICAgICAgIGlmIGsgPT0gMCBvciBrID09IGxlbihhbmMpOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBnYXAgPSBhbmNba10gLSBhbmNbayAtIDFdCiAgICAgICAgICAgICAgICBpZiBnYXAgPj0gRlVTSU9OWydnYXBfZmFjdG9yJ10gKiBtZWQgYW5kIChvIC0gYW5jW2sgLSAxXSkgPj0gMC41ICogbWVkIGFuZCAoYW5jW2tdIC0gbykgPj0gMC41ICogbWVkOgogICAgICAgICAgICAgICAgICAgIGFuYyA9IG5wLmluc2VydChhbmMsIGssIG8pCiAgICAgICAgICAgICAgICAgICAgaW5zICs9IDEKICAgICAgICBkWyduX2luc2VydGVkJ10gPSBpbnQoaW5zKQogICAgICAgIGFuYyA9IGxpc3QoYW5jKQogICAgZWxzZToKICAgICAgICBhbmMgPSBsaXN0KGFuY2hvcnMpICAgICAgICAgICAgICAgICAgIyBmYWxsYmFjazogdGhlIHN1cHBvcnRlZCBhY291c3RpYyBvbnNldHMgc3RheTsgb25seSByZW1vdmFscyBhbmQgaW5zZXJ0aW9ucyBhcmUgc2tpcHBlZAogICAgICAgIGRbJ2Z1c2lvbl9mYWxsYmFjayddID0gVHJ1ZQogICAgYW5jID0gc29ydGVkKHNldChpbnQoYSkgZm9yIGEgaW4gYW5jIGlmIDAgPD0gYSA8PSBMNCkpCiAgICBjeWMsIG1hbGZvcm1lZCwgczJfYWMgPSBbXSwgMCwgMAogICAgczFfY2FwLCBzMl9jYXAgPSBpbnQoRlVTSU9OWydzMV9jYXBfbXMnXSAvIDEwMDAgKiBzciksIGludChGVVNJT05bJ3MyX2NhcF9tcyddIC8gMTAwMCAqIHNyKQogICAgYmFjazIgPSBpbnQoRlVTSU9OWydzMl9vbnNldF9iYWNrX21zJ10gLyAxMDAwICogc3IpCiAgICBmb3IgYSwgYiBpbiB6aXAoYW5jLCBhbmNbMTpdKToKICAgICAgICByciA9IGIgLSBhCiAgICAgICAgaWYgcnIgPD0gMDoKICAgICAgICAgICAgbWFsZm9ybWVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzMWUgPSBhICsgbWluKGludChGVVNJT05bJ3MxX2ZyYWMnXSAqIHJyKSwgczFfY2FwKQogICAgICAgIHcwLCB3MSA9IGEgKyBpbnQoRlVTSU9OWydzMl9sbyddICogcnIpLCBtaW4oYSArIGludChGVVNJT05bJ3MyX2hpJ10gKiByciksIEw0KQogICAgICAgIHMyb24sIGZvdW5kID0gYSArIGludCgwLjUwICogcnIpLCBGYWxzZQogICAgICAgIGlmIHcxID4gdzAgKyAyOgogICAgICAgICAgICBsb2MsIF8gPSBzaWcuZmluZF9wZWFrcyhlbnZbdzA6dzFdLCBwcm9taW5lbmNlPUZVU0lPTlsncGVha19wcm9tJ10pCiAgICAgICAgICAgIGlmIGxlbihsb2MpOgogICAgICAgICAgICAgICAgcCA9IHcwICsgaW50KGxvY1tpbnQobnAuYXJnbWF4KGVudlt3MDp3MV1bbG9jXSkpXSkKICAgICAgICAgICAgICAgIGNhbmRfb24sIG9rID0gb25zZXRfNTAoZW52LCBwLCBiYWNrMikKICAgICAgICAgICAgICAgIGlmIG9rOgogICAgICAgICAgICAgICAgICAgIHMyb24sIGZvdW5kID0gY2FuZF9vbiwgVHJ1ZQogICAgICAgIHMyZSA9IHMyb24gKyBtaW4oaW50KEZVU0lPTlsnczJfZnJhYyddICogcnIpLCBzMl9jYXApCiAgICAgICAgaWYgbm90IChhIDwgczFlIDwgczJvbiA8IHMyZSA8IGIpOgogICAgICAgICAgICBtYWxmb3JtZWQgKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHMyX2FjICs9IGludChmb3VuZCkKICAgICAgICBjeWMuYXBwZW5kKHsnY3ljbGUnOiBsZW4oY3ljKSwgJ3MxJzogKGEsIHMxZSksICdzeXN0b2xlJzogKHMxZSwgczJvbiksICdzMic6IChzMm9uLCBzMmUpLCAnZGlhc3RvbGUnOiAoczJlLCBiKX0pCiAgICBkLnVwZGF0ZShuX21hbGZvcm1lZD1pbnQobWFsZm9ybWVkKSwgZnJhY19zMl9hY291c3RpYz0oczJfYWMgLyBsZW4oY3ljKSkgaWYgY3ljIGVsc2UgbnAubmFuKQogICAgcmV0dXJuIGNoZWNrX2N5Y2xlcyhjeWMsIEw0LCAnYl9jeWNsZXMnKSwgZAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGFybSBUIChuZXc7IGNvbnN0YW50cyBmaXhlZCBiZWZvcmUgYW55IHRhcmdldC1jb2hvcnQgdXNlKQojIGV4cGVjdGVkIFMxLXBlYWstdG8tUzItcGVhayBpbnRlcnZhbCAobXMpIGF0IGhlYXJ0IHJhdGUgSFI6IG1lYW4gb3ZlciB0aGUgc2l4IHBhaXJlZCBkYXRhc2V0cyBvZiBsYXdfUzIgLSBsYXdfUzEgaW4KIyBkZXJpdmVkL2xhYmVsc192My9sYXdzLmpzb24gKFNIQS0yNTYgN2JhMWNlMzMuLi4pOiAzMTMgbXMgYXQgNjAgYnBtLCAyNTEgbXMgYXQgMTAwIGJwbSwgMTc0IG1zIGF0IDE1MCBicG0KUzJfTEFXX01TID0gKDQwNi4yNzY3LCAtMS41NDk5NTcpClRfRVhQT1JUID0gZGljdChzMV9yZWZpbmVfbXM9NDAsIHMyX3NpZ21hX21zPTQwLCBzMl9yZWZpbmVfbXM9NDAsIHMyX2xhd19tcz1TMl9MQVdfTVMsIGxhd3Nfc2hhMjU2X3ByZWZpeD0nN2JhMWNlMzNhNzg5ODQ1ZScpCgoKZGVmIHRfY3ljbGVzKHg0LCBzMV9zLCBwX3MyLCBmcmFtZV90MD0wLjAxNiwgZnJhbWVfcz0wLjAwNCwgczJfbGF3PVMyX0xBV19NUyk6CiAgICAiIiJDeWNsZXMgZnJvbSBUJ3MgZGVjb2RlZCBTMSBldmVudHMgKHMxX3MsIHNlY29uZHMpIGFuZCBpdHMgc2VlZC1hdmVyYWdlZCBTMiBwcm9iYWJpbGl0eSB0cmFjZSAocF9zMiwgYXQgMjUwIEh6LAogICAgZnJhbWUgaiBhdCBmcmFtZV90MCArIGogKiBmcmFtZV9zKS4gUmV0dXJucyAoY3ljbGVzLCBkaWFnbm9zdGljcykuIiIiCiAgICBMNCA9IGxlbih4NCkKICAgIHNyID0gVEFSR0VUX1NSCiAgICBlbnYgPSBhY291c3RpY19lbnZlbG9wZSh4NCkKICAgIGJhY2sxID0gaW50KEZVU0lPTlsnczFfb25zZXRfYmFja19tcyddIC8gMTAwMCAqIHNyKQogICAgYmFjazIgPSBpbnQoRlVTSU9OWydzMl9vbnNldF9iYWNrX21zJ10gLyAxMDAwICogc3IpCiAgICByMSA9IGludChUX0VYUE9SVFsnczFfcmVmaW5lX21zJ10gLyAxMDAwICogc3IpCiAgICByMiA9IGludChUX0VYUE9SVFsnczJfcmVmaW5lX21zJ10gLyAxMDAwICogc3IpCiAgICBzZCA9IFRfRVhQT1JUWydzMl9zaWdtYV9tcyddIC8gMTAwMCAqIHNyCiAgICBzMV9zID0gbnAuYXNhcnJheShzMV9zLCBmbG9hdCkKICAgIGQgPSBkaWN0KG5fYmVhdHM9aW50KGxlbihzMV9zKSksIG5fYW5jaG9ycz0wLCBuX29uc2V0X2ZhbGxiYWNrPTAsIG5fbWFsZm9ybWVkPTAsIGZyYWNfczJfYWNvdXN0aWM9bnAubmFuLAogICAgICAgICAgICAgczJfaGVhZF9hdF9jaG9pY2VfbWVkaWFuPW5wLm5hbikKICAgIGFuY2hvcnMgPSB7fQogICAgZm9yIHQgaW4gczFfczoKICAgICAgICBwID0gaW50KHJvdW5kKHQgKiBzcikpCiAgICAgICAgaWYgbm90IDAgPD0gcCA8IEw0OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGEsIGIgPSBtYXgoMCwgcCAtIHIxKSwgbWluKEw0LCBwICsgcjEgKyAxKQogICAgICAgIHEgPSBhICsgaW50KG5wLmFyZ21heChlbnZbYTpiXSkpICAgICAgICAgICAgICAgICAgICAgICMgUzEgcGVhayBvbiB0aGUgYWNvdXN0aWMgZW52ZWxvcGUKICAgICAgICBvbiwgZm91bmQgPSBvbnNldF81MChlbnYsIHEsIGJhY2sxKQogICAgICAgIGlmIG5vdCBmb3VuZDoKICAgICAgICAgICAgZFsnbl9vbnNldF9mYWxsYmFjayddICs9IDEKICAgICAgICBhbmNob3JzW2ludChvbikgaWYgZm91bmQgZWxzZSBpbnQocSldID0gaW50KHEpICAgICAgICAgICMgYW5jaG9yICg1MCUgb25zZXQpIC0+IGl0cyBTMSBwZWFrOyBkdXBsaWNhdGVzIGNvbGxhcHNlCiAgICBhbmMgPSBzb3J0ZWQoYSBmb3IgYSBpbiBhbmNob3JzIGlmIDAgPD0gYSA8PSBMNCkKICAgIGRbJ25fYW5jaG9ycyddID0gbGVuKGFuYykKICAgIHBfczIgPSBucC5hc2FycmF5KHBfczIsIGZsb2F0KQogICAgdGYgPSBmcmFtZV90MCArIGZyYW1lX3MgKiBucC5hcmFuZ2UobGVuKHBfczIpKQogICAgczFfY2FwLCBzMl9jYXAgPSBpbnQoRlVTSU9OWydzMV9jYXBfbXMnXSAvIDEwMDAgKiBzciksIGludChGVVNJT05bJ3MyX2NhcF9tcyddIC8gMTAwMCAqIHNyKQogICAgY3ljLCBtYWxmb3JtZWQsIHMyX2FjLCBoZWFkID0gW10sIDAsIDAsIFtdCiAgICBmb3IgYSwgYiBpbiB6aXAoYW5jLCBhbmNbMTpdKToKICAgICAgICByciA9IGIgLSBhCiAgICAgICAgaWYgcnIgPD0gMDoKICAgICAgICAgICAgbWFsZm9ybWVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzMWUgPSBhICsgbWluKGludChGVVNJT05bJ3MxX2ZyYWMnXSAqIHJyKSwgczFfY2FwKQogICAgICAgIHcwLCB3MSA9IGEgKyBpbnQoRlVTSU9OWydzMl9sbyddICogcnIpLCBtaW4oYSArIGludChGVVNJT05bJ3MyX2hpJ10gKiByciksIEw0KQogICAgICAgIHMyb24sIGZvdW5kID0gYSArIGludCgwLjUwICogcnIpLCBGYWxzZQogICAgICAgIGlmIHcxID4gdzAgKyAyIGFuZCBsZW4ocF9zMik6CiAgICAgICAgICAgIGhyID0gNjAuMCAqIHNyIC8gcnIKICAgICAgICAgICAgbXUgPSBhbmNob3JzW2FdICsgKHMyX2xhd1swXSArIHMyX2xhd1sxXSAqIGhyKSAvIDEwMDAgKiBzcgogICAgICAgICAgICBncmlkID0gbnAuYXJhbmdlKHcwLCB3MSkKICAgICAgICAgICAgcHMyID0gbnAuaW50ZXJwKGdyaWQgLyBzciwgdGYsIHBfczIpCiAgICAgICAgICAgIHQyID0gaW50KGdyaWRbaW50KG5wLmFyZ21heChwczIgKiBucC5leHAoLTAuNSAqICgoZ3JpZCAtIG11KSAvIHNkKSAqKiAyKSkpXSkKICAgICAgICAgICAgaGVhZC5hcHBlbmQoZmxvYXQobnAuaW50ZXJwKHQyIC8gc3IsIHRmLCBwX3MyKSkpCiAgICAgICAgICAgIGMwLCBjMSA9IG1heCh3MCwgdDIgLSByMiksIG1pbih3MSwgdDIgKyByMiArIDEpCiAgICAgICAgICAgIHEyID0gYzAgKyBpbnQobnAuYXJnbWF4KGVudltjMDpjMV0pKQogICAgICAgICAgICBvbjIsIG9rID0gb25zZXRfNTAoZW52LCBxMiwgYmFjazIpCiAgICAgICAgICAgIGlmIG9rOgogICAgICAgICAgICAgICAgczJvbiwgZm91bmQgPSBpbnQob24yKSwgVHJ1ZQogICAgICAgIHMyZSA9IHMyb24gKyBtaW4oaW50KEZVU0lPTlsnczJfZnJhYyddICogcnIpLCBzMl9jYXApCiAgICAgICAgaWYgbm90IChhIDwgczFlIDwgczJvbiA8IHMyZSA8IGIpOgogICAgICAgICAgICBtYWxmb3JtZWQgKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHMyX2FjICs9IGludChmb3VuZCkKICAgICAgICBjeWMuYXBwZW5kKHsnY3ljbGUnOiBsZW4oY3ljKSwgJ3MxJzogKGEsIHMxZSksICdzeXN0b2xlJzogKHMxZSwgczJvbiksICdzMic6IChzMm9uLCBzMmUpLCAnZGlhc3RvbGUnOiAoczJlLCBiKX0pCiAgICBkLnVwZGF0ZShuX21hbGZvcm1lZD1pbnQobWFsZm9ybWVkKSwgZnJhY19zMl9hY291c3RpYz0oczJfYWMgLyBsZW4oY3ljKSkgaWYgY3ljIGVsc2UgbnAubmFuLAogICAgICAgICAgICAgczJfaGVhZF9hdF9jaG9pY2VfbWVkaWFuPWZsb2F0KG5wLm1lZGlhbihoZWFkKSkgaWYgaGVhZCBlbHNlIG5wLm5hbikKICAgIHJldHVybiBjaGVja19jeWNsZXMoY3ljLCBMNCwgJ3RfY3ljbGVzJyksIGQK"}
_MOD_SHA = {"__init__.py": "012ad603c7da722a8234aa407cbb3b75ce077f4807d0fd0f96a9b6314b8cde63", "frontend.py": "d40f7c891441982239c8a71577e85abada77a3966619e908b97c331bc6f65760", "model.py": "2b73aecb8543b04398cc6ceb2f8a7630662d273b333f3458f6948c6186014b95", "robust_beats.py": "09bb9d7f22105eefd31cd7ae883da95af98e7d05dd8e126c8e20799035d4c077", "decoder.py": "daa8797c9a475e2085b1298ad224563a68080c54ec5c3b7beb47b9b2099913ed", "infer.py": "8a8999e8bf624fb8b6256da6d93c66e66d4fe082d8d8d3c39493ef3dcb21d819", "cycles.py": "6fccf6cef7ee3e19707768f0341d3398ceff0e23a46ce1746d385f36b016498d"}
_code_root = (Path('/content') if IN_COLAB else RUN_DIR) / 'run3_code'
(_code_root / 'extstudy').mkdir(parents=True, exist_ok=True)
for _m, _b in _MOD_B64.items():
    _src = _b64r.b64decode(_b)
    assert hashlib.sha256(_src).hexdigest() == _MOD_SHA[_m], 'embedded module %s corrupted' % _m
    (_code_root / 'extstudy' / _m).write_bytes(_src)
for _k in [k for k in sys.modules if k == 'extstudy' or k.startswith('extstudy.')]:
    del sys.modules[_k]
sys.path.insert(0, str(_code_root))
from extstudy import infer as TWI, cycles as TWC
assert TWC.FUSION == FUSION and TWC.PHASE_KEYS == tuple(PHASE_KEYS) and TWC.TARGET_SR == TARGET_SR, 'cycle constants differ from arm B'

FREEZE_SHA = '30c9ce35ce6aefccc719e0d8750aaca2634eef42ea236017af5ff5601a69861e'
_tw_local = (Path('/content') if IN_COLAB else RUN_DIR) / 'ac_checkpoints' / TW_STAMP
_tw_local.mkdir(parents=True, exist_ok=True)
for _f in sorted(p.name for p in TW_SRC.iterdir() if p.suffix in ('.json', '.pt')):
    _drive_retry(_sh2.copyfile, TW_SRC / _f, _tw_local / _f) if IN_COLAB else _sh2.copyfile(TW_SRC / _f, _tw_local / _f)
assert hashlib.sha256((_tw_local / 'freeze.json').read_bytes()).hexdigest() == FREEZE_SHA, 'freeze.json differs from the frozen record'
FREEZE_REC = json.load(open(_tw_local / 'freeze.json'))
A_MODELS, _ = TWI.load_models(str(_tw_local), 'A', DEV)          # file and weight hashes checked inside
TW_WEIGHTS_OK = True
TW_CFG = dict(freeze_sha256=FREEZE_SHA, modules=_MOD_SHA, win_frames=TWI.WIN_FRAMES, overlap=TWI.OVERLAP, t_export=TWC.T_EXPORT,
              n_min=N_MIN_CYCLES, fusion=FUSION, kind='A')
TW_CFG_HASH = hashlib.sha1(json.dumps(TW_CFG, sort_keys=True, default=str).encode()).hexdigest()[:16]
print('frozen control arm A loaded and verified: %s | configuration hash %s' % ([n for n, _ in A_MODELS], TW_CFG_HASH))

RUN1_CYCLES_HASH = {"pcg_seg": "b30ea82ce6cd1abc", "equal_time": "fab2e1d5ac6454e3", "equal_time_annot": "74c6e04524b445d8"}
RUN1_E_FEATURES_SHA = '251c830ca14f5de6743dbe255f8b017f8d9504ecda43795f91f17ba64871d755'
_E1 = '...'
assert hashlib.sha256(_E1.encode()).hexdigest() == '520bf9cbfb7095b2b62c70f7ce47f9979924321996d5e0db4cb66406cfb2dcea', 'embedded Run 1 arm E predictions corrupted'
RUN1_E = json.loads(_zr.decompress(_b64r.b64decode(_E1)).decode())
RUN1_OOF_SHA256 = 'eeeb2b95237a0fe610b9f97797acdc5cea40c2354ce75deeb46febbccee57bcb'

TW_RAW = {}          # {name: dict(bt, p_s2, rec)} for the forked cycle workers

def _tw_worker(name):
    """T_ac cycles from its S1 and S2 heads (cycles.t_cycles, the construction of T); equal-time fallback below 2 cycles."""
    try:
        L4 = AUDIO_META[name]['L4']
        x4 = np.asarray(AUDIO4[name][:L4], np.float64)
        raw = TW_RAW[name]
        cyc_t, d_t = TWC.t_cycles(x4, raw['bt'], raw['p_s2'], TWI.FRAME_T0, TWI.FRAME_S)
        d = dict(d_t, fallback=False, n_cycles_net=len(cyc_t))
        if len(cyc_t) < N_MIN_CYCLES:
            cyc_t, d['fallback'] = equal_time_cycles(L4, TARGET_SR), True
        return name, {'timing_net_ac': (cyc_t, d)}, None
    except Exception as e:
        return name, None, '%s: %s' % (type(e).__name__, e)

In [ ]:
CYC = {arm: {} for arm in CYC_ARMS}
ARM_DIAG = {arm: {} for arm in CYC_ARMS}
ARM_FAILURES = []
_CD_ERROR = ([], {'worker_error': True, 'n_annotated': 0, 'n_beyond_portion': 0, 'n_phase_overlaps': 0, 'max_phase_overlap_samples': 0,
                  'annotated_coverage_fraction': np.nan, 'annotated_span_fraction': np.nan},
             [], {'n_anchors': 0, 'n_annotation_gaps': 0}, [], {}, [], {'n_matched_to_annotation': 0})

with stage('S4 boundaries'):
    # ---- arms C and E (Run 1's worker; rebuilt only to reproduce Run 1 and to give the annotated stretches) ----
    cd = {}
    t0 = time.time()
    with _pool(N_PROC) as ex:
        for name, payload, err in tqdm(ex.map(_cd_worker, NAMES_ALL, chunksize=16), total=len(NAMES_ALL), desc='arms C/E', leave=False):
            if err is not None:
                ARM_FAILURES.append((name, 'C/E', err))
                payload = _CD_ERROR
            cd[name] = payload
    for n in NAMES_ALL:
        cyc_c, d_c, _cd_d, _dd_d, cyc_e, d_e, _ce2, _de2 = cd[n]
        CYC['pcg_seg'][n], ARM_DIAG['pcg_seg'][n] = cyc_c, d_c
        CYC['equal_time'][n], ARM_DIAG['equal_time'][n] = cyc_e, d_e
    del cd
    print('   arms C and E built for %d recordings in %.0f s' % (len(NAMES_ALL), time.time() - t0))

    # ---- the kept set: Run 1's 3,000 recordings (full run) ----
    if VALIDATION_APPLICABLE:
        _missing = [n for n in PINNED_KEEP if n not in set(NAMES_ALL)]
        assert not _missing, 'Run 1 recordings missing from this index: %s' % _missing[:5]
        KEEP = list(PINNED_KEEP)
    else:
        KEEP = [n for n in NAMES_ALL if len(CYC['pcg_seg'][n]) >= N_MIN_CYCLES and len(CYC['equal_time'][n]) >= N_MIN_CYCLES]
    _kset = set(KEEP)
    DROPPED = [(n, ['not in the Run 1 kept set']) for n in NAMES_ALL if n not in _kset]
    KEEP_ARR = np.array(KEEP)
    KEEP_PINNED_OK = (list(KEEP) == list(PINNED_KEEP)) if VALIDATION_APPLICABLE else None
    print('   kept set: %d recordings%s' % (len(KEEP), ' (the Run 1 kept set, pinned)' if VALIDATION_APPLICABLE else ''))

    # ---- the frozen control arm A on the GPU: seed-averaged traces, beats from the shared decoder ----
    fp_tw = CACHE_DIR / 'tw_traces.pkl'
    if fp_tw.exists():
        try:
            with open(fp_tw, 'rb') as fh:
                _o = pickle.load(fh)
            if _o.get('cfg') == TW_CFG_HASH:
                TW_RAW.update(_o['data'])
        except Exception:
            pass
    todo = [n for n in KEEP if n not in TW_RAW]
    t0 = time.time()
    for n in tqdm(todo, desc='control arm inference', leave=False):
        x4 = np.asarray(AUDIO4[n][:AUDIO_META[n]['L4']], np.float64)
        spec = TWI.spectrogram(x4, TARGET_SR)
        tt = TWI.forward(A_MODELS, spec, 'A', DEV)
        bt, rec_t = TWI.beats(tt, 'A')
        TW_RAW[n] = dict(bt=bt, p_s2=tt[1].astype(np.float32), rec_t=rec_t, trace_sha_t=TWI.trace_hash(tt))
    if todo:
        atomic_pickle(fp_tw, {'cfg': TW_CFG_HASH, 'data': TW_RAW})
        print('   control arm run on %d recordings in %.0f s' % (len(todo), time.time() - t0))
    else:
        print('   control-arm traces loaded from cache')
    TW_TRACES_SHA = hashlib.sha256(''.join(TW_RAW[n]['trace_sha_t'] for n in KEEP).encode()).hexdigest()

    # ---- T_ac cycles (worker pool) and the matched-coverage copy ----
    t0 = time.time()
    tw_out = {}
    with _pool(N_PROC) as ex:
        for name, payload, err in tqdm(ex.map(_tw_worker, KEEP, chunksize=8), total=len(KEEP), desc='arm T_ac', leave=False):
            if err is not None:
                ARM_FAILURES.append((name, 'T_ac', err))
                L4_ = AUDIO_META[name]['L4']
                payload = {a: (equal_time_cycles(L4_, TARGET_SR), dict(fallback=True, worker_error=err)) for a in TW_ARMS}
            tw_out[name] = payload
    for n in KEEP:
        for a in TW_ARMS:
            CYC[a][n], ARM_DIAG[a][n] = tw_out[n][a]
        spans = annotated_spans(CYC['pcg_seg'][n])
        for a, src in (('timing_net_ac_annot', 'timing_net_ac'), ('equal_time_annot', 'equal_time')):
            c_ = restrict_to(CYC[src][n], spans)
            fb = len(c_) < N_MIN_CYCLES
            CYC[a][n] = CYC[src][n] if fb else c_
            ARM_DIAG[a][n] = dict(fallback=bool(fb), n_annotated_spans=len(spans), n_cycles_selected=len(c_))
    del tw_out
    print('   arm T_ac built in %.0f s' % (time.time() - t0))
    TW_FALLBACK = {a: float(np.mean([bool(ARM_DIAG[a][n].get('fallback')) for n in KEEP])) for a in TW_ARMS + TW_COV_ARMS}
    TW_FAILED = [a for a, v in TW_FALLBACK.items() if v > S_FALLBACK_FAIL]
    print('   fallback share (equal-time windows used):', {a: round(100 * v, 2) for a, v in TW_FALLBACK.items()}, '%; arms above',
          100 * S_FALLBACK_FAIL, '% are reported as failed:', TW_FAILED)
    with open(RUN_DIR / 'arm_failures.json', 'w') as fh:
        json.dump(ARM_FAILURES, fh, indent=1)
    for arm in CYC_ARMS:
        for n in KEEP:
            check_cycles(CYC[arm][n], AUDIO_META[n]['L4'], arm + ' ' + n, strict=(arm != 'pcg_seg'))
    print('   every boundary of every arm lies inside its signal')

    # ---- diagnostics against arm C (descriptive; Gate B of the design) ----
    DIAG_ARMS = ['pcg_seg', 'equal_time'] + TW_ARMS + TW_COV_ARMS
    OFFSETS = {arm: {} for arm in DIAG_ARMS}
    DIAG_ROWS = []
    for n in KEEP:
        ref = CYC['pcg_seg'][n]
        for arm in DIAG_ARMS:
            rd, dl = rate_diag(CYC[arm][n], ref)
            OFFSETS[arm][n] = dl
            row = dict(recording=n, patient_id=PID_OF[n], age_band=AGE_OF.get(PID_OF[n], 'Unknown'), murmur=LAB_OF[n]['murmur'],
                       arm=arm, kept=True, analysed_s=AUDIO_META[n]['L4'] / TARGET_SR, n_windows=AUDIO_META[n]['n_win'])
            row.update(rd)
            for k_, v_ in ARM_DIAG[arm][n].items():
                row['arm_' + k_] = v_
            if arm in TW_ARMS:
                rec = TW_RAW[n]['rec_t']
                row.update({'dec_' + k_: v_ for k_, v_ in rec.items()})
            DIAG_ROWS.append(row)
    DIAG = pd.DataFrame(DIAG_ROWS)
    DIAG.to_csv(RUN_DIR / 'diagnostics.csv', index=False)
    DK = DIAG
    cols = ['n_cycles', 'median_rr_ms', 'bpm', 'rr_cv', 'rr_jump', 'frac_fast', 'insertion_rate', 'deletion_rate',
            'median_offset_ms', 'median_abs_offset_ms', 'overdetection_ratio']
    DIAG_SUMMARY = {}
    print('\nrate-fidelity diagnostics against the annotation, medians over %d kept recordings' % len(KEEP))
    print('%-22s %6s %7s %6s %6s %6s %6s %8s %8s %6s' % ('arm', 'cycles', 'RR ms', 'bpm', 'ins', 'del', 'RRcv', 'off ms', '|off| ms', 'x det'))
    for arm in DIAG_ARMS:
        sub = DK[DK.arm == arm]
        med = {c: float(np.nanmedian(sub[c].astype(float))) for c in cols}
        pooled = np.concatenate([OFFSETS[arm][n] for n in KEEP if len(OFFSETS[arm][n])]) if any(len(OFFSETS[arm][n]) for n in KEEP) else np.array([])
        DIAG_SUMMARY[arm] = dict(med, n=int(len(sub)), offset_sd_ms=float(np.std(pooled)) if len(pooled) >= 2 else None, n_matched_pairs=int(len(pooled)))
        if arm in TW_FALLBACK:
            DIAG_SUMMARY[arm].update(fallback_frac=TW_FALLBACK[arm], failed_arm=bool(TW_FALLBACK[arm] > S_FALLBACK_FAIL))
        print('%-22s %6.0f %7.0f %6.0f %6.3f %6.3f %6.3f %+8.1f %8.1f %6.2f' % (
            arm, med['n_cycles'], med['median_rr_ms'], med['bpm'], med['insertion_rate'], med['deletion_rate'], med['rr_cv'],
            med['median_offset_ms'], med['median_abs_offset_ms'], med['overdetection_ratio']))
    _pt = DK[DK.arm == 'timing_net_ac']
    DIAG_SUMMARY['timing_net_ac'].update(onset_fallback_total=int(_pt['arm_n_onset_fallback'].fillna(0).sum()),
                                      malformed_total=int(_pt['arm_n_malformed'].fillna(0).sum()),
                                      frac_s2_acoustic_median=float(np.nanmedian(_pt['arm_frac_s2_acoustic'].astype(float))),
                                      s2_head_at_choice_median=float(np.nanmedian(_pt['arm_s2_head_at_choice_median'].astype(float))))
    GLOBAL_OFFSET = {}
    print('arm T_ac: S2 from an acoustic onset in %.0f%% of cycles (median per recording), median S2 probability at the chosen point %.2f' % (
              100 * DIAG_SUMMARY['timing_net_ac']['frac_s2_acoustic_median'], DIAG_SUMMARY['timing_net_ac']['s2_head_at_choice_median']))

## Stage 5: features

The verbatim 464-feature extractor runs at 4 kHz on the shared audio for every arm, on the common
kept set only. Work is spread over the worker pool (forked processes with single-threaded BLAS,
pinned again with `threadpoolctl` inside each worker; serial where fork is unavailable); results
are written in chunks to a resumable part file (tied to the cycles it was built from, like the
final file) and finally to one `.npz` per arm, written
atomically (temporary file, then `os.replace`) with the feature names, recording names, patient
ids, labels and a content hash of the boundaries it was built from, so a cache built from other
cycles is never reused. An unreadable cache file is removed, rebuilt and listed in the manifest.
Feature names are fixed alphabetically from arm C on the first kept recording and every arm is
aligned to that order.

In [ ]:
def _pool_init():
    warnings.simplefilter('ignore', RuntimeWarning)
    try:
        from threadpoolctl import threadpool_limits
        threadpool_limits(limits=1)
    except ImportError:
        pass

_JOB_ARMS = {}          # {arm_tag: {name: cycles}} visible to forked workers
_JOB_EXTRACTOR = None   # callable(pcg, cycles, sr) -> dict

def _feature_worker(name):
    try:
        pcg = AUDIO4[name]
        return name, {tag: _JOB_EXTRACTOR(pcg, cyc_of[name], TARGET_SR) for tag, cyc_of in _JOB_ARMS.items()}, None
    except Exception as e:
        return name, None, '%s: %s' % (type(e).__name__, e)

def cycles_hash(cyc_of, names):
    """Content hash of what a feature matrix was built from: cycle boundaries as plain ints (independent of numpy
    scalar types, so a fresh run and a resumed run agree), or the job value itself for the window ablation."""
    h = hashlib.sha1()
    for n in names:
        v = cyc_of[n]
        if isinstance(v, list):
            h.update(json.dumps([[int(x) for k in PHASE_KEYS for x in c[k]] for c in v]).encode())
        else:
            h.update(json.dumps(jsonable(v)).encode())
        h.update(b'|')
    return h.hexdigest()[:16]

FEATURE_CACHE_HASHES = {}      # {tag: cycles_hash} for every feature matrix used in this run (written to the manifest)

def build_feature_matrices(arm_cycles, names, feature_names, part_tag, extractor=None):
    """arm_cycles: {tag: {name: cycles}}. Returns {tag: X float32 (n x d)}. Cached per tag as
    cache/features_<tag>.npz (names, feature names and cycles hash must all match); partial progress in cache/feat_part_<part_tag>.npz."""
    global _JOB_ARMS, _JOB_EXTRACTOR
    out, need, chash = {}, [], {}
    for tag in arm_cycles:
        chash[tag] = cycles_hash(arm_cycles[tag], names)
        FEATURE_CACHE_HASHES[tag] = chash[tag]

        def _ok(z, tag=tag):
            if list(z['rec_names']) == list(names) and list(z['feature_names']) == list(feature_names) and str(z['cycles_hash']) == chash[tag]:
                return z['X'].astype(np.float32)
            return None

        X = read_cache(CACHE_DIR / ('features_%s.npz' % tag), _ok)
        if X is not None:
            out[tag] = X
            print('   %-24s loaded from cache %s' % (tag, X.shape))
        else:
            need.append(tag)
    if not need:
        return out
    _JOB_ARMS = {tag: arm_cycles[tag] for tag in need}
    _JOB_EXTRACTOR = extractor or extract_all_v4_features
    part = CACHE_DIR / ('feat_part_%s.npz' % part_tag)
    part_hash = json.dumps({tag: chash[tag] for tag in need}, sort_keys=True)     # the part file is tied to the cycles it was built from
    Xs = {tag: np.zeros((len(names), len(feature_names)), np.float32) for tag in need}
    have = np.zeros(len(names), bool)
    if part.exists():
        try:
            with np.load(part, allow_pickle=False) as z:
                if (set(z.files) == set(need) | {'have', 'rec_names', 'cycles_hashes'} and list(z['rec_names']) == list(names)
                        and str(z['cycles_hashes']) == part_hash):
                    Xs = {tag: z[tag].copy() for tag in need}
                    have = z['have'].copy()
                    print('   resuming %s: %d of %d recordings already done' % (part_tag, int(have.sum()), len(names)))
                else:
                    print('   part file of %s was built from other recordings or cycles: starting again' % part_tag)
        except Exception as e:
            print('   ignoring unreadable part file (%s)' % type(e).__name__)
    todo = np.flatnonzero(~have).tolist()
    pos = {n: i for i, n in enumerate(names)}
    failed = []
    t0 = time.time()
    if todo:
        with _pool(N_PROC, _pool_init) as ex:
            bar = tqdm(total=len(todo), desc='features ' + part_tag, leave=False)
            for s in range(0, len(todo), CHUNK):
                blk = todo[s:s + CHUNK]
                for name, feats, err in ex.map(_feature_worker, [names[i] for i in blk], chunksize=4):
                    i = pos[name]
                    if err is not None:
                        failed.append((name, err))
                    else:
                        for tag in need:
                            Xs[tag][i] = [float(feats[tag].get(k, 0.0)) for k in feature_names]
                        have[i] = True
                    bar.update(1)
                tmp = str(part) + '.tmp.npz'
                np.savez(tmp, have=have, rec_names=np.array(names), cycles_hashes=np.array(part_hash), **Xs)
                os.replace(tmp, str(part))
            bar.close()
    if failed:
        raise RuntimeError('feature extraction failed for %d recordings, e.g. %s' % (len(failed), failed[:3]))
    assert have.all(), 'feature rows missing'
    for tag in need:
        X = np.nan_to_num(Xs[tag], nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        assert X.shape == (len(names), len(feature_names))
        atomic_savez_compressed(CACHE_DIR / ('features_%s.npz' % tag), X=X, feature_names=np.array(feature_names),
                                rec_names=np.array(names), patient_ids=np.array([PID_OF[n] for n in names]),
                                y_murmur=np.array([LAB_OF[n]['murmur'] for n in names]),
                                y_grading=np.array([LAB_OF[n]['grading'] for n in names]),
                                y_timing=np.array([LAB_OF[n]['timing'] for n in names]),
                                y_shape=np.array([LAB_OF[n]['shape'] for n in names]),
                                cycles_hash=np.array(chash[tag]))
        out[tag] = X
        print('   %-24s built %s (cycles hash %s)' % (tag, X.shape, chash[tag]))
    if part.exists():
        os.remove(part)
    print('   %d matrices built in %.1f min' % (len(need), (time.time() - t0) / 60))
    return out

with stage('S5 features'):
    _first = KEEP[0]
    FEATURE_NAMES = sorted(extract_all_v4_features(AUDIO4[_first], CYC['pcg_seg'][_first], TARGET_SR).keys())
    assert len(FEATURE_NAMES) == 464, 'expected 464 features, got %d' % len(FEATURE_NAMES)
    print('feature names fixed from arm C on %s: %d features' % (_first, len(FEATURE_NAMES)))
    FEATS = build_feature_matrices({arm: CYC[arm] for arm in ARMS}, KEEP, FEATURE_NAMES, 'arms')
    Y = {key: np.array([LAB_OF[n][key] for n in KEEP]) for key in ['murmur', 'grading', 'timing', 'shape']}
    G = np.array([CHILD_OF[PID_OF[n]] for n in KEEP])                 # groups = children (Patient ID union Additional ID)
    AGE = np.array([AGE_OF.get(PID_OF[n], 'Unknown') for n in KEEP])
    for arm in ARMS:
        assert FEATS[arm].shape == (len(KEEP), 464), arm
        assert np.all(np.isfinite(FEATS[arm])), arm
    print('kept set: %d recordings, %d children | age bands: %s' % (len(KEEP), len(np.unique(G)), dict(collections.Counter(AGE.tolist()))))
    FEATURE_REPRO = None
    if VALIDATION_APPLICABLE and R1_RUN_DIR and Path(R1_RUN_DIR, 'cache').is_dir():
        FEATURE_REPRO = {}
        for arm in R1_ARMS:
            fp_ = Path(R1_RUN_DIR) / 'cache' / ('features_%s.npz' % arm)
            if fp_.exists():
                z_ = np.load(fp_, allow_pickle=False)
                same_names = list(z_['rec_names']) == list(KEEP)
                FEATURE_REPRO[arm] = dict(same_names=bool(same_names),
                                          max_abs_diff=float(np.max(np.abs(z_['X'] - FEATS[arm]))) if same_names else None)
        print('   R1 feature matrices reproduced:', FEATURE_REPRO)
    E_FEATURES_SHA = hashlib.sha256(np.ascontiguousarray(FEATS['equal_time'].astype(np.float32)).tobytes()).hexdigest()
    print('   arm E features against Run 1: %s (sha256 %s)' % (('reproduced' if E_FEATURES_SHA == RUN1_E_FEATURES_SHA else 'NOT reproduced') if VALIDATION_APPLICABLE else 'not applicable (reduced run)', E_FEATURES_SHA[:16]))

## Stage 6: folds

`StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=42)` grouped by patient, drawn once per
task on the common kept set from the labels and patient ids only, and saved to `folds/<task>.npz`
with the recording names and patient ids so the CNN notebook uses the identical splits. Every arm
and every ablation reads these splits; nothing draws its own. In a reduced run (`DRY_RUN` or
`LIMIT_RECORDINGS`) a task whose labelled set has fewer than `N_SPLITS` patients or a class with
fewer than two members is skipped with a message and listed in `results.json` under
`sample.skipped_tasks`; every later stage iterates the remaining tasks. A full run never skips a
task.

In [ ]:
FOLDS = {}
TASK_KEY = dict(TASKS)        # every task's label key, kept even when a task is skipped below
SKIPPED_TASKS = {}

def draw_or_load_folds(task, names_all, y_all, groups_all, fname):
    """Patient-grouped stratified folds on one labelled set (y == -1 excluded), cached in folds/<fname>.
    Returns the fold record, or None when fewer than two folds are possible."""
    mask = y_all != -1
    ym, gm, nm = y_all[mask], groups_all[mask], names_all[mask]
    fp = FOLD_DIR / fname

    def _ok(z):
        if list(z['names']) == list(nm) and np.array_equal(z['y'], ym):
            k = int(z['n_splits'])
            return dict(mask=mask, y=ym, groups=gm, names=nm, splits=[(z['train_%02d' % i], z['test_%02d' % i]) for i in range(k)], n_splits=k)
        return None

    got = read_cache(fp, _ok)
    if got is not None:
        print('   %-10s folds loaded from %s' % (task, fp.name))
        return got
    n_groups = len(np.unique(gm))
    min_class = int(min(c for c in np.bincount(ym) if c > 0)) if len(ym) else 0
    k = min(N_SPLITS, n_groups, min_class)          # the source's rule; only a reduced run ever lowers it
    if k < 2:
        return None
    if k < N_SPLITS:
        print('   %-10s %d patients, smallest class %d: using %d folds (reduced runs only)' % (task, n_groups, min_class, k))
    splits = list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED).split(ym, ym, groups=gm))
    for tr, te in splits:
        assert not set(gm[tr]) & set(gm[te]), 'patient overlap between train and test'
    fold_id = np.full(len(ym), -1)
    for i, (_, te) in enumerate(splits):
        fold_id[te] = i
    assert (fold_id >= 0).all()
    arrays = {'train_%02d' % i: tr for i, (tr, _) in enumerate(splits)}
    arrays.update({'test_%02d' % i: te for i, (_, te) in enumerate(splits)})
    atomic_savez(fp, names=nm, patient_ids=gm, y=ym, fold_id=fold_id, n_splits=np.array(k),
                 keep_names=names_all, mask_over_keep=mask, seed=np.array(SEED), **arrays)
    print('   %-10s folds drawn and saved to %s' % (task, fp.name))
    return dict(mask=mask, y=ym, groups=gm, names=nm, splits=splits, n_splits=k)

FOLD_CHECK = {}

def embedded_folds(task, names_all, y_all, groups_all, fname):
    """The embedded child-grouped assignment, checked against this notebook's names, labels and child IDs, saved
    in the R1 fold-file layout (so the CNN notebook reads the identical splits)."""
    e = EMBED_FOLDS[task]
    mask = y_all != -1
    ym, gm, nm = y_all[mask], groups_all[mask], names_all[mask]
    checks = dict(names=list(nm) == list(e['names']), labels=bool(np.array_equal(ym, np.asarray(e['y']))),
                  children=bool(np.array_equal(gm, np.asarray(e['child']))))
    fold_id = np.asarray(e['fold_id'], np.int64)
    k = int(fold_id.max()) + 1
    splits = [(np.flatnonzero(fold_id != i), np.flatnonzero(fold_id == i)) for i in range(k)]
    checks['child_overlap_zero'] = all(not set(gm[tr]) & set(gm[te]) for tr, te in splits)
    checks['hash'] = fold_record_hash(nm, gm, ym, fold_id) == e['sha256']
    FOLD_CHECK[task] = checks
    assert all(checks.values()), 'embedded folds for %s fail their checks: %s' % (task, checks)
    arrays = {'train_%02d' % i: tr for i, (tr, _) in enumerate(splits)}
    arrays.update({'test_%02d' % i: te for i, (_, te) in enumerate(splits)})
    atomic_savez(FOLD_DIR / fname, names=nm, patient_ids=gm, patient_ids_original=np.array([PID_OF[n] for n in nm]), y=ym,
                 fold_id=fold_id, n_splits=np.array(k), keep_names=names_all, mask_over_keep=mask, seed=np.array(SEED),
                 grouping=np.array('child = Patient ID union Additional ID'), sha256=np.array(e['sha256']), **arrays)
    print('   %-10s embedded child-grouped folds verified (%d recordings, %d children, sha256 %s)' % (task, len(nm), len(np.unique(gm)), e['sha256'][:12]))
    return dict(mask=mask, y=ym, groups=gm, names=nm, splits=splits, n_splits=k)

with stage('S6 folds'):
    for t_ in list(TASKS):
        y_ = Y[TASKS[t_]]
        m_ = y_ != -1
        n_pat = int(len(np.unique(G[m_]))) if m_.any() else 0
        min_class = int(min(c for c in np.bincount(y_[m_]) if c > 0)) if m_.any() else 0
        if not VALIDATION_APPLICABLE and (n_pat < N_SPLITS or min_class < 2):
            SKIPPED_TASKS[t_] = 'reduced run: %d labelled patients (need %d) and smallest class %d (need 2)' % (n_pat, N_SPLITS, min_class)
            print('   %-10s SKIPPED (%s)' % (t_, SKIPPED_TASKS[t_]))
            del TASKS[t_]
            continue
        if VALIDATION_APPLICABLE:
            f_ = embedded_folds(t_, KEEP_ARR, y_, G, t_ + '.npz')
        else:
            f_ = draw_or_load_folds(t_, KEEP_ARR, y_, G, t_ + '.npz')
        assert f_ is not None, 'not enough patients or classes for grouped CV on %s' % t_
        FOLDS[t_] = f_
        sizes = [len(te) for _, te in f_['splits']]
        print('      %d labelled, %d patients, classes %s, %d folds, test sizes %d-%d' % (
            len(f_['y']), len(np.unique(f_['groups'])), dict(collections.Counter(f_['y'].tolist())), f_['n_splits'], min(sizes), max(sizes)))
    assert 'detection' in TASKS, 'the detection task cannot be skipped'
    if SKIPPED_TASKS:
        print('   tasks in this run: %s | skipped: %s' % (list(TASKS), SKIPPED_TASKS))

## Stage 7: classifiers and evaluation

Per arm and task: `StandardScaler` fitted on the training fold only. The MLP is the verbatim
`LightweightMLP` (464-128-64-C, batch-norm, ReLU, dropout 0.4 / 0.3) trained with AdamW
(lr 1e-3, weight decay 1e-4), mini-batches of 32 drawn from the fold held in GPU memory,
cross-entropy with label smoothing 0.1 and inverse-frequency class weights, gradient-norm clipping
at 1.0, up to 250 epochs, early stopping with patience 25 epochs on the macro-F1 of an inner
patient-disjoint `GroupShuffleSplit(test_size=0.15, random_state=42)` validation split, best state
restored, torch seeded per fold. Two details differ from the thesis V5 training loop and are stated
here so nobody reads them as V5: V5 balanced the classes with an inverse-frequency
`WeightedRandomSampler`, this notebook weights the loss instead; V5 stopped early on inner
accuracy, this notebook stops on inner macro-F1. The per-class counts of every inner validation
split are recorded in `fit_info`. The random forest is 300 trees, depth 15, balanced class weights.
Pooled out-of-fold labels and class probabilities are stored for both. A second recipe,
`fit_mlp_reference` (focal loss gamma 2 with class weights, full-batch steps, 300 epochs,
validation every 5 epochs, patience 5 checks, constant seed), exists only for the validation gate,
which must compare like with like against the mechanism study's numbers.

Metrics on the pooled out-of-fold predictions: accuracy, macro-F1, balanced accuracy, per-class
sensitivity and specificity, AUROC (binary, or one-vs-rest macro: each class scored against the
other two and the three areas averaged), AUPRC (one-vs-rest macro with label binarisation), the
confusion matrix and the majority-class baseline; per fold, accuracy and macro-F1. Uncertainty
comes from a patient-level bootstrap with 2000 resamples: patient draws are made once per task and
turned into a recording-weight matrix, so every confusion-matrix metric is computed for all
replicates in one float64 matrix product, and the same draws score both arms of every paired
difference. A resample whose truth holds a single class is masked (the verbatim oracle's rule) and
`n_boot_valid` records how many remain; AUROC and AUPRC are bootstrapped with a vectorised
weighted-rank computation that is asserted equal to sklearn before it is used. The vectorised
bootstrap is checked against the verbatim `boot_ci` and `boot_diff_ci` oracles on the detection
task and on one three-class task before it is trusted. Fold-wise paired t-tests are reported
alongside. The paired differences cover every arm against the two equal-time controls, B and C
against A, C against B, and two pairs that separate the sources of loss: C minus D (annotated
phase boundaries against fixed RR fractions on the same annotated cycle starts, the cost of the
fixed phase model) and A minus D (synthetic against annotated anchors under the same fixed
fractions, the cost of anchor timing). Every cached CV result stores the labels, a hash of the feature matrix, a hash of the
classifier configuration and the recipe, and is rebuilt when any of them differs.

In [ ]:
class LightweightMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dims=(128, 64), dropouts=(0.4, 0.3)):
        super().__init__()
        layers = []; in_dim = input_dim
        for h, d in zip(hidden_dims, dropouts):
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h, momentum=0.1), nn.ReLU(), nn.Dropout(d)]; in_dim = h
        layers.append(nn.Linear(in_dim, num_classes)); self.model = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x): return self.model(x)


MLP_CFG = dict(lr=1e-3, weight_decay=1e-4, batch=32, max_epochs=250, patience=25, label_smoothing=0.1,
               inner_val=0.15, clip_grad_norm=1.0)
RF_CFG = dict(n_estimators=300, max_depth=15, class_weight='balanced', random_state=SEED, n_jobs=-1)
REFERENCE_MLP_CFG = dict(model='ReferenceMLP, default init (Mechanism cell 30)', loss='focal gamma 2.0 with inverse-frequency class weights',
                         batch='full', max_epochs=300, val_every=5, patience_checks=5, seed='constant SEED', inner_val=0.15,
                         tf32='on during the gate fits, as in the reference run')
CFG_HASH = hashlib.sha1(json.dumps(dict(mlp=MLP_CFG, rf=RF_CFG, reference=REFERENCE_MLP_CFG, seed=SEED, n_splits=N_SPLITS),
                                   sort_keys=True, default=str).encode()).hexdigest()[:16]

def fit_mlp(Xtr, ytr, gtr, ncls, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    info = {'inner_split': 'GroupShuffleSplit', 'recipe': 'primary'}
    if len(np.unique(gtr)) >= 3:
        tr, va = next(GroupShuffleSplit(n_splits=1, test_size=MLP_CFG['inner_val'], random_state=SEED).split(Xtr, ytr, groups=gtr))
    else:
        tr, va = np.arange(len(ytr)), np.arange(len(ytr))       # reduced runs only; recorded
        info['inner_split'] = 'degenerate: fewer than 3 patients in the training fold'
    Xt = torch.tensor(Xtr[tr], dtype=torch.float32, device=DEV)
    yt = torch.tensor(ytr[tr], dtype=torch.long, device=DEV)
    Xv = torch.tensor(Xtr[va], dtype=torch.float32, device=DEV)
    yv = ytr[va]
    cnt = np.bincount(ytr[tr], minlength=ncls).astype(float)
    w = torch.tensor(cnt.sum() / (ncls * np.maximum(cnt, 1.0)), dtype=torch.float32, device=DEV)
    model = LightweightMLP(Xtr.shape[1], ncls).to(DEV)
    opt = AdamW(model.parameters(), lr=MLP_CFG['lr'], weight_decay=MLP_CFG['weight_decay'])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=MLP_CFG['label_smoothing'])
    g = torch.Generator().manual_seed(seed)
    n, bs = len(tr), MLP_CFG['batch']
    n_steps = max(1, n // bs) if n > bs else 1
    labs = list(range(ncls))
    best, state, bad, best_ep, ep = -1.0, None, 0, -1, -1
    for ep in range(MLP_CFG['max_epochs']):
        model.train()
        perm = torch.randperm(n, generator=g).to(DEV)
        for s in range(n_steps):
            idx = perm[s * bs:(s + 1) * bs] if n > bs else perm
            if len(idx) < 2:
                continue
            opt.zero_grad(set_to_none=True)
            loss = crit(model(Xt[idx]), yt[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MLP_CFG['clip_grad_norm'])
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(Xv).argmax(1).cpu().numpy()
        f = f1_score(yv, pv, average='macro', labels=labs, zero_division=0)
        if f > best:
            best, bad, best_ep = f, 0, ep
            state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= MLP_CFG['patience']:
                break
    model.load_state_dict(state)
    model.eval()
    info.update(best_epoch=int(best_ep), epochs_run=int(ep + 1), best_val_macro_f1=float(best), n_inner_train=int(n), n_inner_val=int(len(va)),
                inner_val_class_counts=np.bincount(yv, minlength=ncls).tolist())
    return model, info

class FocalLoss(nn.Module):                      # Mechanism_Rhythm_Integrity cell 30, verbatim semantics
    def __init__(self, g=2.0, w=None):
        super().__init__()
        self.g, self.w = g, w

    def forward(self, lo, y):
        ce = F.cross_entropy(lo, y, weight=self.w, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.g * ce).mean()

class ReferenceMLP(nn.Module):       # Mechanism_Rhythm_Integrity cell 30 as written there: PyTorch default initialisation, no Kaiming step
    def __init__(self, d, c, hidden=(128, 64), drops=(0.4, 0.3)):
        super().__init__()
        L, prev = [], d
        for h, p in zip(hidden, drops):
            L += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p)]
            prev = h
        L.append(nn.Linear(prev, c))
        self.net = nn.Sequential(*L)

    def forward(self, x):
        return self.net(x)

def fit_mlp_reference(Xtr, ytr, gtr, ncls, seed=SEED):
    """The recipe that produced the gate's reference values: the reference study's own model class (default
    initialisation, not the V5 class with its Kaiming step), focal(2.0), full batch, 300 epochs, validation every
    5 epochs, patience 5 checks, constant seed."""
    torch.manual_seed(seed)
    labs = list(range(ncls))
    tr, va = next(GroupShuffleSplit(1, test_size=0.15, random_state=seed).split(Xtr, ytr, groups=gtr))
    Xt = torch.tensor(Xtr[tr], dtype=torch.float32, device=DEV)
    yt = torch.tensor(ytr[tr], dtype=torch.long, device=DEV)
    Xv = torch.tensor(Xtr[va], dtype=torch.float32, device=DEV)
    cnt = np.bincount(ytr[tr], minlength=ncls).astype(float)
    w = torch.tensor(cnt.sum() / (ncls * np.maximum(cnt, 1)), dtype=torch.float32, device=DEV)
    m = ReferenceMLP(Xtr.shape[1], ncls).to(DEV)
    opt = AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = FocalLoss(2.0, w)
    best, state, bad = -1.0, None, 0
    for ep in range(300):
        m.train()
        opt.zero_grad()
        crit(m(Xt), yt).backward()
        opt.step()
        if (ep + 1) % 5 == 0:
            m.eval()
            with torch.no_grad():
                pv = m(Xv).argmax(1).cpu().numpy()
            f = f1_score(ytr[va], pv, average='macro', labels=labs, zero_division=0)
            if f > best:
                best, bad, state = f, 0, {k: v.detach().clone() for k, v in m.state_dict().items()}
            else:
                bad += 1
                if bad >= 5:
                    break
    if state:
        m.load_state_dict(state)
    return m.eval(), dict(inner_split='GroupShuffleSplit', recipe='reference-focal-fullbatch', best_val_macro_f1=float(best),
                          inner_val_class_counts=np.bincount(ytr[va], minlength=ncls).tolist())

# ---- Extension Run 1: parallel GPU workers for the fold-by-seed MLP fits ----
GPU_WORKERS = max(1, min(8, (os.cpu_count() or 2) - 2))
MLP_POOL, _mlpw = None, None
if (DEV.type == 'cuda' or os.environ.get('RUN1_POOL_ON_CPU')) and GPU_WORKERS > 1:
    import base64 as _b64w, importlib as _importlib
    _wsrc = _b64w.b64decode('IiIiRXh0ZW5zaW9uIFJ1biAxIE1MUCB3b3JrZXI6IExpZ2h0d2VpZ2h0TUxQLCBmaXRfbWxwIGFuZCBtbHBfcHJvYmEgY29waWVkIHZlcmJhdGltIGZyb20gdGhlIG5vdGVib29rIGNlbGwuIiIiCmltcG9ydCBvcywgcmFuZG9tCm9zLmVudmlyb24uc2V0ZGVmYXVsdCgnQ1VCTEFTX1dPUktTUEFDRV9DT05GSUcnLCAnOjQwOTY6OCcpCmZvciBfdiBpbiAoJ09NUF9OVU1fVEhSRUFEUycsICdPUEVOQkxBU19OVU1fVEhSRUFEUycsICdNS0xfTlVNX1RIUkVBRFMnLCAnTlVNRVhQUl9OVU1fVEhSRUFEUycsICdWRUNMSUJfTUFYSU1VTV9USFJFQURTJyk6CiAgICBvcy5lbnZpcm9uW192XSA9ICcxJwppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRvcmNoLm9wdGltIGltcG9ydCBBZGFtVwpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBHcm91cFNodWZmbGVTcGxpdApmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgZjFfc2NvcmUKCk1MUF9DRkcsIFNFRUQsIERFViA9IE5vbmUsIE5vbmUsIE5vbmUKCgpkZWYgaW5pdChjZmcsIHNlZWQpOgogICAgIiIiU2FtZSBkZXRlcm1pbmlzbSBzZXR0aW5ncyBhcyB0aGUgbm90ZWJvb2sncyBlbnZpcm9ubWVudCBjZWxsLiIiIgogICAgZ2xvYmFsIE1MUF9DRkcsIFNFRUQsIERFVgogICAgTUxQX0NGRywgU0VFRCA9IGRpY3QoY2ZnKSwgc2VlZAogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDEpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5kZXRlcm1pbmlzdGljID0gVHJ1ZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKICAgIHRvcmNoLmJhY2tlbmRzLmN1ZGEubWF0bXVsLmFsbG93X3RmMzIgPSBGYWxzZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYWxsb3dfdGYzMiA9IEZhbHNlCiAgICB0b3JjaC51c2VfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKFRydWUsIHdhcm5fb25seT1UcnVlKQogICAgREVWID0gdG9yY2guZGV2aWNlKCdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCgoKY2xhc3MgTGlnaHR3ZWlnaHRNTFAobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dF9kaW0sIG51bV9jbGFzc2VzLCBoaWRkZW5fZGltcz0oMTI4LCA2NCksIGRyb3BvdXRzPSgwLjQsIDAuMykpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGxheWVycyA9IFtdOyBpbl9kaW0gPSBpbnB1dF9kaW0KICAgICAgICBmb3IgaCwgZCBpbiB6aXAoaGlkZGVuX2RpbXMsIGRyb3BvdXRzKToKICAgICAgICAgICAgbGF5ZXJzICs9IFtubi5MaW5lYXIoaW5fZGltLCBoKSwgbm4uQmF0Y2hOb3JtMWQoaCwgbW9tZW50dW09MC4xKSwgbm4uUmVMVSgpLCBubi5Ecm9wb3V0KGQpXTsgaW5fZGltID0gaAogICAgICAgIGxheWVycy5hcHBlbmQobm4uTGluZWFyKGluX2RpbSwgbnVtX2NsYXNzZXMpKTsgc2VsZi5tb2RlbCA9IG5uLlNlcXVlbnRpYWwoKmxheWVycykKICAgICAgICBmb3IgbSBpbiBzZWxmLm1vZHVsZXMoKToKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBubi5MaW5lYXIpOiBubi5pbml0LmthaW1pbmdfbm9ybWFsXyhtLndlaWdodCwgbW9kZT0nZmFuX291dCcsIG5vbmxpbmVhcml0eT0ncmVsdScpCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KTogcmV0dXJuIHNlbGYubW9kZWwoeCkKCgpkZWYgZml0X21scChYdHIsIHl0ciwgZ3RyLCBuY2xzLCBzZWVkKToKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIGluZm8gPSB7J2lubmVyX3NwbGl0JzogJ0dyb3VwU2h1ZmZsZVNwbGl0JywgJ3JlY2lwZSc6ICdwcmltYXJ5J30KICAgIGlmIGxlbihucC51bmlxdWUoZ3RyKSkgPj0gMzoKICAgICAgICB0ciwgdmEgPSBuZXh0KEdyb3VwU2h1ZmZsZVNwbGl0KG5fc3BsaXRzPTEsIHRlc3Rfc2l6ZT1NTFBfQ0ZHWydpbm5lcl92YWwnXSwgcmFuZG9tX3N0YXRlPVNFRUQpLnNwbGl0KFh0ciwgeXRyLCBncm91cHM9Z3RyKSkKICAgIGVsc2U6CiAgICAgICAgdHIsIHZhID0gbnAuYXJhbmdlKGxlbih5dHIpKSwgbnAuYXJhbmdlKGxlbih5dHIpKSAgICAgICAjIHJlZHVjZWQgcnVucyBvbmx5OyByZWNvcmRlZAogICAgICAgIGluZm9bJ2lubmVyX3NwbGl0J10gPSAnZGVnZW5lcmF0ZTogZmV3ZXIgdGhhbiAzIHBhdGllbnRzIGluIHRoZSB0cmFpbmluZyBmb2xkJwogICAgWHQgPSB0b3JjaC50ZW5zb3IoWHRyW3RyXSwgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPURFVikKICAgIHl0ID0gdG9yY2gudGVuc29yKHl0clt0cl0sIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1ERVYpCiAgICBYdiA9IHRvcmNoLnRlbnNvcihYdHJbdmFdLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9REVWKQogICAgeXYgPSB5dHJbdmFdCiAgICBjbnQgPSBucC5iaW5jb3VudCh5dHJbdHJdLCBtaW5sZW5ndGg9bmNscykuYXN0eXBlKGZsb2F0KQogICAgdyA9IHRvcmNoLnRlbnNvcihjbnQuc3VtKCkgLyAobmNscyAqIG5wLm1heGltdW0oY250LCAxLjApKSwgZHR5cGU9dG9yY2guZmxvYXQzMiwgZGV2aWNlPURFVikKICAgIG1vZGVsID0gTGlnaHR3ZWlnaHRNTFAoWHRyLnNoYXBlWzFdLCBuY2xzKS50byhERVYpCiAgICBvcHQgPSBBZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPU1MUF9DRkdbJ2xyJ10sIHdlaWdodF9kZWNheT1NTFBfQ0ZHWyd3ZWlnaHRfZGVjYXknXSkKICAgIGNyaXQgPSBubi5Dcm9zc0VudHJvcHlMb3NzKHdlaWdodD13LCBsYWJlbF9zbW9vdGhpbmc9TUxQX0NGR1snbGFiZWxfc21vb3RoaW5nJ10pCiAgICBnID0gdG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoc2VlZCkKICAgIG4sIGJzID0gbGVuKHRyKSwgTUxQX0NGR1snYmF0Y2gnXQogICAgbl9zdGVwcyA9IG1heCgxLCBuIC8vIGJzKSBpZiBuID4gYnMgZWxzZSAxCiAgICBsYWJzID0gbGlzdChyYW5nZShuY2xzKSkKICAgIGJlc3QsIHN0YXRlLCBiYWQsIGJlc3RfZXAsIGVwID0gLTEuMCwgTm9uZSwgMCwgLTEsIC0xCiAgICBmb3IgZXAgaW4gcmFuZ2UoTUxQX0NGR1snbWF4X2Vwb2NocyddKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgcGVybSA9IHRvcmNoLnJhbmRwZXJtKG4sIGdlbmVyYXRvcj1nKS50byhERVYpCiAgICAgICAgZm9yIHMgaW4gcmFuZ2Uobl9zdGVwcyk6CiAgICAgICAgICAgIGlkeCA9IHBlcm1bcyAqIGJzOihzICsgMSkgKiBic10gaWYgbiA+IGJzIGVsc2UgcGVybQogICAgICAgICAgICBpZiBsZW4oaWR4KSA8IDI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBvcHQuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgIGxvc3MgPSBjcml0KG1vZGVsKFh0W2lkeF0pLCB5dFtpZHhdKQogICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgTUxQX0NGR1snY2xpcF9ncmFkX25vcm0nXSkKICAgICAgICAgICAgb3B0LnN0ZXAoKQogICAgICAgIG1vZGVsLmV2YWwoKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBwdiA9IG1vZGVsKFh2KS5hcmdtYXgoMSkuY3B1KCkubnVtcHkoKQogICAgICAgIGYgPSBmMV9zY29yZSh5diwgcHYsIGF2ZXJhZ2U9J21hY3JvJywgbGFiZWxzPWxhYnMsIHplcm9fZGl2aXNpb249MCkKICAgICAgICBpZiBmID4gYmVzdDoKICAgICAgICAgICAgYmVzdCwgYmFkLCBiZXN0X2VwID0gZiwgMCwgZXAKICAgICAgICAgICAgc3RhdGUgPSB7azogdi5kZXRhY2goKS5jbG9uZSgpIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGJhZCArPSAxCiAgICAgICAgICAgIGlmIGJhZCA+PSBNTFBfQ0ZHWydwYXRpZW5jZSddOgogICAgICAgICAgICAgICAgYnJlYWsKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZSkKICAgIG1vZGVsLmV2YWwoKQogICAgaW5mby51cGRhdGUoYmVzdF9lcG9jaD1pbnQoYmVzdF9lcCksIGVwb2Noc19ydW49aW50KGVwICsgMSksIGJlc3RfdmFsX21hY3JvX2YxPWZsb2F0KGJlc3QpLCBuX2lubmVyX3RyYWluPWludChuKSwgbl9pbm5lcl92YWw9aW50KGxlbih2YSkpLAogICAgICAgICAgICAgICAgaW5uZXJfdmFsX2NsYXNzX2NvdW50cz1ucC5iaW5jb3VudCh5diwgbWlubGVuZ3RoPW5jbHMpLnRvbGlzdCgpKQogICAgcmV0dXJuIG1vZGVsLCBpbmZvCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgbWxwX3Byb2JhKG1vZGVsLCBYKToKICAgIHJldHVybiB0b3JjaC5zb2Z0bWF4KG1vZGVsKHRvcmNoLnRlbnNvcihYLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9REVWKSksIDEpLmNwdSgpLm51bXB5KCkKCgpkZWYgZml0X2pvYihYdHIsIHl0ciwgZ3RyLCBYdGUsIG5jbHMsIHNlZWQpOgogICAgIiIiT25lIGZpdCwgYXMgcnVuX2N2IHBlcmZvcm1zIGl0IGluLXByb2Nlc3M6IGZpdF9tbHAsIHRoZW4gbWxwX3Byb2JhIG9uIHRoZSB0ZXN0IHJvd3MuIiIiCiAgICBtb2RlbCwgaW5mbyA9IGZpdF9tbHAoWHRyLCB5dHIsIGd0ciwgbmNscywgc2VlZCkKICAgIHByID0gbWxwX3Byb2JhKG1vZGVsLCBYdGUpCiAgICByZXR1cm4gcHIsIGluZm8K').decode('utf-8')
    assert hashlib.sha256(_wsrc.encode('utf-8')).hexdigest() == '547fe1eae49c2ce1af49cae89debf3db73a394018fd2a2a13a2daf6e8eb8656d', 'worker source corrupted'
    _wdir = Path('/content') if IN_COLAB else RUN_DIR
    (_wdir / 'run1_mlp_worker.py').write_text(_wsrc, encoding='utf-8')
    if str(_wdir) not in sys.path:
        sys.path.insert(0, str(_wdir))
    _mlpw = _importlib.import_module('run1_mlp_worker')
    MLP_POOL = ProcessPoolExecutor(max_workers=GPU_WORKERS, mp_context=mp.get_context('spawn'),
                                   initializer=_mlpw.init, initargs=(MLP_CFG, SEED))
    print('MLP fits run in %d parallel %s worker processes (spawn); worker source sha256 %s' % (GPU_WORKERS, DEV.type.upper(), '547fe1eae49c2ce1af49cae89debf3db73a394018fd2a2a13a2daf6e8eb8656d'[:16]))
else:
    print('MLP fits run one at a time in this process (no GPU worker pool)')

@torch.no_grad()
def mlp_proba(model, X):
    return torch.softmax(model(torch.tensor(X, dtype=torch.float32, device=DEV)), 1).cpu().numpy()

def run_cv(X, task, tag, scaler='fold', do_mlp=True, do_rf=True, seed=SEED, recipe='primary', folds=None):
    """Patient-grouped CV on the saved folds (or on `folds`). Cached under cache/cv/<tag>__<task>.npz; the cache is reused
    only when the recording names, labels, feature-matrix hash, configuration hash, recipe and flags all match."""
    Fd = folds if folds is not None else FOLDS[task]
    Xm, ym, gm, nm = X[Fd['mask']], Fd['y'], Fd['groups'], Fd['names']
    ncls = NCLS[task]
    n = len(ym)
    fp = CV_DIR / ('%s__%s__ens%d.npz' % (tag, task, len(ENSEMBLE_SEEDS) if recipe == 'primary' else 1))
    x_hash = hashlib.sha1(np.ascontiguousarray(Xm, dtype=np.float32).tobytes()).hexdigest()[:16]

    def _ok(z):
        if (list(z['names']) == list(nm) and np.array_equal(z['y'], ym) and str(z['cfg_hash']) == CFG_HASH and str(z['x_hash']) == x_hash
                and str(z['recipe']) == recipe and bool(z['do_mlp']) == do_mlp and bool(z['do_rf']) == do_rf and str(z['scaler']) == scaler):
            res = {k: z[k] for k in z.files if k not in ('names', 'y', 'do_mlp', 'do_rf', 'scaler', 'fit_info', 'cfg_hash', 'x_hash', 'recipe')}
            res['fit_info'] = json.loads(str(z['fit_info']))
            res['names'], res['y'], res['groups'] = nm, ym, gm
            return res
        return None

    got = read_cache(fp, _ok)
    if got is not None:
        return got
    t0 = time.time()
    res = dict(fold_id=np.full(n, -1), y=ym, groups=gm, names=nm,
               pred_mlp=np.full(n, -1), proba_mlp=np.zeros((n, ncls), np.float32), fold_f1_mlp=[], fold_acc_mlp=[],
               pred_rf=np.full(n, -1), proba_rf=np.zeros((n, ncls), np.float32), fold_f1_rf=[], fold_acc_rf=[], fit_info=[],
               proba_mlp_seeds=np.zeros((n, ncls, len(ENSEMBLE_SEEDS)), np.float32))
    sc_full = StandardScaler().fit(Xm) if scaler == 'full' else None
    labs = list(range(ncls))
    futs = {}
    if do_mlp and recipe == 'primary' and MLP_POOL is not None:     # submit every fold-by-seed fit; RF runs below meanwhile
        for k, (tr, te) in enumerate(Fd['splits']):
            if scaler == 'fold':
                sc_ = StandardScaler().fit(Xm[tr])
                Xtr_, Xte_ = sc_.transform(Xm[tr]), sc_.transform(Xm[te])
            else:
                Xtr_, Xte_ = sc_full.transform(Xm[tr]), sc_full.transform(Xm[te])
            for sb in ENSEMBLE_SEEDS:
                futs[(k, sb)] = MLP_POOL.submit(_mlpw.fit_job, Xtr_, ym[tr], gm[tr], Xte_, ncls, sb + k)
    for k, (tr, te) in enumerate(Fd['splits']):
        res['fold_id'][te] = k
        if scaler == 'fold':
            sc = StandardScaler().fit(Xm[tr])
            Xtr, Xte = sc.transform(Xm[tr]), sc.transform(Xm[te])
        else:
            Xtr, Xte = sc_full.transform(Xm[tr]), sc_full.transform(Xm[te])
        if do_mlp:
            if recipe == 'primary':                 # Run 1: probabilities averaged over the five seed bases plus the fold index
                prs = []
                for sb in ENSEMBLE_SEEDS:
                    if futs:
                        p_, info = futs[(k, sb)].result()
                    else:
                        mdl, info = fit_mlp(Xtr, ym[tr], gm[tr], ncls, sb + k)
                        p_ = mlp_proba(mdl, Xte)
                        del mdl
                    prs.append(p_)
                    res['fit_info'].append(dict(info, seed=int(sb + k)))
                res['proba_mlp_seeds'][te] = np.stack(prs, -1)
                pr = np.mean(prs, 0).astype(np.float32)
                mdl = None
            else:
                mdl, info = fit_mlp_reference(Xtr, ym[tr], gm[tr], ncls, seed)   # constant seed, as the reference run
                pr = mlp_proba(mdl, Xte)
            res['proba_mlp'][te] = pr
            res['pred_mlp'][te] = pr.argmax(1)
            res['fold_f1_mlp'].append(f1_score(ym[te], pr.argmax(1), average='macro', labels=labs, zero_division=0))
            res['fold_acc_mlp'].append(accuracy_score(ym[te], pr.argmax(1)))
            if recipe != 'primary':
                res['fit_info'].append(info)
            del mdl
        if do_rf:
            rf = RandomForestClassifier(**RF_CFG).fit(Xtr, ym[tr])
            pr = np.zeros((len(te), ncls), np.float32)
            pr[:, rf.classes_] = rf.predict_proba(Xte)
            res['proba_rf'][te] = pr
            res['pred_rf'][te] = pr.argmax(1)
            res['fold_f1_rf'].append(f1_score(ym[te], pr.argmax(1), average='macro', labels=labs, zero_division=0))
            res['fold_acc_rf'].append(accuracy_score(ym[te], pr.argmax(1)))
            del rf
    assert (res['fold_id'] >= 0).all()
    for k_ in ('fold_f1_mlp', 'fold_acc_mlp', 'fold_f1_rf', 'fold_acc_rf'):
        res[k_] = np.array(res[k_], float)
    res['seconds'] = np.array(time.time() - t0)
    save = {k: v for k, v in res.items() if k not in ('y', 'groups', 'names', 'fit_info')}
    atomic_savez(fp, names=nm, y=ym, do_mlp=np.array(do_mlp), do_rf=np.array(do_rf), scaler=np.array(scaler), recipe=np.array(recipe),
                 cfg_hash=np.array(CFG_HASH), x_hash=np.array(x_hash), fit_info=np.array(json.dumps(res['fit_info'])), **save)
    return res

# ------------------------------- metrics and bootstrap -------------------------------
def cm_metrics(M):
    """M: (..., C, C) confusion counts, rows = truth. Returns arrays over the leading axes."""
    M = np.asarray(M, np.float64)
    tp = np.diagonal(M, axis1=-2, axis2=-1)
    row, col, tot = M.sum(-1), M.sum(-2), M.sum((-1, -2))
    fp_, fn_ = col - tp, row - tp
    tn = tot[..., None] - row - col + tp
    with np.errstate(divide='ignore', invalid='ignore'):
        prec = np.where(tp + fp_ > 0, tp / np.maximum(tp + fp_, 1e-12), 0.0)
        rec = np.where(row > 0, tp / np.maximum(row, 1e-12), 0.0)
        spec = np.where(tn + fp_ > 0, tn / np.maximum(tn + fp_, 1e-12), 0.0)
        f1 = np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)
        acc = tp.sum(-1) / np.maximum(tot, 1e-12)
        sup = row > 0
        bacc = (rec * sup).sum(-1) / np.maximum(sup.sum(-1), 1)
    return dict(acc=acc, macro_f1=f1.mean(-1), bacc=bacc, recall=rec, specificity=spec)

def confusion(y, pred, ncls):
    return confusion_matrix(y, pred, labels=list(range(ncls))).astype(float)

def score_metrics(y, proba, ncls, w=None):
    """(AUROC, AUPRC, note). OvR macro for three classes; None when undefined."""
    P = proba.astype(np.float64)
    P = P / np.clip(P.sum(1, keepdims=True), 1e-12, None)
    present = np.unique(y if w is None else y[w > 0])
    if len(present) < 2:
        return np.nan, np.nan, 'fewer than two classes present'
    if ncls == 2:
        return (roc_auc_score(y, P[:, 1], sample_weight=w), average_precision_score(y, P[:, 1], sample_weight=w), '')
    if len(present) < ncls:
        return np.nan, np.nan, 'a class is absent, OvR macro undefined'
    Yb = label_binarize(y, classes=list(range(ncls)))
    return (roc_auc_score(y, P, multi_class='ovr', average='macro', sample_weight=w),
            average_precision_score(Yb, P, average='macro', sample_weight=w), '')

def _boot_binary_scores(yb, s, W):
    """Weighted AUROC and average precision for every bootstrap replicate at once. yb: (n,) 0/1 truth, s: (n,) scores,
    W: (B, n) resample multiplicities. Same distinct-threshold set and tie handling as sklearn's roc_curve and
    precision_recall_curve; NaN where a replicate lacks positives or negatives."""
    order = np.argsort(-s, kind='mergesort')
    s_, y_ = s[order], yb[order].astype(np.float64)
    Wo = W[:, order].astype(np.float64)
    tp = np.cumsum(Wo * y_, 1)
    fp_ = np.cumsum(Wo * (1.0 - y_), 1)
    last = np.r_[np.flatnonzero(np.diff(s_) != 0), len(s_) - 1]          # last index of every distinct score
    tp_t, fp_t = tp[:, last], fp_[:, last]
    P, N = tp[:, -1:], fp_[:, -1:]
    with np.errstate(divide='ignore', invalid='ignore'):
        tpr, fpr = tp_t / P, fp_t / N
        prec = np.where(tp_t + fp_t > 0, tp_t / np.maximum(tp_t + fp_t, 1e-300), 0.0)
        z = np.zeros((len(W), 1))
        tpr0, fpr0 = np.concatenate([z, tpr[:, :-1]], 1), np.concatenate([z, fpr[:, :-1]], 1)
        auroc = ((fpr - fpr0) * (tpr + tpr0) / 2.0).sum(1)                   # trapezoid over the ROC = weighted rank statistic
        ap = ((tpr - tpr0) * prec).sum(1)                                     # sum of (R_k - R_k-1) P_k, as average_precision_score
    ok = (P[:, 0] > 0) & (N[:, 0] > 0)
    return np.where(ok, auroc, np.nan), np.where(ok, ap, np.nan)

def boot_score_metrics(y, proba, ncls, W):
    """(AUROC, AUPRC) per replicate: binary, or the one-vs-rest macro average over classes (sklearn's 'ovr', 'macro').
    NaN where a class has no weight in a replicate, where sklearn would refuse to score."""
    P = proba.astype(np.float64)
    P = P / np.clip(P.sum(1, keepdims=True), 1e-12, None)
    if ncls == 2:
        return _boot_binary_scores((y == 1).astype(int), P[:, 1], W)
    per = [_boot_binary_scores((y == c).astype(int), P[:, c], W) for c in range(ncls)]
    return np.stack([p[0] for p in per], 1).mean(1), np.stack([p[1] for p in per], 1).mean(1)

_W_CACHE = {}
def patient_weights(groups, n_boot, seed=SEED):
    """Vectorised patient-level resampling: (n_boot x n) multiplicity of every recording.
    Draws exactly what the verbatim boot_ci draws: rng.choice(pats, size=len(pats), replace=True)."""
    key = (hashlib.sha1(np.asarray(groups).tobytes()).hexdigest(), n_boot, seed)
    if key in _W_CACHE:
        return _W_CACHE[key]
    rng = np.random.default_rng(seed)
    pats, inv = np.unique(groups, return_inverse=True)
    W = np.zeros((n_boot, len(groups)), np.int32)
    for b in range(n_boot):
        take = rng.choice(pats, size=len(pats), replace=True)
        W[b] = np.bincount(np.searchsorted(pats, take), minlength=len(pats))[inv]
    _W_CACHE[key] = W
    return W

def boot_confusions(y, pred, W, ncls):
    onehot = np.zeros((len(y), ncls * ncls), np.float64)
    onehot[np.arange(len(y)), y * ncls + pred] = 1.0
    return (W.astype(np.float64) @ onehot).reshape(len(W), ncls, ncls)

def ci95(a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    if len(a) < 10:
        return [None, None]
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def pct(ci):
    return [None if v is None else 100 * v for v in ci]

def ci_str(ci, fmt='%5.1f'):
    """'[lo, hi]' for printing, or an open marker when a bound is undefined (fewer than 10 valid bootstrap draws)."""
    if not ci or ci[0] is None or ci[1] is None:
        return '[  n/a  ]'
    return '[' + (fmt % ci[0]) + ', ' + (fmt % ci[1]) + ']'

def evaluate_predictions(y, pred, proba, groups, ncls, fold_f1=None, fold_acc=None, with_scores=True, n_boot=None):
    """Pooled metrics with patient-bootstrap CIs; percentages for rates, [0,1] for AUROC/AUPRC. Resamples whose truth
    holds one class are masked, as the verbatim oracle skips them; n_boot_valid counts the rest."""
    n_boot = n_boot or N_BOOT
    M0 = confusion(y, pred, ncls)
    m0 = cm_metrics(M0)
    assert abs(m0['macro_f1'] - f1_score(y, pred, average='macro', labels=list(range(ncls)), zero_division=0)) < 1e-9
    assert abs(m0['acc'] - accuracy_score(y, pred)) < 1e-9
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')          # sklearn warns when y_pred holds a class absent from y_true
        assert abs(m0['bacc'] - balanced_accuracy_score(y, pred)) < 1e-9
    W = patient_weights(groups, n_boot)
    Mb = boot_confusions(y, pred, W, ncls)
    valid = (Mb.sum(2) > 0).sum(1) >= 2          # the verbatim oracle's rule
    mb = cm_metrics(Mb)
    for k_ in ('acc', 'macro_f1', 'bacc'):
        mb[k_] = np.where(valid, mb[k_], np.nan)
    mb['recall'] = np.where(valid[:, None], mb['recall'], np.nan)
    mb['specificity'] = np.where(valid[:, None], mb['specificity'], np.nan)
    out = dict(n=int(len(y)), n_patients=int(len(np.unique(groups))), n_boot=int(n_boot), n_boot_valid=int(valid.sum()),
               accuracy=100 * float(m0['acc']), accuracy_ci=pct(ci95(mb['acc'])),
               macro_f1=100 * float(m0['macro_f1']), macro_f1_ci=pct(ci95(mb['macro_f1'])),
               balanced_accuracy=100 * float(m0['bacc']), balanced_accuracy_ci=pct(ci95(mb['bacc'])),
               sensitivity=[100 * float(v) for v in m0['recall']], sensitivity_ci=[pct(ci95(mb['recall'][:, c])) for c in range(ncls)],
               specificity=[100 * float(v) for v in m0['specificity']], specificity_ci=[pct(ci95(mb['specificity'][:, c])) for c in range(ncls)],
               confusion_matrix=M0.astype(int).tolist(), boot_macro_f1=mb['macro_f1'], boot_acc=mb['acc'])
    if with_scores and proba is not None:
        au, ap, note = score_metrics(y, proba, ncls)
        out.update(auroc=float(au), auprc=float(ap), score_note=note)
        if note == '':
            bau, bap = boot_score_metrics(y, proba, ncls, W)
            out.update(auroc_ci=ci95(bau), auprc_ci=ci95(bap), n_boot_valid_scores=int(np.isfinite(bau).sum()))
        else:
            out.update(auroc_ci=[None, None], auprc_ci=[None, None], n_boot_valid_scores=0)
    if fold_f1 is not None and len(fold_f1):
        out.update(fold_macro_f1=[100 * float(v) for v in fold_f1], fold_macro_f1_mean=100 * float(np.mean(fold_f1)), fold_macro_f1_sd=100 * float(np.std(fold_f1)),
                   fold_accuracy=[100 * float(v) for v in fold_acc], fold_accuracy_mean=100 * float(np.mean(fold_acc)), fold_accuracy_sd=100 * float(np.std(fold_acc)))
    return out

def majority_baseline(y, ncls):
    cnt = np.bincount(y, minlength=ncls)
    top = int(np.argmax(cnt))
    pred = np.full(len(y), top)
    return dict(majority_class=top, accuracy=100 * float(accuracy_score(y, pred)),
                macro_f1=100 * float(f1_score(y, pred, average='macro', labels=list(range(ncls)), zero_division=0)),
                class_counts=cnt.tolist())

def paired_difference(ea, eb, fold_a=None, fold_b=None):
    """ea, eb: evaluate_predictions outputs sharing the same bootstrap draws. P(>0) and the CI use the finite draws only."""
    out = {}
    for m, key in [('macro_f1', 'boot_macro_f1'), ('accuracy', 'boot_acc')]:
        d = 100 * (ea[key] - eb[key])
        fin = np.isfinite(d)
        out[m] = dict(point=float(ea[m] - eb[m]), boot_mean=float(np.mean(d[fin])) if fin.any() else None, ci=ci95(d),
                      p_gt0=float(np.mean(d[fin] > 0)) if fin.any() else None, n_boot_valid=int(fin.sum()))
    if fold_a is not None and fold_b is not None and len(fold_a) == len(fold_b) and len(fold_a) >= 3:
        a, b = 100 * np.asarray(fold_a), 100 * np.asarray(fold_b)
        if np.allclose(a, b):
            out['fold_ttest'] = dict(t=None, p=None, n_folds=int(len(a)), mean_diff=0.0, note='identical fold scores')
        else:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                t_, p_ = ttest_rel(a, b)
            out['fold_ttest'] = dict(t=float(t_), p=float(p_), n_folds=int(len(a)), mean_diff=float(np.mean(a - b)))
    return out

def strip_boot(e):
    return {k: v for k, v in e.items() if not k.startswith('boot_')}

print('classifier and evaluation machinery defined | MLP', MLP_CFG, '| RF', RF_CFG, '| configuration hash', CFG_HASH)

In [ ]:
# C-D: annotated phase boundaries against fixed RR fractions on the same annotated cycle starts (the cost of the fixed phase
# model, R3-M4). A-D: synthetic anchors against annotated anchors under the same fixed fractions (the cost of anchor timing).
PAIRS = [('timing_net_ac', 'equal_time')]

SCHEMA = {
    'run': 'flags, paths, seeds, sample sizes, tasks run and equal-time definitions of this run',
    'sample': 'stage-1 counts, kept set (names), drop reasons, tasks skipped in a reduced run, per-task n / patients / class counts',
    'diagnostics_summary': 'per arm: medians over kept recordings of the rate-fidelity diagnostics, Pearson r of median RR vs annotation, offset SD (signed, matched pairs) and IQR of per-recording median |offset|, detector paths (A), fusion statistics (B), annotation gaps (D), annotated coverage (E2). Sign convention: median_offset_ms = arm cycle start minus annotated S1 onset (negative = arm precedes S1); arm_mean_syn_ac_offset_ms = acoustic S1 onset minus synthetic R-peak (positive = acoustic onset follows R)',
    'global_offset_ms': 'offset maximising S1-onset agreement for arms A and B (verbatim best_offset); negative = peak precedes S1',
    'arms.<task>.<arm>': 'n, n_patients, majority baseline, and for mlp / rf: accuracy, macro_f1, balanced_accuracy, sensitivity and specificity per class (all in %), auroc and auprc (0-1), *_ci = 95% patient-bootstrap percentile interval over the valid resamples (n_boot_valid; [null, null] when fewer than 10), confusion_matrix (rows = truth), fold_* = per-fold scores, fit_info = early-stopping record per fold',
    'paired.<task>.<X-Y>.<clf>': 'pairs A-E, B-E, C-E, D-E, B-A, C-A, C-B, C-D (annotated phases against fixed fractions on the same cycle starts), A-D (synthetic against annotated anchors under the same fixed fractions), A-E2, B-E2, C-E2, E2-E; X minus Y on macro_f1 and accuracy: point estimate, bootstrap mean, 95% CI, P(diff>0) over the finite paired draws (n_boot_valid) from the same patient resamples; fold_ttest = scipy ttest_rel over the per-fold scores',
    'validation_gate': 'sample check (per-task n and patients against the reference run) and arm C / arm E macro-F1 under the reference recipe (ReferenceMLP, TF32 on) against the mechanism-study values; a cell is within when |delta| <= GATE_TOL or the reference value lies inside the 95% patient-bootstrap interval of the gate value (ci, within per cell); refset = the same on the reference-rule set when the kept set differs',
    'ablations': 'window_length (R1-M2), timing_budget (R1-M1), redundancy (R1-m3), scaling (R3-m3), subgroups (R3-M4)',
    'stage_seconds': 'wall-clock seconds per stage in this session',
    'units': 'rates in percent (0-100); auroc / auprc in [0,1]; offsets in ms; RR in ms',
}
RESULTS = {'schema': SCHEMA,
           'run': dict(run_dir=str(RUN_DIR), started=datetime.datetime.now().isoformat(timespec='seconds'), dry_run=DRY_RUN,
                       limit_recordings=LIMIT_RECORDINGS, n_boot=N_BOOT, seed=SEED, n_splits=N_SPLITS, gate_tol=GATE_TOL, n_min_cycles=N_MIN_CYCLES,
                       run_flags=RUN_FLAGS, dataset=str(AUDIO_PATH), n_pairs=N_PAIRS, checkpoint=str(GAN_CHECKPOINT), checkpoint_sha256=CKPT_SHA256,
                       gpu=GPU_NAME, pool=POOL_KIND, mlp_cfg=MLP_CFG, rf_cfg=RF_CFG, reference_mlp_cfg=REFERENCE_MLP_CFG, fusion_cfg=FUSION,
                       match_ms=MATCH_MS, join_ms=JOIN_MS, realign_ms=REALIGN_MS, cfg_hash=CFG_HASH, arm_cfg_hash=ARM_CFG_HASH,
                       equal_time_definitions=EQUAL_TIME_DEFINITIONS, arms=ARMS, arm_letters=ARM_LETTER, arm_labels=ARM_LABEL, tasks=list(TASKS)),
           'sample': dict(n_indexed=len(NAMES_ALL), n_kept=len(KEEP), n_dropped=len(DROPPED), kept_names=list(KEEP),
                          dropped={n: bad for n, bad in DROPPED}, excluded_stage1=EXCLUDED, skipped_tasks=SKIPPED_TASKS,
                          per_task={t: dict(n=int(len(FOLDS[t]['y'])), n_patients=int(len(np.unique(FOLDS[t]['groups']))),
                                            n_splits=FOLDS[t]['n_splits'], class_counts=np.bincount(FOLDS[t]['y'], minlength=NCLS[t]).tolist()) for t in TASKS}),
           'diagnostics_summary': DIAG_SUMMARY, 'global_offset_ms': GLOBAL_OFFSET,
           'arms': {}, 'paired': {}, 'validation_gate': None, 'ablations': {}}

def write_results():
    RESULTS['stage_seconds'] = dict(STAGE_SECONDS)
    tmp = str(RUN_DIR / 'results.json') + '.tmp'
    with open(tmp, 'w') as fh:
        json.dump(jsonable(RESULTS), fh, indent=1)
    os.replace(tmp, str(RUN_DIR / 'results.json'))

OOF = {}
def write_oof():
    """Pooled out-of-fold predictions of every arm and every ablation seen so far; rewritten after each stage that adds some."""
    tmp = str(RUN_DIR / 'oof_predictions.npz') + '.tmp.npz'
    np.savez_compressed(tmp, arms=np.array(ARMS), tasks=np.array(list(TASKS)), **OOF)
    os.replace(tmp, str(RUN_DIR / 'oof_predictions.npz'))

def add_oof(task, tag, r, clfs=('mlp', 'rf')):
    if 'proba_mlp_seeds' in r:
        OOF['%s__%s__mlp__proba_seeds' % (task, tag)] = r['proba_mlp_seeds']
    OOF['%s__%s__names' % (task, tag)] = np.asarray(r['names'])
    OOF['%s__%s__groups' % (task, tag)] = np.asarray(r['groups'])
    OOF['%s__%s__y' % (task, tag)] = np.asarray(r['y'])
    for clf in clfs:
        OOF['%s__%s__%s__pred' % (task, tag, clf)] = r['pred_' + clf]
        OOF['%s__%s__%s__proba' % (task, tag, clf)] = r['proba_' + clf]

CV, EVAL = {}, {}
if DRY_RUN or LIMIT_RECORDINGS:   # Run 1 rule: one logged scoring per arm; a smoke test stops before any classification
    raise SystemExit('Smoke test complete: stages 0 to 6 ran. Run 3 classifies only in the full, pre-registered run.')
with stage('S7 classifiers and evaluation'):
    RESULTS['run'].update(ensemble_seeds=list(ENSEMBLE_SEEDS), groups='child', springer_cfg_hash=S_CFG_HASH, springer_variants=S_VARIANTS,
                          dataset_on_drive=str(AUDIO_PATH_DRIVE), dataset_read_from=str(AUDIO_PATH), prereg_sha256=PREREG_SHA256,
                          fallback_share=TW_FALLBACK, fallback_fail_share=S_FALLBACK_FAIL, failed_arms=TW_FAILED, tw_stamp=TW_STAMP, exp2_plan_sha256=EXP2_PLAN_SHA256,
                          tw_freeze_sha256=FREEZE_SHA, tw_cfg_hash=TW_CFG_HASH, tw_traces_sha256=TW_TRACES_SHA, run1_oof_sha256=RUN1_OOF_SHA256, fold_sha256={t: EMBED_FOLDS[t]['sha256'] for t in EMBED_FOLDS})
    for task in TASKS:
        CV[task], EVAL[task] = {}, {}
        RESULTS['arms'].setdefault(task, {})
        for arm in ARMS:
            CV[task][arm] = run_cv(FEATS[arm], task, arm)
            print('   %-10s %-20s MLP macro-F1 %5.1f acc %5.1f | RF macro-F1 %5.1f acc %5.1f | %.0f s' % (
                task, arm, 100 * f1_score(CV[task][arm]['y'], CV[task][arm]['pred_mlp'], average='macro', labels=list(range(NCLS[task])), zero_division=0),
                100 * accuracy_score(CV[task][arm]['y'], CV[task][arm]['pred_mlp']),
                100 * f1_score(CV[task][arm]['y'], CV[task][arm]['pred_rf'], average='macro', labels=list(range(NCLS[task])), zero_division=0),
                100 * accuracy_score(CV[task][arm]['y'], CV[task][arm]['pred_rf']), float(CV[task][arm]['seconds'])))

    # ---- bootstrap oracle check: vectorised == verbatim boot_ci / boot_diff_ci (same draws), detection and one 3-class task ----
    def _oracle_check(task):
        r_, e_ = CV[task]['timing_net_ac'], CV[task]['equal_time']     # Run 3: T_ac in place of C (C is not rescored)
        ncls = NCLS[task]
        v0, lo0, hi0 = boot_ci(r_['y'], r_['pred_mlp'], r_['groups'], metric='f1', n_boot=N_BOOT, seed=SEED)
        d0, dlo0, dhi0 = boot_diff_ci(r_['y'], r_['pred_mlp'], e_['pred_mlp'], r_['groups'], metric='f1', n_boot=N_BOOT, seed=SEED)
        ea = evaluate_predictions(r_['y'], r_['pred_mlp'], None, r_['groups'], ncls, with_scores=False)
        eb = evaluate_predictions(e_['y'], e_['pred_mlp'], None, e_['groups'], ncls, with_scores=False)
        lo1, hi1 = ea['macro_f1_ci']
        dlo1, dhi1 = paired_difference(ea, eb)['macro_f1']['ci']
        Mb_ = boot_confusions(r_['y'], r_['pred_mlp'], patient_weights(r_['groups'], N_BOOT), ncls)
        valid_ = (Mb_.sum(2) > 0).sum(1) >= 2
        # the verbatim oracle averages macro-F1 over the classes present in a resample's truth or prediction, the vectorised path
        # over every class; the two agree exactly when every valid resample shows all classes (always true for detection)
        all_present = bool(np.all(((Mb_.sum(2) > 0) | (Mb_.sum(1) > 0))[valid_].all(1)))
        print('   %-10s arm C: verbatim boot_ci [%.3f, %.3f] vs vectorised [%.3f, %.3f] | C - E: verbatim boot_diff_ci [%.3f, %.3f] vs vectorised [%.3f, %.3f] | %d of %d one-class resamples masked' % (
            task, lo0, hi0, lo1, hi1, dlo0, dhi0, dlo1, dhi1, int((~valid_).sum()), N_BOOT))
        if all_present:
            assert abs(lo0 - lo1) < 1e-6 and abs(hi0 - hi1) < 1e-6 and abs(dlo0 - dlo1) < 1e-6 and abs(dhi0 - dhi1) < 1e-6, \
                'vectorised bootstrap disagrees with the verbatim oracle on %s' % task
            print('   %-10s vectorised bootstrap reproduces the verbatim oracle exactly' % task)
        else:
            assert not VALIDATION_APPLICABLE, 'a full run must be comparable to the oracle on %s' % task
            print('   %-10s not compared: some valid resamples lack a class entirely, where the oracle averages over fewer classes (reduced runs only)' % task)

    _oracle_check('detection')
    _three = [t for t in TASKS if NCLS[t] == 3]
    if _three:
        _oracle_check(_three[0])

    # ---- vectorised AUROC / AUPRC bootstrap == sklearn with sample weights (point estimate and three replicates) ----
    for task in ['detection'] + _three[:1]:
        r_ = CV[task]['timing_net_ac']
        ncls = NCLS[task]
        W_ = patient_weights(r_['groups'], N_BOOT)
        Wchk = np.vstack([np.ones((1, len(r_['y'])), np.int32), W_[:3]])
        au_, ap_ = boot_score_metrics(r_['y'], r_['proba_mlp'], ncls, Wchk)
        for b in range(len(Wchk)):
            sa, sp, note = score_metrics(r_['y'], r_['proba_mlp'], ncls, w=None if b == 0 else Wchk[b].astype(float))
            if note == '':
                assert abs(sa - au_[b]) < 1e-9 and abs(sp - ap_[b]) < 1e-9, ('vectorised AUROC / AUPRC disagree with sklearn', task, b, sa, au_[b], sp, ap_[b])
            else:
                assert not (np.isfinite(au_[b]) or np.isfinite(ap_[b])), ('vectorised score defined where sklearn refuses', task, b)
        print('   %-10s vectorised AUROC / AUPRC equal sklearn (weighted) on the point estimate and three replicates' % task)

    # ---- pooled metrics with CIs ----
    for task in TASKS:
        ncls = NCLS[task]
        for arm in ARMS:
            r = CV[task][arm]
            EVAL[task][arm] = {}
            for clf in ('mlp', 'rf'):
                EVAL[task][arm][clf] = evaluate_predictions(r['y'], r['pred_' + clf], r['proba_' + clf], r['groups'], ncls,
                                                            r['fold_f1_' + clf], r['fold_acc_' + clf])
            RESULTS['arms'][task][arm] = dict(n=int(len(r['y'])), n_patients=int(len(np.unique(r['groups']))),
                                              majority=majority_baseline(r['y'], ncls),
                                              mlp=strip_boot(EVAL[task][arm]['mlp']), rf=strip_boot(EVAL[task][arm]['rf']),
                                              fit_info=r['fit_info'], cv_seconds=float(r['seconds']))
        RESULTS['paired'][task] = {}
        for a, b in PAIRS:
            ra, rb = CV[task][a], CV[task][b]
            assert np.array_equal(ra['y'], rb['y']) and np.array_equal(ra['groups'], rb['groups'])
            RESULTS['paired'][task]['%s-%s' % (ARM_LETTER[a], ARM_LETTER[b])] = {
                'arms': [a, b],
                'mlp': paired_difference(EVAL[task][a]['mlp'], EVAL[task][b]['mlp'], ra['fold_f1_mlp'], rb['fold_f1_mlp']),
                'rf': paired_difference(EVAL[task][a]['rf'], EVAL[task][b]['rf'], ra['fold_f1_rf'], rb['fold_f1_rf'])}
    write_results()

    # ---- pooled out-of-fold predictions ----
    for task in TASKS:
        Fd = FOLDS[task]
        OOF[task + '__y'] = Fd['y']
        OOF[task + '__names'] = Fd['names']
        OOF[task + '__patient_ids'] = Fd['groups']
        OOF[task + '__fold_id'] = CV[task][ARMS[0]]['fold_id']
        for arm in ARMS:
            add_oof(task, arm, CV[task][arm])
    write_oof()
    print('written results.json and oof_predictions.npz')

    # ---- tables ----
    for clf in ('mlp', 'rf'):
        print('\n%s macro-F1 [95%% patient-bootstrap CI], pooled out-of-fold; [n/a] = fewer than 10 valid bootstrap draws' % clf.upper())
        print('%-24s' % 'arm' + ''.join('%22s' % t for t in TASKS))
        print('-' * (24 + 22 * len(TASKS)))
        for arm in ARMS:
            print('%-24s' % ARM_LABEL[arm] + ''.join('%22s' % ('%5.1f %s' % (EVAL[t][arm][clf]['macro_f1'], ci_str(EVAL[t][arm][clf]['macro_f1_ci']))) for t in TASKS))
        print('%-24s' % 'majority class' + ''.join('%22.1f' % RESULTS['arms'][t][ARMS[0]]['majority']['macro_f1'] for t in TASKS))
    print('\npaired differences in MLP macro-F1 (patient bootstrap 95% CI, P(diff>0) over finite draws; fold t-test p)')
    for task in TASKS:
        print('  ' + task)
        for key, v in RESULTS['paired'][task].items():
            m = v['mlp']['macro_f1']
            tt = v['mlp'].get('fold_ttest', {})
            print('     %-6s %+6.1f  %s  P(>0)=%s   fold t-test p=%s' % (
                key, m['point'], ci_str(m['ci'], '%+6.1f'), ('%.3f' % m['p_gt0']) if m['p_gt0'] is not None else 'n/a',
                ('%.3f' % tt['p']) if tt.get('p') is not None else 'n/a'))

## Stage 8: integrity gate (Run 3)

Label-blind checks, as Run 2: the kept set equals the 3,000 Run 1 recordings; the folds equal the embedded child-grouped assignment; arms C, E and E_annot have Run 1's cycle hashes; arm E's feature matrix and its out-of-fold predictions equal Run 1's; the frozen control-arm weights match the freeze record; a repeat five-seed fit returns identical probabilities; the parallel workers return what an in-process fit returns. A failed gate means the run is not used.

In [ ]:

with stage('S8 integrity gate'):
    gate = dict(kind='integrity (Run 3)', applicable=bool(VALIDATION_APPLICABLE))
    if not VALIDATION_APPLICABLE:
        VALIDATION_OK = None
        gate.update(ok=None, decided_on='not applicable (reduced run)')
        print('integrity gate: not applicable in a reduced run')
    else:
        gate['kept_set_pinned'] = bool(KEEP_PINNED_OK)
        gate['folds'] = FOLD_CHECK
        gate['folds_ok'] = all(all(v.values()) for v in FOLD_CHECK.values()) and set(FOLD_CHECK) == set(TASKS)
        gate['run1_cycles'] = {a: dict(run2=cycles_hash(CYC[a], KEEP), run1=RUN1_CYCLES_HASH[a]) for a in RUN1_CYCLES_HASH}
        gate['run1_cycles_ok'] = all(v['run2'] == v['run1'] for v in gate['run1_cycles'].values())
        gate['run1_e_features_ok'] = bool(E_FEATURES_SHA == RUN1_E_FEATURES_SHA)
        e_chk = {}
        for t in TASKS:
            r = CV[t]['equal_time']; ref = RUN1_E[t]
            same_rows = list(map(str, r['names'])) == ref['names'] and np.array_equal(np.asarray(r['y']), np.asarray(ref['y']))
            e_chk[t] = dict(same_rows=bool(same_rows),
                            mlp_pred_identical=bool(same_rows and np.array_equal(np.asarray(r['pred_mlp']), np.asarray(ref['mlp_pred']))),
                            mlp_proba_max_abs_diff=float(np.max(np.abs(np.asarray(r['proba_mlp'], float) - np.asarray(ref['mlp_proba'], float)))) if same_rows else None,
                            rf_pred_identical=bool(same_rows and np.array_equal(np.asarray(r['pred_rf']), np.asarray(ref['rf_pred']))))
        gate['run1_e_predictions'] = e_chk
        gate['run1_e_predictions_ok'] = all(v['mlp_pred_identical'] and v['rf_pred_identical'] and v['mlp_proba_max_abs_diff'] is not None
                                            and v['mlp_proba_max_abs_diff'] < 1e-5 for v in e_chk.values())
        gate['tw_weights_ok'] = bool(TW_WEIGHTS_OK)
        gate['tw_fallback'] = TW_FALLBACK
        gate['tw_failed_arms_reported'] = TW_FAILED
        Fd_ = FOLDS['detection']; tr_, te_ = Fd_['splits'][0]
        Xm_ = FEATS['equal_time'][Fd_['mask']]
        sc_ = StandardScaler().fit(Xm_[tr_])
        reps = []
        for _ in range(2):
            prs_ = []
            for sb in ENSEMBLE_SEEDS:
                m_, _i = fit_mlp(sc_.transform(Xm_[tr_]), Fd_['y'][tr_], Fd_['groups'][tr_], 2, sb)
                prs_.append(mlp_proba(m_, sc_.transform(Xm_[te_])))
            reps.append(np.mean(prs_, 0))
        gate['repeat_fit_max_abs_diff'] = float(np.max(np.abs(reps[0] - reps[1])))
        gate['repeat_fit_ok'] = gate['repeat_fit_max_abs_diff'] < 1e-6
        checks = [gate['kept_set_pinned'], gate['folds_ok'], gate['run1_cycles_ok'], gate['run1_e_features_ok'],
                  gate['run1_e_predictions_ok'], gate['tw_weights_ok'], gate['repeat_fit_ok']]
        if MLP_POOL is not None:
            Xtr0_, Xte0_ = sc_.transform(Xm_[tr_]), sc_.transform(Xm_[te_])
            p_pool_, _ = MLP_POOL.submit(_mlpw.fit_job, Xtr0_, Fd_['y'][tr_], Fd_['groups'][tr_], Xte0_, 2, ENSEMBLE_SEEDS[0]).result()
            m0_, _ = fit_mlp(Xtr0_, Fd_['y'][tr_], Fd_['groups'][tr_], 2, ENSEMBLE_SEEDS[0])
            gate['pool_vs_inprocess_max_abs_diff'] = float(np.max(np.abs(p_pool_ - mlp_proba(m0_, Xte0_))))
            gate['pool_ok'] = gate['pool_vs_inprocess_max_abs_diff'] < 1e-6
            checks.append(gate['pool_ok'])
        VALIDATION_OK = bool(all(checks))
        gate.update(ok=VALIDATION_OK, decided_on='integrity checks')
        print('integrity gate:', 'PASSED' if VALIDATION_OK else 'FAILED',
              {k: v for k, v in gate.items() if k not in ('folds', 'run1_e_predictions', 'run1_cycles')})
        print('   Run 1 cycle hashes:', gate['run1_cycles'])
        print('   arm E against Run 1:', gate['run1_e_predictions'])
    RESULTS['validation_gate'] = gate
    write_results()
    if MLP_POOL is not None:
        MLP_POOL.shutdown()


## Stages 9 and 10: not run

The R1 reviewer ablations and R1 figures are not part of Run 1; Run 1 figures are drawn locally from `results.json` and `oof_predictions.npz`.

In [ ]:
FIG_CAPTIONS, DRAW_FIGURES, FIGURE_ERRORS = {}, False, {}
print('Stages 9 and 10 are not run in Extension Run 1')

## Provenance of the verbatim blocks

Recorded at build time by `build_gating_campaign.py`: the source file, line range or function name,
and the SHA-256 of the exact text that was embedded in this notebook. The run manifest copies it,
together with `NOTEBOOK_SOURCE_SHA256`, the SHA-256 over every other cell's source at build time,
which Stage 11 compares with the live notebook when Colab exposes it.

In [ ]:
VERBATIM_PROVENANCE = [
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "160-231",
  "anchor": "class MultiHeadSelfAttention",
  "sha256": "5b1e93c513e32b279671f1509a7d9ecbbc8ad1176f9891c9dd4c7c45b1b59d85"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "233-239",
  "anchor": "gen = ImprovedGeneratorNoTanh",
  "sha256": "40cd4ef9fc2f54a68f6640c87197b51fca56e3423096c0e0dd9913c8ee3d40be"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "247-261",
  "anchor": "GAN_SR = 8000",
  "sha256": "1d0ca348ed8cb3b4bec73fd4773467482d0c8c8fdd8199e9e8f3e033a00e8e00"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "282-291",
  "anchor": "def segment_phases_samples",
  "sha256": "a34b8cff4529308f031ab6a3caf2d365736f9f949a7da83da9030120ac53459c"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "336-602",
  "anchor": "def _fast_mfcc",
  "sha256": "24e20cc577a75f91d87090e864176ebc43aa94d3bf4f4afdcec4f5ee5ea0f3ac"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "604-646",
  "anchor": "def parse_tsv_annotations",
  "sha256": "89e7d7d12d2845bd5aed18b765435fc3949e4fea4a228eb44bb92acf49c805cd"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "656-668",
  "anchor": "metadata_df = pd.read_csv",
  "sha256": "875bdfc2444ebfcc733c6c8ff505bf8844a84c2ea026a419c4d502bf6be25e95"
 },
 {
  "source": "Synthetic_vs_Expert_Gating_CV.py",
  "lines": "750-759",
  "anchor": "class LightweightMLP",
  "sha256": "abe2c174d8e3537b680b210d8bb02e6e2d9b62939b81a38c88613b1c9049da3d"
 },
 {
  "source": "Mechanism_Rhythm_Integrity.ipynb",
  "cell": 9,
  "function": "cycles_to_peaks",
  "sha256": "41195fd9f44bc4db11b5fa63ed2b34e2453e55e9c5a7b554c73e3d5506dd63bd"
 },
 {
  "source": "Mechanism_Rhythm_Integrity.ipynb",
  "cell": 9,
  "function": "rr_intervals",
  "sha256": "23ede9fef2d13d259c9d588cdb6ea259bda3fef0042e85724a68ae09da83030a"
 },
 {
  "source": "Mechanism_Rhythm_Integrity.ipynb",
  "cell": 9,
  "function": "rhythm",
  "sha256": "c947baf4a41f96623d029d96ea6ac7ec6e834a6c29cd915ff33c3dfbf01e05a1"
 },
 {
  "source": "Mechanism_Rhythm_Integrity.ipynb",
  "cell": 13,
  "function": "phases_from",
  "sha256": "0bc06c308f088991dabb881bb226a4f945cb2f992d64243dcbba7fe2a025ea63"
 },
 {
  "source": "Mechanism_Rhythm_Integrity.ipynb",
  "cell": 16,
  "function": "_sos_env",
  "sha256": "236195c3c5ea480348df7add34ff7350fbb158579f72d19901ea636d2a58c997"
 },
 {
  "source": "Mechanism_Rhythm_Integrity.ipynb",
  "cell": 16,
  "function": "_detect_sos",
  "sha256": "2f7aeaeaf1c03ef81b3d6e7f89380c626b36d0739acef278ed1b7f79da2f39cf"
 },
 {
  "source": "Peak_Detection_Diagnostic.ipynb",
  "cell": 9,
  "function": "match_one_to_one",
  "sha256": "e6a1925fb62cb82b5238f0e1c28cf956f8047570601f4341bbb25528c72ee17c"
 },
 {
  "source": "Peak_Detection_Diagnostic.ipynb",
  "cell": 9,
  "function": "match_stats",
  "sha256": "8999e625020cad545a90310f550655ab7cee7f583d7d99d7b140c431a5e43ec5"
 },
 {
  "source": "Peak_Detection_Diagnostic.ipynb",
  "cell": 9,
  "function": "best_offset",
  "sha256": "55430fe509899a25a2edade142df7fe50b8f6c654578a1acc1b98642e319edc5"
 },
 {
  "source": "Jitter_Timing_Tolerance.ipynb",
  "cell": 15,
  "function": "jitter_cycles",
  "sha256": "17c0e8b14e9f0d9d4ec4f1bc8c76f48841a6b39782086845ab6c9af8a45ee291"
 },
 {
  "source": "Jitter_Timing_Tolerance.ipynb",
  "cell": 25,
  "function": "equivalent_sigma",
  "sha256": "419031e51371a56ee4e74b4b875b692db69317dd5fbea9cf7ee1ac22f81e0806"
 },
 {
  "source": "Fair_MLP_Rerun_Bootstrap.ipynb",
  "cell": 15,
  "function": "boot_ci",
  "sha256": "649cfb0c2b35db6a95f75493d6c1cbc12bc25bbf76fd11e2bf71d5facf2c945f"
 },
 {
  "source": "Fair_MLP_Rerun_Bootstrap.ipynb",
  "cell": 15,
  "function": "boot_diff_ci",
  "sha256": "e3e8e4ce1dc8fc4eff893d1d9168c09f1312c0990974e54633ab10423d9cd9fd"
 },
 {
  "source": "ABLATION_GATING_VALIDATION.ipynb",
  "cell": 6,
  "function": "get_nongated_cycles",
  "sha256": "e8292c6336c2a2acf1b5d187091e3eb07652e467fd3df70cc4140f154093a206"
 }
]
NOTEBOOK_SOURCE_SHA256 = '40212c2b32f7067724eea13bdf406fb88e4755c6910887e0333c16eb9c2da049'   # filled in by build_run3.py
print('%d verbatim blocks embedded | notebook source sha256 %s' % (len(VERBATIM_PROVENANCE), NOTEBOOK_SOURCE_SHA256[:16]))
for _p in VERBATIM_PROVENANCE:
    print('   %-40s %-28s %s' % (_p['source'], _p.get('lines') or _p.get('function'), _p['sha256'][:16]))

## Stage 11: run manifest

Versions, GPU, seeds, run flags, dataset path and pair count, checkpoint SHA-256, the worker-pool
kind, per-stage seconds for this session and cumulative over every session of the run folder, a
git-style blob hash of every cache file (taken at write time; only files written by an earlier
notebook version are hashed here), the configuration, arm and feature-cache hashes, the cache
rebuilds and worker failures, the validation gate result, the verbatim-block provenance, the
build-time hash of the notebook's cell sources (and the live notebook's hash when Colab exposes
it), and which equal-time definition every results entry and figure used.

In [ ]:
def notebook_source_hash(cells):
    """SHA-256 over the sources of every cell except the provenance cell: the rule the builder applies at build time."""
    h = hashlib.sha256()
    for c in cells:
        src = c.get('source', '') if isinstance(c, dict) else c
        src = ''.join(src) if isinstance(src, list) else src
        if src.startswith('VERBATIM_PROVENANCE = '):
            continue
        h.update(src.encode('utf-8'))
        h.update(b'\x00')
    return h.hexdigest()

with stage('S11 manifest'):
    _t11 = time.time()
    live_hash = None
    try:
        from google.colab import _message
        _nb = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
        live_hash = notebook_source_hash(_nb['ipynb']['cells'])
    except Exception as e:
        print('   live notebook hash unavailable outside Colab (%s); the build-time hash is recorded' % type(e).__name__)
    cache_files = sorted(list(CACHE_DIR.glob('*.npz')) + list(CACHE_DIR.glob('*.pkl')) + list(CACHE_DIR.glob('*.npy')) +
                         list(CV_DIR.glob('*.npz')) + list(FOLD_DIR.glob('*.npz')))
    n_now = 0
    for f in cache_files:
        rel = str(f.relative_to(RUN_DIR))
        if rel not in CACHE_HASHES or CACHE_HASHES[rel].get('bytes') != os.path.getsize(f):
            CACHE_HASHES[rel] = dict(git_blob_sha1=git_blob_sha1(f), bytes=os.path.getsize(f))   # only files not hashed at write time
            n_now += 1
    cache_hashes = {str(f.relative_to(RUN_DIR)): CACHE_HASHES[str(f.relative_to(RUN_DIR))] for f in cache_files}
    eq_usage = {}
    for t in TASKS:
        eq_usage['arms.%s.equal_time' % t] = 'equal_time'
        eq_usage['arms.%s.equal_time_matched' % t] = 'equal_time_matched'
        for key in RESULTS['paired'][t]:
            a_, b_ = RESULTS['paired'][t][key]['arms']
            used = [x for x in (a_, b_) if x.startswith('equal_time')]
            if used:
                eq_usage['paired.%s.%s' % (t, key)] = ' and '.join(used)
    eq_usage['validation_gate.equal_time'] = 'equal_time'
    eq_usage['ablations.window_length.reference_arm_e_mlp'] = 'equal_time'
    eq_usage['figures.F1_forest_macro_f1'] = 'equal_time and equal_time_matched'
    eq_usage['figures.F2_paired_vs_equal_time'] = 'equal_time'
    eq_usage['figures.F3_timing_budget.reference_line'] = 'equal_time'
    eq_usage['figures.F4_offset_histograms'] = 'equal_time and equal_time_matched'
    eq_usage['figures.F5_subgroups'] = 'none (arms A, B, C only)'
    eq_usage['figures.F6_window_length.reference_line'] = 'equal_time'
    eq_usage['figures.F7_confusion_matrices'] = 'none (arms A, B, C only)'
    eq_usage['figures.F8_redundancy_heatmap'] = 'none (arm C features)'
    _cum = stage_seconds_cumulative()
    _cum['S11 manifest'] = _cum.get('S11 manifest', 0.0) + (time.time() - _t11)
    MANIFEST = dict(
        created=datetime.datetime.now().isoformat(timespec='seconds'), run_dir=str(RUN_DIR), resumed_from=RESUMED_FROM,
        session_id=SESSION_ID, sessions=STAGE_SESSIONS_PREV + [SESSION_ID], dry_run=DRY_RUN,
        limit_recordings=LIMIT_RECORDINGS, n_boot=N_BOOT, seed=SEED, n_splits=N_SPLITS, gate_tol=GATE_TOL, n_min_cycles=N_MIN_CYCLES, run_flags=RUN_FLAGS,
        versions=dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, scipy=scipy.__version__,
                      sklearn=sklearn.__version__, matplotlib=matplotlib.__version__, pandas=pd.__version__, pywt=pywt.__version__,
                      cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version() if torch.cuda.is_available() else None),
        gpu=GPU_NAME, cpu_count=os.cpu_count(), feature_workers=N_PROC, pool=dict(kind=POOL_KIND, uses=len(POOL_USES)),
        determinism=dict(cudnn_deterministic=torch.backends.cudnn.deterministic, cudnn_benchmark=torch.backends.cudnn.benchmark,
                         matmul_tf32=torch.backends.cuda.matmul.allow_tf32, cudnn_tf32=torch.backends.cudnn.allow_tf32,
                         deterministic_algorithms=torch.are_deterministic_algorithms_enabled()),
        dataset=dict(training_data=str(AUDIO_PATH), metadata=str(METADATA_PATH), paired_wav_tsv=N_PAIRS,
                     n_indexed=len(NAMES_ALL), n_kept=len(KEEP), n_dropped=len(DROPPED), n_excluded_stage1=len(EXCLUDED),
                     tasks=list(TASKS), skipped_tasks=SKIPPED_TASKS),
        checkpoint=dict(ac_checkpoint_dir=str(TW_SRC), freeze_sha256=FREEZE_SHA, models={k: v['weights_sha256'] for k, v in FREEZE_REC['models'].items()},
                        tw_cfg_hash=TW_CFG_HASH, tw_traces_sha256=TW_TRACES_SHA, embedded_modules=_MOD_SHA, exp2_plan_sha256=EXP2_PLAN_SHA256),
        generator=None,
        classifier=dict(mlp=MLP_CFG, rf=RF_CFG, reference=REFERENCE_MLP_CFG), fusion=FUSION,
        arm_constants=dict(join_ms=JOIN_MS, realign_ms=REALIGN_MS, match_ms=MATCH_MS, arm_code_version=ARM_CODE_VERSION),
        hashes=dict(cfg_hash=CFG_HASH, arm_cfg_hash=ARM_CFG_HASH, feature_cycles_hashes=FEATURE_CACHE_HASHES,
                    notebook_source_sha256_build=NOTEBOOK_SOURCE_SHA256, notebook_source_sha256_live=live_hash,
                    notebook_matches_build=(live_hash == NOTEBOOK_SOURCE_SHA256) if live_hash else None),
        arm_failures=ARM_FAILURES, cache_rebuilds=CACHE_REBUILDS,
        equal_time_definitions=EQUAL_TIME_DEFINITIONS, equal_time_usage_by_results_entry=eq_usage,
        stage_seconds_this_session=STAGE_SECONDS, stage_seconds_cumulative=_cum,
        cache_files=cache_hashes, cache_files_hashed_at_write_time=len(cache_files) - n_now,
        validation_gate=RESULTS['validation_gate'], verbatim_provenance=VERBATIM_PROVENANCE, figures=FIG_CAPTIONS,
        figures_drawn=DRAW_FIGURES, figure_errors=FIGURE_ERRORS,
        outputs=dict(results='results.json', oof='oof_predictions.npz', diagnostics='diagnostics.csv', index='index.json',
                     arm_failures='arm_failures.json', run_flags='run_flags.json', stage_seconds='stage_seconds.json', cache_hashes='cache_hashes.json',
                     folds=[str(p.name) for p in sorted(FOLD_DIR.glob('*.npz'))]))
    with open(RUN_DIR / 'run_manifest.json', 'w') as fh:
        json.dump(jsonable(MANIFEST), fh, indent=1)
    RESULTS['run']['finished'] = MANIFEST['created']
    write_results()
    print('run_manifest.json written | %d cache files (%d hashed now, the rest at write time) | elapsed this session %.1f min | cumulative over %d session(s) %.1f min' % (
        len(cache_hashes), n_now, sum(STAGE_SECONDS.values()) / 60, len(STAGE_SESSIONS_PREV) + 1, sum(_cum.values()) / 60))
    print('\nstage seconds (this session):')
    for k_, v_ in STAGE_SECONDS.items():
        print('   %-36s %8.1f' % (k_, v_))
    print('\nrun folder:', RUN_DIR)
    print('validation gate:', RESULTS['validation_gate']['ok'], '| decided on:', RESULTS['validation_gate'].get('decided_on'), '| figures drawn:', DRAW_FIGURES,
          '| pool:', POOL_KIND, '| cache rebuilds:', len(CACHE_REBUILDS), '| worker failures:', len(ARM_FAILURES), '| skipped tasks:', list(SKIPPED_TASKS))

# a figure that failed its checks was reported in Stage 10; it is raised here, after the manifest is on disk
if FIGURE_ERRORS:
    raise RuntimeError('figures not written (the manifest records this too): %s' % FIGURE_ERRORS)